In [3]:
pip install nba_api pandas numpy scikit-learn xgboost notebook

   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   --- ------------------------------------ 0.8/10.0 MB 3.6 MB/s eta 0:00:03
   ------- -------------------------------- 1.8/10.0 MB 4.0 MB/s eta 0:00:03
   ------------ --------------------------- 3.1/10.0 MB 4.8 MB/s eta 0:00:02
   ------------------ --------------------- 4.7/10.0 MB 5.3 MB/s eta 0:00:01
   ------------------------- -------------- 6.3/10.0 MB 5.7 MB/s eta 0:00:01
   ------------------------------- -------- 7.9/10.0 MB 5.9 MB/s eta 0:00:01
   ----------------------------------- ---- 8.9/10.0 MB 5.8 MB/s eta 0:00:01
   ---------------------------------------- 10.0/10.0 MB 5.6 MB/s  0:00:01
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---- ----------------------------------- 1.3/12.6 MB 6.5 MB/s eta 0:00:02
   ------ --------------------------------- 2.1/12.6 MB 5.2 MB/s eta 0:00:03
   ---------- ----------------------------- 3.4/12.6 MB 5.4 MB/s eta 0:00:02
   ------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: Could not install packages due to an OSError: [WinError 32] The p

  Using cached nba_api-1.11.4-py3-none-any.whl.metadata (5.8 kB)
  Using cached pandas-3.0.5-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached scikit_learn-1.9.0-cp314-cp314-win_amd64.whl.metadata (11 kB)
  Using cached xgboost-3.4.1-py3-none-win_amd64.whl.metadata (2.0 kB)
  Using cached notebook-7.6.2-py3-none-any.whl.metadata (10 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached charset_normalizer-3.5.1-cp314-cp314-win_amd64.whl.metadata (46 kB)
  Using cached idna-3.19-py3-none-any.whl.metadata (9.2 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached scipy-1.18.0-cp314-cp314-win_amd64.whl.metadata (61 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached narwhals-2.25.0-py3-none-any.whl.metadata (15 kB)
  Using cached jupyter_builder-1.2.2-py3-none-any.whl.metadata (7.7 kB)
  Using cached jupyter_server-2.20.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached jupyterlab_server-

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [2]:
print("hello")

hello


In [3]:
import time
import pandas as pd
import numpy as np
from nba_api.stats.endpoints import leaguegamelog
from nba_api.stats.library.http import NBAStatsHTTP

# --------------------------------------------------------------------------
# CONFIG
# --------------------------------------------------------------------------
SEASONS = ['2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']
SEASON_TYPE = 'Regular Season'
REQUEST_DELAY = 2.0     # increased delay between calls
MAX_RETRIES = 5
API_TIMEOUT = 120        # generous timeout for slow stats.nba.com responses

# Some environments (esp. Colab) need explicit browser-like headers
# or stats.nba.com silently stalls the connection.
NBAStatsHTTP.headers.update({
    'Host': 'stats.nba.com',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                   '(KHTML, like Gecko) Chrome/120.0 Safari/537.36',
    'Referer': 'https://www.nba.com/',
    'Origin': 'https://www.nba.com',
    'Accept': 'application/json, text/plain, */*',
})


def fetch_season_game_log(season: str, season_type: str = SEASON_TYPE) -> pd.DataFrame:
    """
    Fetches ALL team game logs for a given season in a single API call.
    Retries with exponential backoff on timeout/connection errors.
    """
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            log = leaguegamelog.LeagueGameLog(
                season=season,
                season_type_all_star=season_type,
                player_or_team_abbreviation='T',
                timeout=API_TIMEOUT
            )
            df = log.get_data_frames()[0]
            df['SEASON'] = season
            time.sleep(REQUEST_DELAY)
            return df
        except Exception as e:
            wait = REQUEST_DELAY * (2 ** (attempt - 1))  # exponential backoff
            print(f"  [Attempt {attempt}/{MAX_RETRIES}] {season} failed: "
                  f"{type(e).__name__}: {e} -> retrying in {wait:.0f}s")
            time.sleep(wait)
    raise RuntimeError(f"Failed to fetch season {season} after {MAX_RETRIES} attempts.")


def fetch_multi_season_logs(seasons: list) -> pd.DataFrame:
    all_logs = []
    for season in seasons:
        print(f"Fetching {season}...")
        season_df = fetch_season_game_log(season)
        print(f"  -> {len(season_df)} rows")
        all_logs.append(season_df)
    return pd.concat(all_logs, ignore_index=True)

In [4]:
raw_long_df = fetch_multi_season_logs(SEASONS)
print(raw_long_df.shape)

Fetching 2019-20...
  -> 2118 rows
Fetching 2020-21...
  -> 2160 rows
Fetching 2021-22...
  -> 2460 rows
Fetching 2022-23...
  -> 2460 rows
Fetching 2023-24...
  -> 2460 rows
Fetching 2024-25...
  -> 2460 rows
(14118, 30)


In [7]:
def parse_matchup_column(df: pd.DataFrame) -> pd.DataFrame:
    """
    LeagueGameLog's MATCHUP column looks like:
        'BOS @ NYK'   -> BOS is away, NYK is home
        'BOS vs. NYK' -> BOS is home, NYK is away
    We derive an explicit HOME_AWAY flag and OPPONENT abbreviation.
    """
    df = df.copy()
    df['IS_HOME'] = df['MATCHUP'].str.contains('vs.', regex=False)
    df['OPPONENT_ABBREVIATION'] = np.where(
        df['IS_HOME'],
        df['MATCHUP'].str.split('vs. ').str[1],
        df['MATCHUP'].str.split('@ ').str[1]
    )
    return df


def build_game_level_df(long_df: pd.DataFrame) -> pd.DataFrame:
    """
    Reshapes the team-level long dataframe into a game-level wide
    dataframe: one row per GAME_ID, with HOME and AWAY columns aligned.

    Vectorized via a self-merge on GAME_ID (no row-wise loops).
    """
    df = parse_matchup_column(long_df)

    # Split into home and away slices
    home_df = df[df['IS_HOME']].copy()
    away_df = df[~df['IS_HOME']].copy()

    shared_cols = ['GAME_ID', 'GAME_DATE', 'SEASON']

    home_df = home_df.add_suffix('_HOME')
    away_df = away_df.add_suffix('_AWAY')

    home_df = home_df.rename(columns={
        'GAME_ID_HOME': 'GAME_ID',
        'GAME_DATE_HOME': 'GAME_DATE',
        'SEASON_HOME': 'SEASON'
    })
    away_df = away_df.rename(columns={'GAME_ID_AWAY': 'GAME_ID'})
    away_df = away_df.drop(columns=[c for c in away_df.columns
                                     if c.replace('_AWAY', '') in ['GAME_DATE', 'SEASON']])

    # Merge home and away rows on GAME_ID -> one row per game
    game_df = pd.merge(home_df, away_df, on='GAME_ID', how='inner')

    # --------------------------------------------------------------
    # Target variable scaffolding (raw — feature engineering is Phase 2)
    # --------------------------------------------------------------
    game_df['HOME_WIN'] = (game_df['WL_HOME'] == 'W').astype(int)
    game_df['POINT_DIFF'] = game_df['PTS_HOME'] - game_df['PTS_AWAY']   # spread target
    game_df['TOTAL_PTS'] = game_df['PTS_HOME'] + game_df['PTS_AWAY']    # totals target

    game_df['GAME_DATE'] = pd.to_datetime(game_df['GAME_DATE'])
    game_df = game_df.sort_values('GAME_DATE').reset_index(drop=True)

    return game_df

In [8]:
game_level_df = build_game_level_df(raw_long_df)

print(f"Total games: {len(game_level_df)}")
print(f"Date range: {game_level_df['GAME_DATE'].min()} -> {game_level_df['GAME_DATE'].max()}")
print(f"Columns: {game_level_df.shape[1]}")

game_level_df.to_csv('nba_games_raw.csv', index=False)
print("Saved to nba_games_raw.csv")

display_cols = ['GAME_ID', 'GAME_DATE', 'TEAM_ABBREVIATION_HOME', 'TEAM_ABBREVIATION_AWAY',
                 'PTS_HOME', 'PTS_AWAY', 'HOME_WIN', 'POINT_DIFF', 'TOTAL_PTS']
print(game_level_df[display_cols].tail(10))

Total games: 7054
Date range: 2019-10-22 00:00:00 -> 2025-04-13 00:00:00
Columns: 64
Saved to nba_games_raw.csv
         GAME_ID  GAME_DATE TEAM_ABBREVIATION_HOME TEAM_ABBREVIATION_AWAY  \
7044  0022401193 2025-04-13                    HOU                    DEN   
7045  0022401189 2025-04-13                    CLE                    IND   
7046  0022401191 2025-04-13                    PHI                    CHI   
7047  0022401200 2025-04-13                    SAC                    PHX   
7048  0022401186 2025-04-13                    ATL                    ORL   
7049  0022401188 2025-04-13                    BKN                    NYK   
7050  0022401199 2025-04-13                    POR                    LAL   
7051  0022401192 2025-04-13                    MIL                    DET   
7052  0022401194 2025-04-13                    MEM                    DAL   
7053  0022401187 2025-04-13                    BOS                    CHA   

      PTS_HOME  PTS_AWAY  HOME_WIN  POIN

In [9]:
"""
Phase 2a: Reshape to long format (one row per TEAM per GAME) and
compute leakage-safe rolling form features.
"""

import pandas as pd
import numpy as np

# --------------------------------------------------------------------------
# STEP 1: Reshape game-level (wide) back into team-level (long) format
# --------------------------------------------------------------------------
def build_team_games_long(game_df: pd.DataFrame) -> pd.DataFrame:
    """
    Converts the HOME/AWAY wide game-level dataframe into a long-format
    dataframe: one row per TEAM per GAME. This is the format we need
    to correctly compute each team's rolling form across ALL their games
    (home and away combined), ordered strictly by date.
    """
    # Columns that describe the game itself (not a specific team's stats)
    base_cols = ['GAME_ID', 'GAME_DATE', 'SEASON']

    # Identify all HOME columns and their AWAY counterparts
    home_cols = [c for c in game_df.columns if c.endswith('_HOME')]
    stat_names = [c.replace('_HOME', '') for c in home_cols]

    # --- Home perspective rows ---
    home_rows = game_df[base_cols + home_cols].copy()
    home_rows.columns = base_cols + stat_names
    home_rows['IS_HOME'] = 1
    home_rows['OPPONENT'] = game_df['TEAM_ABBREVIATION_AWAY'].values
    home_rows['OPPONENT_PTS'] = game_df['PTS_AWAY'].values

    # --- Away perspective rows ---
    away_cols = [c for c in game_df.columns if c.endswith('_AWAY')]
    away_rows = game_df[base_cols + away_cols].copy()
    away_rows.columns = base_cols + [c.replace('_AWAY', '') for c in away_cols]
    away_rows['IS_HOME'] = 0
    away_rows['OPPONENT'] = game_df['TEAM_ABBREVIATION_HOME'].values
    away_rows['OPPONENT_PTS'] = game_df['PTS_HOME'].values

    # Stack home + away perspectives -> one row per team per game
    team_games = pd.concat([home_rows, away_rows], ignore_index=True)

    # WON column (from that team's own perspective)
    team_games['WON'] = (team_games['WL'] == 'W').astype(int)

    # Sort by team then date -- CRITICAL for correct rolling window behavior
    team_games = team_games.sort_values(
        ['TEAM_ABBREVIATION', 'GAME_DATE']
    ).reset_index(drop=True)

    return team_games


# --------------------------------------------------------------------------
# STEP 2: Compute leakage-safe rolling form features
# --------------------------------------------------------------------------
def add_rolling_features(team_games: pd.DataFrame, windows=(5, 10)) -> pd.DataFrame:
    """
    Adds rolling-average features per team, using ONLY prior games.

    LEAKAGE PREVENTION: .shift(1) BEFORE .rolling() ensures that the
    rolling window for game N covers games [N-window, N-1] -- it
    NEVER includes game N's own result. Without the shift, a team's
    rolling PTS average would partially include the very game we're
    trying to predict, which is a textbook leakage bug.
    """
    team_games = team_games.copy()

    # Metrics we want rolling averages for
    roll_metrics = ['PTS', 'OPPONENT_PTS', 'FG_PCT', 'FG3_PCT', 'FT_PCT',
                     'REB', 'AST', 'TOV', 'STL', 'BLK', 'WON']

    # Ensure all metrics exist and are numeric
    roll_metrics = [m for m in roll_metrics if m in team_games.columns]

    grouped = team_games.groupby('TEAM_ABBREVIATION', group_keys=False)

    for window in windows:
        for metric in roll_metrics:
            col_name = f'{metric}_ROLL{window}'
            team_games[col_name] = grouped[metric].transform(
                lambda s: s.shift(1).rolling(window, min_periods=1).mean()
            )

    return team_games


# --------------------------------------------------------------------------
# STEP 3: Rest days feature (also leakage-safe by construction --
# it only ever looks at the PREVIOUS game's date)
# --------------------------------------------------------------------------
def add_rest_days(team_games: pd.DataFrame) -> pd.DataFrame:
    """
    Days of rest before each game = days since that team's previous game.
    First game of a season/dataset gets NaN (no prior game to compare) --
    we fill with a neutral default (e.g. 3 days) rather than 0, since 0
    would incorrectly signal a back-to-back for a season-opener.
    """
    team_games = team_games.copy()
    team_games['PREV_GAME_DATE'] = team_games.groupby('TEAM_ABBREVIATION')['GAME_DATE'].shift(1)
    team_games['REST_DAYS'] = (team_games['GAME_DATE'] - team_games['PREV_GAME_DATE']).dt.days
    team_games['REST_DAYS'] = team_games['REST_DAYS'].fillna(3)
    team_games['IS_BACK_TO_BACK'] = (team_games['REST_DAYS'] <= 1).astype(int)
    team_games = team_games.drop(columns=['PREV_GAME_DATE'])
    return team_games


# --------------------------------------------------------------------------
# EXECUTION
# --------------------------------------------------------------------------
team_games = build_team_games_long(game_level_df)
team_games = add_rolling_features(team_games, windows=(5, 10))
team_games = add_rest_days(team_games)

print(f"team_games shape: {team_games.shape}")
print(f"Unique teams: {team_games['TEAM_ABBREVIATION'].nunique()}")

# --------------------------------------------------------------------------
# SANITY CHECK: eyeball a single team's rolling window to confirm no leakage
# --------------------------------------------------------------------------
check_cols = ['GAME_DATE', 'TEAM_ABBREVIATION', 'OPPONENT', 'PTS',
              'PTS_ROLL5', 'PTS_ROLL10', 'REST_DAYS', 'IS_BACK_TO_BACK']
print("\nBOS sample (first 8 games of 2023-24 season):")
bos_check = team_games[
    (team_games['TEAM_ABBREVIATION'] == 'BOS') &
    (team_games['SEASON'] == '2023-24')
].head(8)
print(bos_check[check_cols].to_string(index=False))

team_games shape: (14108, 60)
Unique teams: 30

BOS sample (first 8 games of 2023-24 season):
 GAME_DATE TEAM_ABBREVIATION OPPONENT  PTS  PTS_ROLL5  PTS_ROLL10  REST_DAYS  IS_BACK_TO_BACK
2023-10-25               BOS      NYK  108      112.2       120.1      199.0                0
2023-10-27               BOS      MIA  119      109.4       117.7        2.0                0
2023-10-30               BOS      WAS  126      113.0       117.6        3.0                0
2023-11-01               BOS      IND  155      118.8       116.5        2.0                0
2023-11-04               BOS      BKN  124      125.6       120.9        3.0                0
2023-11-06               BOS      MIN  109      126.4       119.3        2.0                0
2023-11-08               BOS      PHI  103      126.6       118.0        2.0                0
2023-11-10               BOS      BKN  121      123.4       118.2        2.0                0


In [10]:
def add_rolling_features(team_games: pd.DataFrame, windows=(5, 10)) -> pd.DataFrame:
    """
    Adds rolling-average features per team, using ONLY prior games
    WITHIN THE SAME SEASON.

    Grouping by (TEAM_ABBREVIATION, SEASON) instead of just TEAM_ABBREVIATION
    ensures rolling windows reset at the start of every season -- avoiding
    stale cross-season signal (rosters change significantly during the
    off-season via trades/free agency).
    """
    team_games = team_games.copy()

    roll_metrics = ['PTS', 'OPPONENT_PTS', 'FG_PCT', 'FG3_PCT', 'FT_PCT',
                     'REB', 'AST', 'TOV', 'STL', 'BLK', 'WON']
    roll_metrics = [m for m in roll_metrics if m in team_games.columns]

    grouped = team_games.groupby(['TEAM_ABBREVIATION', 'SEASON'], group_keys=False)

    for window in windows:
        for metric in roll_metrics:
            col_name = f'{metric}_ROLL{window}'
            team_games[col_name] = grouped[metric].transform(
                lambda s: s.shift(1).rolling(window, min_periods=1).mean()
            )

    return team_games


def add_rest_days(team_games: pd.DataFrame) -> pd.DataFrame:
    """
    Days of rest since each team's previous game, WITHIN THE SAME SEASON.

    Season openers get NaN (no in-season prior game) and are filled with
    a neutral default of 3 days -- avoids nonsensical multi-month gaps
    from carrying over the off-season.
    """
    team_games = team_games.copy()
    team_games['PREV_GAME_DATE'] = (
        team_games.groupby(['TEAM_ABBREVIATION', 'SEASON'])['GAME_DATE'].shift(1)
    )
    team_games['REST_DAYS'] = (team_games['GAME_DATE'] - team_games['PREV_GAME_DATE']).dt.days
    team_games['REST_DAYS'] = team_games['REST_DAYS'].fillna(3)
    team_games['IS_BACK_TO_BACK'] = (team_games['REST_DAYS'] <= 1).astype(int)
    team_games = team_games.drop(columns=['PREV_GAME_DATE'])
    return team_games


# --------------------------------------------------------------------------
# RE-RUN with the season-aware fix
# --------------------------------------------------------------------------
team_games = build_team_games_long(game_level_df)
team_games = add_rolling_features(team_games, windows=(5, 10))
team_games = add_rest_days(team_games)

print(f"team_games shape: {team_games.shape}")

check_cols = ['GAME_DATE', 'TEAM_ABBREVIATION', 'OPPONENT', 'PTS',
              'PTS_ROLL5', 'PTS_ROLL10', 'REST_DAYS', 'IS_BACK_TO_BACK']
print("\nBOS sample (first 8 games of 2023-24 season):")
bos_check = team_games[
    (team_games['TEAM_ABBREVIATION'] == 'BOS') &
    (team_games['SEASON'] == '2023-24')
].head(8)
print(bos_check[check_cols].to_string(index=False))

team_games shape: (14108, 60)

BOS sample (first 8 games of 2023-24 season):
 GAME_DATE TEAM_ABBREVIATION OPPONENT  PTS  PTS_ROLL5  PTS_ROLL10  REST_DAYS  IS_BACK_TO_BACK
2023-10-25               BOS      NYK  108        NaN         NaN        3.0                0
2023-10-27               BOS      MIA  119 108.000000  108.000000        2.0                0
2023-10-30               BOS      WAS  126 113.500000  113.500000        3.0                0
2023-11-01               BOS      IND  155 117.666667  117.666667        2.0                0
2023-11-04               BOS      BKN  124 127.000000  127.000000        3.0                0
2023-11-06               BOS      MIN  109 126.400000  126.400000        2.0                0
2023-11-08               BOS      PHI  103 126.600000  123.500000        2.0                0
2023-11-10               BOS      BKN  121 123.400000  120.571429        2.0                0


In [11]:
"""
Phase 2b: Possessions estimate + Offensive/Defensive Rating,
computed per-game then rolled the same leakage-safe way as Phase 2a.
"""

def add_possessions_and_ratings(team_games: pd.DataFrame) -> pd.DataFrame:
    """
    Estimates possessions per game using the standard formula:
        POSS = FGA - OREB + TOV + (0.4 * FTA)

    Then computes:
        OFF_RATING = 100 * PTS / POSS           (points scored per 100 possessions)
        DEF_RATING = 100 * OPPONENT_PTS / POSS   (points allowed per 100 possessions)

    These are GAME-LEVEL stats (not yet rolling) -- rolling versions are
    added in the next step, using the same shift-then-roll pattern.
    """
    team_games = team_games.copy()

    # Guard against missing columns across different nba_api response versions
    required = ['FGA', 'OREB', 'TOV', 'FTA', 'PTS', 'OPPONENT_PTS']
    missing = [c for c in required if c not in team_games.columns]
    if missing:
        raise ValueError(f"Missing required columns for possession estimate: {missing}")

    team_games['POSSESSIONS'] = (
        team_games['FGA'] - team_games['OREB'] + team_games['TOV'] + (0.4 * team_games['FTA'])
    )

    # Avoid division by zero on any malformed row (shouldn't happen, but defensive coding)
    safe_poss = team_games['POSSESSIONS'].replace(0, np.nan)

    team_games['OFF_RATING'] = 100 * team_games['PTS'] / safe_poss
    team_games['DEF_RATING'] = 100 * team_games['OPPONENT_PTS'] / safe_poss

    return team_games


def add_rating_rolling_features(team_games: pd.DataFrame, windows=(5, 10)) -> pd.DataFrame:
    """
    Rolling OFF_RATING / DEF_RATING, same leakage-safe shift-then-roll
    pattern, reset per season -- consistent with add_rolling_features.
    """
    team_games = team_games.copy()
    grouped = team_games.groupby(['TEAM_ABBREVIATION', 'SEASON'], group_keys=False)

    for window in windows:
        for metric in ['OFF_RATING', 'DEF_RATING', 'POSSESSIONS']:
            col_name = f'{metric}_ROLL{window}'
            team_games[col_name] = grouped[metric].transform(
                lambda s: s.shift(1).rolling(window, min_periods=1).mean()
            )

    return team_games


# --------------------------------------------------------------------------
# EXECUTION
# --------------------------------------------------------------------------
team_games = add_possessions_and_ratings(team_games)
team_games = add_rating_rolling_features(team_games, windows=(5, 10))

print(f"team_games shape: {team_games.shape}")

check_cols = ['GAME_DATE', 'TEAM_ABBREVIATION', 'OPPONENT', 'PTS', 'POSSESSIONS',
              'OFF_RATING', 'DEF_RATING', 'OFF_RATING_ROLL5', 'DEF_RATING_ROLL5']
print("\nBOS sample (first 8 games of 2023-24 season):")
bos_check = team_games[
    (team_games['TEAM_ABBREVIATION'] == 'BOS') &
    (team_games['SEASON'] == '2023-24')
].head(8)
print(bos_check[check_cols].to_string(index=False))

team_games shape: (14108, 69)

BOS sample (first 8 games of 2023-24 season):
 GAME_DATE TEAM_ABBREVIATION OPPONENT  PTS  POSSESSIONS  OFF_RATING  DEF_RATING  OFF_RATING_ROLL5  DEF_RATING_ROLL5
2023-10-25               BOS      NYK  108         93.4  115.631692  111.349036               NaN               NaN
2023-10-27               BOS      MIA  119        101.6  117.125984  109.251969        115.631692        111.349036
2023-10-30               BOS      WAS  126        107.8  116.883117   99.257885        116.378838        110.300502
2023-11-01               BOS      IND  155        108.2  143.253235   96.118299        116.546931        106.619630
2023-11-04               BOS      BKN  124        101.8  121.807466  111.984283        123.223507        103.994297
2023-11-06               BOS      MIN  109        112.6   96.802842  101.243339        122.940299        105.592294
2023-11-08               BOS      PHI  103        102.6  100.389864  103.313840        119.174529        103.57

In [12]:
"""
Phase 2c: Merge the engineered team_games features back onto the
game-level dataframe, producing HOME_* and AWAY_* feature columns
aligned on the same row -- ready for modeling in Phase 3.
"""

def merge_features_to_game_level(game_df: pd.DataFrame, team_games: pd.DataFrame) -> pd.DataFrame:
    """
    Joins team_games (long format, engineered features) back onto
    game_df (wide format) twice -- once for the home team's perspective,
    once for the away team's perspective -- keyed on GAME_ID + TEAM.
    """
    # Only pull engineered feature columns (not raw box score stats we
    # already have in game_df) to avoid duplicate/conflicting columns
    feature_cols = [c for c in team_games.columns if
                    'ROLL' in c or c in ['REST_DAYS', 'IS_BACK_TO_BACK',
                                         'OFF_RATING', 'DEF_RATING', 'POSSESSIONS']]

    merge_keys = ['GAME_ID', 'TEAM_ABBREVIATION']
    feature_slice = team_games[merge_keys + feature_cols].copy()

    # --- Merge HOME team's features ---
    home_features = feature_slice.add_suffix('_HOME')
    home_features = home_features.rename(columns={
        'GAME_ID_HOME': 'GAME_ID',
        'TEAM_ABBREVIATION_HOME': 'TEAM_ABBREVIATION_HOME'  # already matches game_df's naming
    })
    merged = pd.merge(
        game_df, home_features,
        on=['GAME_ID', 'TEAM_ABBREVIATION_HOME'], how='left'
    )

    # --- Merge AWAY team's features ---
    away_features = feature_slice.add_suffix('_AWAY')
    away_features = away_features.rename(columns={
        'GAME_ID_AWAY': 'GAME_ID',
        'TEAM_ABBREVIATION_AWAY': 'TEAM_ABBREVIATION_AWAY'
    })
    merged = pd.merge(
        merged, away_features,
        on=['GAME_ID', 'TEAM_ABBREVIATION_AWAY'], how='left'
    )

    return merged


# --------------------------------------------------------------------------
# EXECUTION
# --------------------------------------------------------------------------
model_ready_df = merge_features_to_game_level(game_level_df, team_games)

print(f"model_ready_df shape: {model_ready_df.shape}")

# --------------------------------------------------------------------------
# SANITY CHECK: confirm HOME and AWAY features landed on the same row,
# and that early-season rows correctly show NaN (season reset working
# end-to-end through the merge)
# --------------------------------------------------------------------------
check_cols = ['GAME_DATE', 'TEAM_ABBREVIATION_HOME', 'TEAM_ABBREVIATION_AWAY',
              'PTS_HOME', 'PTS_AWAY',
              'OFF_RATING_ROLL5_HOME', 'DEF_RATING_ROLL5_HOME',
              'OFF_RATING_ROLL5_AWAY', 'DEF_RATING_ROLL5_AWAY',
              'REST_DAYS_HOME', 'REST_DAYS_AWAY']

print("\nFirst BOS home game of 2023-24 season (should show NaN rolling stats):")
print(model_ready_df[
    (model_ready_df['TEAM_ABBREVIATION_HOME'] == 'BOS') &
    (model_ready_df['SEASON'] == '2023-24')
].head(3)[check_cols].to_string(index=False))

print("\nRandom mid-season sample (should show populated rolling stats):")
print(model_ready_df[model_ready_df['SEASON'] == '2023-24'].iloc[100:103][check_cols].to_string(index=False))

# Cache the fully feature-engineered dataset
model_ready_df.to_csv('nba_model_ready.csv', index=False)
print("\nSaved to nba_model_ready.csv")
print(f"Total columns: {model_ready_df.shape[1]}")
print(f"NaN count in OFF_RATING_ROLL5_HOME: {model_ready_df['OFF_RATING_ROLL5_HOME'].isna().sum()}")

model_ready_df shape: (7054, 130)

First BOS home game of 2023-24 season (should show NaN rolling stats):
 GAME_DATE TEAM_ABBREVIATION_HOME TEAM_ABBREVIATION_AWAY  PTS_HOME  PTS_AWAY  OFF_RATING_ROLL5_HOME  DEF_RATING_ROLL5_HOME  OFF_RATING_ROLL5_AWAY  DEF_RATING_ROLL5_AWAY  REST_DAYS_HOME  REST_DAYS_AWAY
2023-10-27                    BOS                    MIA       119       111             115.631692             111.349036             110.278373             109.207709             2.0             2.0
2023-11-01                    BOS                    IND       155       104             116.546931             106.619630             116.470124             108.047476             2.0             2.0
2023-11-10                    BOS                    BKN       121       107             115.827305             102.383529             109.647257             109.757913             2.0             2.0

Random mid-season sample (should show populated rolling stats):
 GAME_DATE TEAM_ABBREVIAT

In [13]:
"""
Phase 3a: Prepare the feature matrix and target variables, and build
a walk-forward (expanding window) cross-validation split generator
based on season boundaries.
"""

import pandas as pd
import numpy as np

# --------------------------------------------------------------------------
# STEP 1: Drop rows with incomplete rolling features (early-season games)
# --------------------------------------------------------------------------
def prepare_modeling_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Drops rows where either team's rolling features are NaN (i.e. their
    first in-season game). These rows can't be used for training/eval
    since they lack the predictive features entirely.
    """
    df = df.copy()

    # Any row missing core rolling features on EITHER side gets dropped
    critical_cols = ['OFF_RATING_ROLL5_HOME', 'DEF_RATING_ROLL5_HOME',
                      'OFF_RATING_ROLL5_AWAY', 'DEF_RATING_ROLL5_AWAY']

    before = len(df)
    df = df.dropna(subset=critical_cols).reset_index(drop=True)
    after = len(df)
    print(f"Dropped {before - after} rows with incomplete rolling features "
          f"({before} -> {after})")

    return df


# --------------------------------------------------------------------------
# STEP 2: Explicitly define the feature set (LEAKAGE GUARD)
# --------------------------------------------------------------------------
def get_feature_columns(df: pd.DataFrame) -> list:
    """
    Whitelist approach: only columns containing rolling stats, rest days,
    or back-to-back flags are eligible as features. This explicitly
    EXCLUDES any raw same-game box score column (PTS_HOME, FG_PCT_HOME,
    etc.) which would only be known AFTER the game finishes -- including
    those would be catastrophic leakage since they're derived from the
    very outcome we're trying to predict.
    """
    allowed_patterns = ['_ROLL5', '_ROLL10', 'REST_DAYS', 'IS_BACK_TO_BACK']
    feature_cols = [c for c in df.columns
                     if any(pattern in c for pattern in allowed_patterns)]

    print(f"Selected {len(feature_cols)} leakage-safe feature columns.")
    return feature_cols


# --------------------------------------------------------------------------
# STEP 3: Walk-forward CV split generator (expanding window, by season)
# --------------------------------------------------------------------------
def walk_forward_season_splits(df: pd.DataFrame, season_col: str = 'SEASON'):
    """
    Yields (train_idx, test_idx, test_season) tuples for an expanding-
    window walk-forward validation scheme:

        Fold 1: Train=[S1]         Test=[S2]
        Fold 2: Train=[S1,S2]      Test=[S3]
        Fold 3: Train=[S1,S2,S3]   Test=[S4]
        ... etc.

    This mirrors how the model would actually be deployed in practice --
    always trained on everything known SO FAR, tested on the next
    unseen season. No fold ever trains on future data relative to its
    own test season.
    """
    seasons_sorted = sorted(df[season_col].unique())

    for i in range(1, len(seasons_sorted)):
        train_seasons = seasons_sorted[:i]
        test_season = seasons_sorted[i]

        train_idx = df[df[season_col].isin(train_seasons)].index
        test_idx = df[df[season_col] == test_season].index

        yield train_idx, test_idx, test_season, train_seasons


# --------------------------------------------------------------------------
# EXECUTION
# --------------------------------------------------------------------------
model_df = prepare_modeling_data(model_ready_df)
feature_cols = get_feature_columns(model_df)

# Target variables
TARGETS = {
    'moneyline': 'HOME_WIN',      # classification
    'spread': 'POINT_DIFF',       # regression (home margin)
    'total': 'TOTAL_PTS'          # regression
}

print(f"\nModeling dataset: {model_df.shape[0]} games, {len(feature_cols)} features")
print(f"Seasons available: {sorted(model_df['SEASON'].unique())}")

print("\n--- Walk-forward fold structure ---")
for train_idx, test_idx, test_season, train_seasons in walk_forward_season_splits(model_df):
    print(f"Train seasons: {train_seasons} ({len(train_idx)} games)  "
          f"->  Test season: {test_season} ({len(test_idx)} games)")

# Sanity check: print the actual feature column list so we can eyeball
# that nothing leaky slipped through the whitelist
print("\n--- Feature columns (first 15) ---")
print(feature_cols[:15])

Dropped 95 rows with incomplete rolling features (7054 -> 6959)
Selected 60 leakage-safe feature columns.

Modeling dataset: 6959 games, 60 features
Seasons available: ['2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']

--- Walk-forward fold structure ---
Train seasons: ['2019-20'] (1043 games)  ->  Test season: 2020-21 (1064 games)
Train seasons: ['2019-20', '2020-21'] (2107 games)  ->  Test season: 2021-22 (1214 games)
Train seasons: ['2019-20', '2020-21', '2021-22'] (3321 games)  ->  Test season: 2022-23 (1214 games)
Train seasons: ['2019-20', '2020-21', '2021-22', '2022-23'] (4535 games)  ->  Test season: 2023-24 (1215 games)
Train seasons: ['2019-20', '2020-21', '2021-22', '2022-23', '2023-24'] (5750 games)  ->  Test season: 2024-25 (1209 games)

--- Feature columns (first 15) ---
['PTS_ROLL5_HOME', 'OPPONENT_PTS_ROLL5_HOME', 'FG_PCT_ROLL5_HOME', 'FG3_PCT_ROLL5_HOME', 'FT_PCT_ROLL5_HOME', 'REB_ROLL5_HOME', 'AST_ROLL5_HOME', 'TOV_ROLL5_HOME', 'STL_ROLL5_HOME', 'BLK_

In [14]:
"""
Phase 3b: Train Logistic Regression (baseline) and XGBoost across
every walk-forward fold, plus a naive "home team always wins" baseline
as a floor the models must beat to be worth anything.
"""

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, log_loss, brier_score_loss
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')  # sklearn convergence warnings on small folds are noisy, not informative here


def run_walk_forward_moneyline(df: pd.DataFrame, feature_cols: list, target_col: str = 'HOME_WIN'):
    """
    Trains Logistic Regression and XGBoost on each walk-forward fold,
    evaluates on the held-out next season, and collects per-fold metrics
    for later variance/stress analysis.

    Also computes the naive baseline (home team always wins) on each
    fold's test set as a sanity floor.
    """
    results = []

    for train_idx, test_idx, test_season, train_seasons in walk_forward_season_splits(df):
        X_train = df.loc[train_idx, feature_cols]
        y_train = df.loc[train_idx, target_col]
        X_test = df.loc[test_idx, feature_cols]
        y_test = df.loc[test_idx, target_col]

        # Impute any remaining NaNs (e.g. edge-case missing box score fields)
        # with the TRAINING set's median only -- using test-set stats here
        # would itself be a subtle form of leakage.
        train_medians = X_train.median()
        X_train = X_train.fillna(train_medians)
        X_test = X_test.fillna(train_medians)

        # --- Naive baseline: home team always predicted to win ---
        naive_preds = np.ones(len(y_test))
        naive_acc = accuracy_score(y_test, naive_preds)

        # --- Logistic Regression (needs scaled features) ---
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        logreg = LogisticRegression(max_iter=1000, C=1.0)
        logreg.fit(X_train_scaled, y_train)
        logreg_proba = logreg.predict_proba(X_test_scaled)[:, 1]
        logreg_preds = (logreg_proba >= 0.5).astype(int)

        # --- XGBoost ---
        xgb = XGBClassifier(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric='logloss',
            random_state=42
        )
        xgb.fit(X_train, y_train)
        xgb_proba = xgb.predict_proba(X_test)[:, 1]
        xgb_preds = (xgb_proba >= 0.5).astype(int)

        fold_result = {
            'test_season': test_season,
            'train_games': len(train_idx),
            'test_games': len(test_idx),
            'naive_accuracy': naive_acc,
            'logreg_accuracy': accuracy_score(y_test, logreg_preds),
            'logreg_logloss': log_loss(y_test, logreg_proba),
            'logreg_brier': brier_score_loss(y_test, logreg_proba),
            'xgb_accuracy': accuracy_score(y_test, xgb_preds),
            'xgb_logloss': log_loss(y_test, xgb_proba),
            'xgb_brier': brier_score_loss(y_test, xgb_proba),
        }
        results.append(fold_result)

        print(f"Season {test_season}: naive={naive_acc:.3f}  "
              f"logreg={fold_result['logreg_accuracy']:.3f} (logloss={fold_result['logreg_logloss']:.3f})  "
              f"xgb={fold_result['xgb_accuracy']:.3f} (logloss={fold_result['xgb_logloss']:.3f})")

    return pd.DataFrame(results)


# --------------------------------------------------------------------------
# EXECUTION
# --------------------------------------------------------------------------
moneyline_results = run_walk_forward_moneyline(model_df, feature_cols)

print("\n=== Summary across folds ===")
print(moneyline_results[['test_season', 'naive_accuracy', 'logreg_accuracy', 'xgb_accuracy']])

print(f"\nNaive baseline:  mean={moneyline_results['naive_accuracy'].mean():.3f}  std={moneyline_results['naive_accuracy'].std():.3f}")
print(f"LogReg:          mean={moneyline_results['logreg_accuracy'].mean():.3f}  std={moneyline_results['logreg_accuracy'].std():.3f}")
print(f"XGBoost:         mean={moneyline_results['xgb_accuracy'].mean():.3f}  std={moneyline_results['xgb_accuracy'].std():.3f}")

Season 2020-21: naive=0.543  logreg=0.585 (logloss=0.693)  xgb=0.575 (logloss=0.726)
Season 2021-22: naive=0.543  logreg=0.603 (logloss=0.672)  xgb=0.612 (logloss=0.673)
Season 2022-23: naive=0.581  logreg=0.586 (logloss=0.671)  xgb=0.572 (logloss=0.687)
Season 2023-24: naive=0.543  logreg=0.623 (logloss=0.642)  xgb=0.623 (logloss=0.650)
Season 2024-25: naive=0.548  logreg=0.662 (logloss=0.626)  xgb=0.634 (logloss=0.638)

=== Summary across folds ===
  test_season  naive_accuracy  logreg_accuracy  xgb_accuracy
0     2020-21        0.543233         0.584586      0.575188
1     2021-22        0.542834         0.602965      0.612026
2     2022-23        0.580725         0.586491      0.572488
3     2023-24        0.543210         0.623045      0.623045
4     2024-25        0.547560         0.661704      0.633581

Naive baseline:  mean=0.552  std=0.016
LogReg:          mean=0.612  std=0.032
XGBoost:         mean=0.603  std=0.028


In [15]:
"""
Phase 3c: Stress testing -- calibration check, subgroup performance
slicing, and feature importance. This tells us WHERE the model is
weak, not just THAT it's inconsistent.
"""

from sklearn.calibration import calibration_curve

def stress_test_calibration(df: pd.DataFrame, feature_cols: list, target_col: str = 'HOME_WIN'):
    """
    Trains on all seasons except the last, tests on the last season,
    and checks whether predicted probabilities match actual outcome
    frequencies (calibration) -- e.g. do games predicted at ~70%
    confidence actually get won ~70% of the time?
    """
    seasons_sorted = sorted(df['SEASON'].unique())
    train_seasons = seasons_sorted[:-1]
    test_season = seasons_sorted[-1]

    train_idx = df[df['SEASON'].isin(train_seasons)].index
    test_idx = df[df['SEASON'] == test_season].index

    X_train = df.loc[train_idx, feature_cols].fillna(df.loc[train_idx, feature_cols].median())
    y_train = df.loc[train_idx, target_col]
    X_test = df.loc[test_idx, feature_cols].fillna(df.loc[train_idx, feature_cols].median())
    y_test = df.loc[test_idx, target_col]

    xgb = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                         subsample=0.8, colsample_bytree=0.8,
                         eval_metric='logloss', random_state=42)
    xgb.fit(X_train, y_train)
    proba = xgb.predict_proba(X_test)[:, 1]

    # Calibration: bucket predictions into bins, compare predicted vs actual
    prob_true, prob_pred = calibration_curve(y_test, proba, n_bins=10, strategy='quantile')

    print("=== Calibration Check (predicted probability vs actual win rate) ===")
    print(f"{'Predicted':>10} {'Actual':>10} {'Diff':>8}")
    for pt, pp in zip(prob_true, prob_pred):
        diff = pt - pp
        flag = "  <-- off by >5pts" if abs(diff) > 0.05 else ""
        print(f"{pp:>10.3f} {pt:>10.3f} {diff:>8.3f}{flag}")

    return xgb, X_test, y_test, proba, test_idx


def stress_test_subgroups(df: pd.DataFrame, test_idx, y_test, proba):
    """
    Slices model performance by scenario -- does accuracy hold up
    for back-to-backs, close matchups, and away underdogs, or does
    the model quietly fail on specific game types?
    """
    test_df = df.loc[test_idx].copy()
    test_df['PRED_PROBA'] = proba
    test_df['PRED'] = (proba >= 0.5).astype(int)
    test_df['CORRECT'] = (test_df['PRED'] == y_test.values).astype(int)

    print("\n=== Subgroup Performance (test season) ===")

    # Back-to-back vs rested
    for label, mask in [
        ('Home team on B2B', test_df['IS_BACK_TO_BACK_HOME'] == 1),
        ('Away team on B2B', test_df['IS_BACK_TO_BACK_AWAY'] == 1),
        ('Neither on B2B', (test_df['IS_BACK_TO_BACK_HOME'] == 0) & (test_df['IS_BACK_TO_BACK_AWAY'] == 0)),
    ]:
        subset = test_df[mask]
        if len(subset) > 10:
            print(f"{label:<20} n={len(subset):>4}  acc={subset['CORRECT'].mean():.3f}")

    # Rating mismatch vs close matchup (using OFF_RATING_ROLL10 gap as proxy for "how different are these teams")
    test_df['RATING_GAP'] = (
        (test_df['OFF_RATING_ROLL10_HOME'] - test_df['DEF_RATING_ROLL10_AWAY']) -
        (test_df['OFF_RATING_ROLL10_AWAY'] - test_df['DEF_RATING_ROLL10_HOME'])
    ).abs()
    median_gap = test_df['RATING_GAP'].median()

    close = test_df[test_df['RATING_GAP'] <= median_gap]
    mismatch = test_df[test_df['RATING_GAP'] > median_gap]
    print(f"{'Close matchups':<20} n={len(close):>4}  acc={close['CORRECT'].mean():.3f}")
    print(f"{'Mismatched teams':<20} n={len(mismatch):>4}  acc={mismatch['CORRECT'].mean():.3f}")

    # High-confidence predictions only -- does the model know when it knows?
    high_conf = test_df[(test_df['PRED_PROBA'] >= 0.65) | (test_df['PRED_PROBA'] <= 0.35)]
    print(f"{'High-confidence preds':<20} n={len(high_conf):>4}  acc={high_conf['CORRECT'].mean():.3f}")


def stress_test_feature_importance(xgb_model, feature_cols):
    """
    Shows which features are actually driving predictions -- helps
    catch cases where the model is leaning heavily on a narrow,
    possibly spurious signal rather than broad team quality.
    """
    importances = pd.Series(xgb_model.feature_importances_, index=feature_cols)
    importances = importances.sort_values(ascending=False)

    print("\n=== Top 15 Feature Importances (XGBoost) ===")
    print(importances.head(15).to_string())


# --------------------------------------------------------------------------
# EXECUTION
# --------------------------------------------------------------------------
xgb_model, X_test, y_test, proba, test_idx = stress_test_calibration(model_df, feature_cols)
stress_test_subgroups(model_df, test_idx, y_test, proba)
stress_test_feature_importance(xgb_model, feature_cols)

=== Calibration Check (predicted probability vs actual win rate) ===
 Predicted     Actual     Diff
     0.271      0.273    0.002
     0.378      0.413    0.035
     0.448      0.405   -0.043
     0.502      0.455   -0.048
     0.548      0.537   -0.010
     0.589      0.567   -0.022
     0.636      0.661    0.025
     0.683      0.645   -0.038
     0.741      0.694   -0.047
     0.823      0.826    0.004

=== Subgroup Performance (test season) ===
Home team on B2B     n= 221  acc=0.602
Away team on B2B     n= 229  acc=0.620
Neither on B2B       n= 830  acc=0.641
Close matchups       n= 605  acc=0.635
Mismatched teams     n= 604  acc=0.632
High-confidence preds n= 535  acc=0.721

=== Top 15 Feature Importances (XGBoost) ===
WON_ROLL10_HOME             0.065153
WON_ROLL10_AWAY             0.048385
IS_BACK_TO_BACK_HOME        0.022473
OFF_RATING_ROLL10_HOME      0.022200
OFF_RATING_ROLL10_AWAY      0.021623
OPPONENT_PTS_ROLL10_HOME    0.021355
WON_ROLL5_HOME              0.018980
IS_BAC

In [16]:
"""
Phase 3d: Diagnose why 'close matchups' and 'mismatched teams' show
statistically identical accuracy -- is the RATING_GAP proxy flawed,
or does the model just not translate rating gaps into confidence?
"""

def diagnose_rating_gap(df: pd.DataFrame, test_idx, y_test, proba, feature_cols):
    test_df = df.loc[test_idx].copy()
    test_df['PRED_PROBA'] = proba
    test_df['PRED'] = (proba >= 0.5).astype(int)
    test_df['CORRECT'] = (test_df['PRED'] == y_test.values).astype(int)

    # --- Simpler, more direct rating gap: just compare team quality ---
    # (OFF_RATING - DEF_RATING) is a clean per-team "net rating" proxy.
    # The gap between home and away net ratings should be a MUCH more
    # direct measure of "how different are these two teams" than my
    # earlier cross-term formula.
    test_df['NET_RATING_HOME'] = test_df['OFF_RATING_ROLL10_HOME'] - test_df['DEF_RATING_ROLL10_HOME']
    test_df['NET_RATING_AWAY'] = test_df['OFF_RATING_ROLL10_AWAY'] - test_df['DEF_RATING_ROLL10_AWAY']
    test_df['NET_RATING_GAP'] = (test_df['NET_RATING_HOME'] - test_df['NET_RATING_AWAY']).abs()

    # --- Check 1: does prediction confidence actually scale with gap size? ---
    correlation = test_df['NET_RATING_GAP'].corr(
        (test_df['PRED_PROBA'] - 0.5).abs()  # distance from 50/50 = model's confidence
    )
    print(f"Correlation between NET_RATING_GAP and model confidence: {correlation:.3f}")
    print("(Should be clearly positive -- bigger gaps SHOULD produce more confident predictions)")

    # --- Check 2: accuracy by NET_RATING_GAP quintile ---
    test_df['GAP_QUINTILE'] = pd.qcut(test_df['NET_RATING_GAP'], 5, labels=[
        'Q1 (closest)', 'Q2', 'Q3', 'Q4', 'Q5 (biggest gap)'
    ])

    print("\n=== Accuracy by matchup gap quintile (using cleaner NET_RATING_GAP) ===")
    summary = test_df.groupby('GAP_QUINTILE', observed=True).agg(
        n=('CORRECT', 'size'),
        accuracy=('CORRECT', 'mean'),
        avg_confidence=('PRED_PROBA', lambda x: (x - 0.5).abs().mean())
    )
    print(summary)

    return test_df


# --------------------------------------------------------------------------
# EXECUTION
# --------------------------------------------------------------------------
diagnosed_df = diagnose_rating_gap(model_df, test_idx, y_test, proba, feature_cols)

Correlation between NET_RATING_GAP and model confidence: 0.475
(Should be clearly positive -- bigger gaps SHOULD produce more confident predictions)

=== Accuracy by matchup gap quintile (using cleaner NET_RATING_GAP) ===
                    n  accuracy  avg_confidence
GAP_QUINTILE                                   
Q1 (closest)      242  0.508264        0.099204
Q2                242  0.603306        0.101148
Q3                241  0.609959        0.133794
Q4                242  0.661157        0.160798
Q5 (biggest gap)  242  0.785124        0.222506


In [17]:
"""
Phase 2d: Compute travel distance between each team's consecutive
games using a static arena location lookup + haversine formula.
Leakage-safe by construction -- only ever looks at the PREVIOUS
game's location, same pattern as REST_DAYS.
"""

import numpy as np

# --------------------------------------------------------------------------
# Static arena locations (lat/long) -- doesn't change season to season
# except for rare relocations, negligible impact on travel distance calcs
# --------------------------------------------------------------------------
ARENA_LOCATIONS = {
    'ATL': (33.7573, -84.3963), 'BOS': (42.3662, -71.0621), 'BKN': (40.6826, -73.9754),
    'CHA': (35.2251, -80.8392), 'CHI': (41.8807, -87.6742), 'CLE': (41.4965, -81.6882),
    'DAL': (32.7905, -96.8103), 'DEN': (39.7487, -105.0077), 'DET': (42.3410, -83.0550),
    'GSW': (37.7680, -122.3877), 'HOU': (29.7508, -95.3621), 'IND': (39.7640, -86.1555),
    'LAC': (34.0430, -118.2673), 'LAL': (34.0430, -118.2673), 'MEM': (35.1382, -90.0505),
    'MIA': (25.7814, -80.1870), 'MIL': (43.0451, -87.9172), 'MIN': (44.9795, -93.2760),
    'NOP': (29.9490, -90.0821), 'NYK': (40.7505, -73.9934), 'OKC': (35.4634, -97.5151),
    'ORL': (28.5392, -81.3839), 'PHI': (39.9012, -75.1720), 'PHX': (33.4457, -112.0712),
    'POR': (45.5316, -122.6668), 'SAC': (38.5802, -121.4997), 'SAS': (29.4269, -98.4375),
    'TOR': (43.6435, -79.3791), 'UTA': (40.7683, -111.9011), 'WAS': (38.8981, -77.0209),
}


def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Great-circle distance in miles between two lat/long points.
    Vectorized -- works on numpy arrays, not just scalars.
    """
    R = 3958.8  # Earth radius in miles
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c


def add_travel_distance(team_games: pd.DataFrame) -> pd.DataFrame:
    """
    For each team-game row, determines WHERE that game was played
    (home team's arena), then computes the distance traveled since
    that team's PREVIOUS game -- leakage-safe, same shift-based
    pattern as REST_DAYS, and reset per season for the same reasons.
    """
    team_games = team_games.copy()

    # Game location = own arena if playing at home, else opponent's arena
    def get_game_location(row):
        team = row['TEAM_ABBREVIATION'] if row['IS_HOME'] == 1 else row['OPPONENT']
        return ARENA_LOCATIONS.get(team, (np.nan, np.nan))

    locations = team_games.apply(get_game_location, axis=1)
    team_games['GAME_LAT'] = [loc[0] for loc in locations]
    team_games['GAME_LON'] = [loc[1] for loc in locations]

    # Shift to get PREVIOUS game's location (within same team + season)
    grouped = team_games.groupby(['TEAM_ABBREVIATION', 'SEASON'])
    team_games['PREV_LAT'] = grouped['GAME_LAT'].shift(1)
    team_games['PREV_LON'] = grouped['GAME_LON'].shift(1)

    team_games['TRAVEL_DISTANCE'] = haversine_distance(
        team_games['PREV_LAT'], team_games['PREV_LON'],
        team_games['GAME_LAT'], team_games['GAME_LON']
    )

    # Season openers have no previous game -> fill with 0 (no travel
    # burden carried into a fresh season, consistent with REST_DAYS
    # logic of not penalizing/crediting off-season gaps)
    team_games['TRAVEL_DISTANCE'] = team_games['TRAVEL_DISTANCE'].fillna(0)

    team_games = team_games.drop(columns=['GAME_LAT', 'GAME_LON', 'PREV_LAT', 'PREV_LON'])

    return team_games


# --------------------------------------------------------------------------
# EXECUTION
# --------------------------------------------------------------------------
team_games = add_travel_distance(team_games)

print(f"team_games shape: {team_games.shape}")

# Sanity check: a team's travel distance should be 0 on consecutive
# HOME games (no travel), and a large number after a cross-country trip
check_cols = ['GAME_DATE', 'TEAM_ABBREVIATION', 'OPPONENT', 'IS_HOME', 'TRAVEL_DISTANCE', 'REST_DAYS']
print("\nLAL sample (first 10 games of 2023-24 season):")
lal_check = team_games[
    (team_games['TEAM_ABBREVIATION'] == 'LAL') &
    (team_games['SEASON'] == '2023-24')
].head(10)
print(lal_check[check_cols].to_string(index=False))

team_games shape: (14108, 70)

LAL sample (first 10 games of 2023-24 season):
 GAME_DATE TEAM_ABBREVIATION OPPONENT  IS_HOME  TRAVEL_DISTANCE  REST_DAYS
2023-10-24               LAL      DEN        0         0.000000        3.0
2023-10-26               LAL      PHX        1       830.767801        2.0
2023-10-29               LAL      SAC        0       361.413608        3.0
2023-10-30               LAL      ORL        1       361.413608        1.0
2023-11-01               LAL      LAC        1         0.000000        2.0
2023-11-04               LAL      ORL        0      2198.636737        3.0
2023-11-06               LAL      MIA        0       204.255953        2.0
2023-11-08               LAL      HOU        0       966.575497        2.0
2023-11-10               LAL      PHX        0      1014.652677        2.0
2023-11-12               LAL      POR        1       358.314453        2.0


In [18]:
"""
Phase 2e: Add win/loss streak length and home-specific / away-specific
rolling form. Both leakage-safe (shift-before-computing), reset per
season, consistent with all prior feature logic.
"""

def add_streak_features(team_games: pd.DataFrame) -> pd.DataFrame:
    """
    Current win/loss streak length going INTO this game (not including
    this game's own result). Positive = win streak, negative = loss streak.

    Computed via a standard "streak reset on change" trick: group
    consecutive identical WON values, then count position within group.
    """
    team_games = team_games.copy()
    team_games = team_games.sort_values(['TEAM_ABBREVIATION', 'SEASON', 'GAME_DATE']).reset_index(drop=True)

    def compute_streak(group):
        # Shift WON by 1 so the streak going INTO a game excludes that game's own result
        prior_won = group['WON'].shift(1)

        # Identify consecutive-run groups (changes whenever result flips)
        change = (prior_won != prior_won.shift(1)).cumsum()

        # Running count within each same-result run
        streak_len = prior_won.groupby(change).cumcount() + 1

        # Sign: positive for win streaks, negative for loss streaks
        signed_streak = np.where(prior_won == 1, streak_len,
                                  np.where(prior_won == 0, -streak_len, np.nan))
        return pd.Series(signed_streak, index=group.index)

    team_games['STREAK'] = team_games.groupby(
        ['TEAM_ABBREVIATION', 'SEASON'], group_keys=False
    ).apply(compute_streak)

    # No prior game -> neutral streak of 0
    team_games['STREAK'] = team_games['STREAK'].fillna(0)

    return team_games


def add_home_away_split_form(team_games: pd.DataFrame, window: int = 10) -> pd.DataFrame:
    """
    Rolling form computed SEPARATELY for home games and away games,
    rather than blended. E.g. a team's PTS_ROLL10 might be strong
    overall but their AWAY-specific form might be much weaker --
    this captures that split.

    Still leakage-safe: shift(1) before rolling, and critically the
    rolling window only advances across games of the SAME home/away
    type (so a team's "home form" only updates on home games).
    """
    team_games = team_games.copy()

    split_metrics = ['PTS', 'OPPONENT_PTS', 'WON']
    grouped = team_games.groupby(['TEAM_ABBREVIATION', 'SEASON', 'IS_HOME'], group_keys=False)

    for metric in split_metrics:
        col_name = f'{metric}_ROLL{window}_SPLIT'
        team_games[col_name] = grouped[metric].transform(
            lambda s: s.shift(1).rolling(window, min_periods=1).mean()
        )

    return team_games


# --------------------------------------------------------------------------
# EXECUTION
# --------------------------------------------------------------------------
team_games = add_streak_features(team_games)
team_games = add_home_away_split_form(team_games, window=10)

print(f"team_games shape: {team_games.shape}")

check_cols = ['GAME_DATE', 'TEAM_ABBREVIATION', 'OPPONENT', 'WON', 'STREAK',
              'IS_HOME', 'WON_ROLL10_SPLIT', 'PTS_ROLL10_SPLIT']
print("\nBOS sample (first 12 games of 2023-24 season):")
bos_check = team_games[
    (team_games['TEAM_ABBREVIATION'] == 'BOS') &
    (team_games['SEASON'] == '2023-24')
].head(12)
print(bos_check[check_cols].to_string(index=False))

team_games shape: (14108, 74)

BOS sample (first 12 games of 2023-24 season):
 GAME_DATE TEAM_ABBREVIATION OPPONENT  WON  STREAK  IS_HOME  WON_ROLL10_SPLIT  PTS_ROLL10_SPLIT
2023-10-25               BOS      NYK    1     0.0        0               NaN               NaN
2023-10-27               BOS      MIA    1     1.0        1               NaN               NaN
2023-10-30               BOS      WAS    1     2.0        0          1.000000        108.000000
2023-11-01               BOS      IND    1     3.0        1          1.000000        119.000000
2023-11-04               BOS      BKN    1     4.0        0          1.000000        117.000000
2023-11-06               BOS      MIN    0     5.0        0          1.000000        119.333333
2023-11-08               BOS      PHI    0    -1.0        0          0.750000        116.750000
2023-11-10               BOS      BKN    1    -2.0        1          1.000000        137.000000
2023-11-11               BOS      TOR    1     1.0        

In [21]:
def get_feature_columns(df: pd.DataFrame) -> list:
    """
    Whitelist approach: only columns matching known leakage-safe
    feature patterns are eligible. Updated to include STREAK,
    TRAVEL_DISTANCE, and home/away SPLIT features from Phase 2d/2e.
    """
    allowed_patterns = ['_ROLL5', '_ROLL10', 'REST_DAYS', 'IS_BACK_TO_BACK',
                         'STREAK', 'TRAVEL_DISTANCE', '_SPLIT']
    feature_cols = [c for c in df.columns
                     if any(pattern in c for pattern in allowed_patterns)]

    print(f"Selected {len(feature_cols)} leakage-safe feature columns.")
    return feature_cols

In [22]:
"""
Phase 2f: Re-run the HOME/AWAY merge with the expanded feature set,
then re-run the full walk-forward moneyline evaluation to measure
the actual impact of travel distance, streaks, and home/away splits.
"""

# Re-run the merge (same function from Phase 2c, now picks up the new columns automatically)
model_ready_df = merge_features_to_game_level(game_level_df, team_games)

print(f"model_ready_df shape: {model_ready_df.shape}")

# Re-run data prep (drop incomplete rolling rows) and rebuild feature list
model_df = prepare_modeling_data(model_ready_df)
feature_cols = get_feature_columns(model_df)

print(f"\nModeling dataset: {model_df.shape[0]} games, {len(feature_cols)} features")

# Confirm new features are present
new_feature_check = [c for c in feature_cols if 'STREAK' in c or 'TRAVEL' in c or 'SPLIT' in c]
print(f"New features included: {new_feature_check}")

# --------------------------------------------------------------------------
# Re-run walk-forward moneyline evaluation with expanded feature set
# --------------------------------------------------------------------------
print("\n=== Walk-forward results WITH new features (travel, streak, home/away split) ===")
moneyline_results_v2 = run_walk_forward_moneyline(model_df, feature_cols)

print("\n=== Summary comparison ===")
print("BEFORE (Phase 3b):")
print(f"  LogReg:  mean=0.612  std=0.032")
print(f"  XGBoost: mean=0.603  std=0.028")
print("\nAFTER (with new features):")
print(f"  LogReg:  mean={moneyline_results_v2['logreg_accuracy'].mean():.3f}  std={moneyline_results_v2['logreg_accuracy'].std():.3f}")
print(f"  XGBoost: mean={moneyline_results_v2['xgb_accuracy'].mean():.3f}  std={moneyline_results_v2['xgb_accuracy'].std():.3f}")

model_ready_df shape: (7054, 140)
Dropped 95 rows with incomplete rolling features (7054 -> 6959)
Selected 70 leakage-safe feature columns.

Modeling dataset: 6959 games, 70 features
New features included: ['TRAVEL_DISTANCE_HOME', 'STREAK_HOME', 'PTS_ROLL10_SPLIT_HOME', 'OPPONENT_PTS_ROLL10_SPLIT_HOME', 'WON_ROLL10_SPLIT_HOME', 'TRAVEL_DISTANCE_AWAY', 'STREAK_AWAY', 'PTS_ROLL10_SPLIT_AWAY', 'OPPONENT_PTS_ROLL10_SPLIT_AWAY', 'WON_ROLL10_SPLIT_AWAY']

=== Walk-forward results WITH new features (travel, streak, home/away split) ===
Season 2020-21: naive=0.543  logreg=0.570 (logloss=0.704)  xgb=0.587 (logloss=0.712)
Season 2021-22: naive=0.543  logreg=0.617 (logloss=0.667)  xgb=0.609 (logloss=0.678)
Season 2022-23: naive=0.581  logreg=0.602 (logloss=0.666)  xgb=0.602 (logloss=0.675)
Season 2023-24: naive=0.543  logreg=0.645 (logloss=0.630)  xgb=0.635 (logloss=0.634)
Season 2024-25: naive=0.548  logreg=0.639 (logloss=0.623)  xgb=0.617 (logloss=0.641)

=== Summary comparison ===
BEFORE (Phas

In [23]:
"""
Phase 3e: Hyperparameter tuning via walk-forward CV on seasons
2020-21 through 2023-24 ONLY. The final season (2024-25) is held out
completely from tuning -- used once at the end as an untouched final
evaluation, preserving the integrity of the stress test.
"""

from itertools import product

def tune_xgboost_walk_forward(df: pd.DataFrame, feature_cols: list,
                                target_col: str = 'HOME_WIN',
                                holdout_season: str = '2024-25'):
    """
    Grid search over a small, sensible XGBoost hyperparameter space.
    Each candidate config is evaluated across walk-forward folds that
    EXCLUDE the holdout_season entirely -- neither as train nor test --
    so the final evaluation on holdout_season remains untouched.
    """
    # Modest grid -- kept small deliberately. Wide grids on ~4 folds of
    # ~1,200 games each risk finding configs that fit fold-specific
    # noise rather than genuine signal.
    param_grid = {
        'max_depth': [3, 4, 5],
        'learning_rate': [0.03, 0.05, 0.08],
        'n_estimators': [150, 250],
        'min_child_weight': [1, 5],
    }

    keys = list(param_grid.keys())
    combos = list(product(*param_grid.values()))
    print(f"Testing {len(combos)} hyperparameter combinations...")

    tune_df = df[df['SEASON'] != holdout_season].reset_index(drop=True)

    results = []
    for combo in combos:
        params = dict(zip(keys, combo))
        fold_loglosses = []
        fold_accuracies = []

        for train_idx, test_idx, test_season, train_seasons in walk_forward_season_splits(tune_df):
            X_train = tune_df.loc[train_idx, feature_cols]
            y_train = tune_df.loc[train_idx, target_col]
            X_test = tune_df.loc[test_idx, feature_cols]
            y_test = tune_df.loc[test_idx, target_col]

            train_medians = X_train.median()
            X_train = X_train.fillna(train_medians)
            X_test = X_test.fillna(train_medians)

            model = XGBClassifier(
                **params, subsample=0.8, colsample_bytree=0.8,
                eval_metric='logloss', random_state=42
            )
            model.fit(X_train, y_train)
            proba = model.predict_proba(X_test)[:, 1]
            preds = (proba >= 0.5).astype(int)

            fold_loglosses.append(log_loss(y_test, proba))
            fold_accuracies.append(accuracy_score(y_test, preds))

        results.append({
            **params,
            'avg_logloss': np.mean(fold_loglosses),
            'avg_accuracy': np.mean(fold_accuracies),
            'std_accuracy': np.std(fold_accuracies)
        })

    results_df = pd.DataFrame(results).sort_values('avg_logloss')
    return results_df


# --------------------------------------------------------------------------
# EXECUTION: Tune (excluding 2024-25 entirely)
# --------------------------------------------------------------------------
tuning_results = tune_xgboost_walk_forward(model_df, feature_cols)

print("\n=== Top 10 hyperparameter configs (sorted by avg log loss) ===")
print(tuning_results.head(10).to_string(index=False))

best_params = tuning_results.iloc[0][['max_depth', 'learning_rate', 'n_estimators', 'min_child_weight']].to_dict()
best_params['max_depth'] = int(best_params['max_depth'])
best_params['n_estimators'] = int(best_params['n_estimators'])
best_params['min_child_weight'] = int(best_params['min_child_weight'])
print(f"\nBest config: {best_params}")

Testing 36 hyperparameter combinations...

=== Top 10 hyperparameter configs (sorted by avg log loss) ===
 max_depth  learning_rate  n_estimators  min_child_weight  avg_logloss  avg_accuracy  std_accuracy
         3           0.03           150                 1     0.657016      0.616564      0.018142
         3           0.03           150                 5     0.657032      0.614007      0.017718
         4           0.03           150                 5     0.660412      0.616302      0.021712
         3           0.03           250                 1     0.660442      0.612802      0.021652
         3           0.05           150                 5     0.660688      0.612771      0.019973
         3           0.03           250                 5     0.660713      0.612477      0.021764
         4           0.03           150                 1     0.661155      0.610096      0.021683
         5           0.03           150                 5     0.661457      0.612477      0.027766
   

In [24]:
"""
Final evaluation: train on ALL non-holdout seasons (2019-20 through
2023-24) using the tuned hyperparameters, evaluate ONCE on the
untouched 2024-25 holdout season.
"""

def final_holdout_evaluation(df: pd.DataFrame, feature_cols: list, best_params: dict,
                               target_col: str = 'HOME_WIN', holdout_season: str = '2024-25'):
    train_df = df[df['SEASON'] != holdout_season]
    test_df = df[df['SEASON'] == holdout_season]

    X_train = train_df[feature_cols]
    y_train = train_df[target_col]
    X_test = test_df[feature_cols]
    y_test = test_df[target_col]

    train_medians = X_train.median()
    X_train = X_train.fillna(train_medians)
    X_test = X_test.fillna(train_medians)

    model = XGBClassifier(**best_params, subsample=0.8, colsample_bytree=0.8,
                           eval_metric='logloss', random_state=42)
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    preds = (proba >= 0.5).astype(int)

    naive_acc = accuracy_score(y_test, np.ones(len(y_test)))

    print(f"=== FINAL HOLDOUT EVALUATION: {holdout_season} (never used in tuning) ===")
    print(f"Naive baseline accuracy: {naive_acc:.3f}")
    print(f"Tuned XGBoost accuracy:  {accuracy_score(y_test, preds):.3f}")
    print(f"Tuned XGBoost log loss:  {log_loss(y_test, proba):.3f}")
    print(f"Tuned XGBoost precision: {precision_score(y_test, preds):.3f}")
    print(f"Tuned XGBoost recall:    {recall_score(y_test, preds):.3f}")
    print(f"Tuned XGBoost Brier:     {brier_score_loss(y_test, proba):.3f}")

    return model, X_test, y_test, proba


final_model, X_test_final, y_test_final, proba_final = final_holdout_evaluation(
    model_df, feature_cols, best_params
)

=== FINAL HOLDOUT EVALUATION: 2024-25 (never used in tuning) ===
Naive baseline accuracy: 0.548
Tuned XGBoost accuracy:  0.643
Tuned XGBoost log loss:  0.635
Tuned XGBoost precision: 0.646
Tuned XGBoost recall:    0.769
Tuned XGBoost Brier:     0.222


In [25]:
"""
Phase 3f: Baseline regression models (Linear Regression + XGBoost
Regressor) for point spread prediction, walk-forward evaluated on
the same season boundaries and leakage-safe feature set.
"""

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor


def run_walk_forward_spread(df: pd.DataFrame, feature_cols: list, target_col: str = 'POINT_DIFF'):
    """
    Trains Linear Regression and XGBoost Regressor on each walk-forward
    fold. Also computes a naive baseline: predicting the historical
    AVERAGE home margin (computed from TRAINING data only) for every
    game -- the regression equivalent of "home team always wins."
    """
    results = []

    for train_idx, test_idx, test_season, train_seasons in walk_forward_season_splits(df):
        X_train = df.loc[train_idx, feature_cols]
        y_train = df.loc[train_idx, target_col]
        X_test = df.loc[test_idx, feature_cols]
        y_test = df.loc[test_idx, target_col]

        train_medians = X_train.median()
        X_train = X_train.fillna(train_medians)
        X_test = X_test.fillna(train_medians)

        # --- Naive baseline: predict the training set's average home margin ---
        naive_pred_value = y_train.mean()
        naive_preds = np.full(len(y_test), naive_pred_value)
        naive_mae = mean_absolute_error(y_test, naive_preds)

        # --- Linear Regression ---
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        linreg = LinearRegression()
        linreg.fit(X_train_scaled, y_train)
        linreg_preds = linreg.predict(X_test_scaled)

        # --- XGBoost Regressor (using the tuned hyperparameter direction
        # from moneyline as a reasonable starting point, not blindly reused) ---
        xgb_reg = XGBRegressor(
            n_estimators=150, max_depth=3, learning_rate=0.03,
            min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
            random_state=42
        )
        xgb_reg.fit(X_train, y_train)
        xgb_preds = xgb_reg.predict(X_test)

        # Directional accuracy: did we at least get the WINNER right,
        # even if the margin was off? Useful cross-check against the
        # moneyline classifier's accuracy.
        actual_home_win = (y_test > 0).astype(int)
        linreg_direction = (linreg_preds > 0).astype(int)
        xgb_direction = (xgb_preds > 0).astype(int)

        fold_result = {
            'test_season': test_season,
            'naive_mae': naive_mae,
            'linreg_mae': mean_absolute_error(y_test, linreg_preds),
            'linreg_rmse': np.sqrt(mean_squared_error(y_test, linreg_preds)),
            'linreg_direction_acc': accuracy_score(actual_home_win, linreg_direction),
            'xgb_mae': mean_absolute_error(y_test, xgb_preds),
            'xgb_rmse': np.sqrt(mean_squared_error(y_test, xgb_preds)),
            'xgb_direction_acc': accuracy_score(actual_home_win, xgb_direction),
        }
        results.append(fold_result)

        print(f"Season {test_season}: naive_mae={naive_mae:.2f}  "
              f"linreg_mae={fold_result['linreg_mae']:.2f} (dir_acc={fold_result['linreg_direction_acc']:.3f})  "
              f"xgb_mae={fold_result['xgb_mae']:.2f} (dir_acc={fold_result['xgb_direction_acc']:.3f})")

    return pd.DataFrame(results)


# --------------------------------------------------------------------------
# EXECUTION
# --------------------------------------------------------------------------
spread_results = run_walk_forward_spread(model_df, feature_cols)

print("\n=== Summary across folds ===")
print(spread_results[['test_season', 'naive_mae', 'linreg_mae', 'xgb_mae']])
print(f"\nNaive baseline MAE:  mean={spread_results['naive_mae'].mean():.2f}  std={spread_results['naive_mae'].std():.2f}")
print(f"LinReg MAE:          mean={spread_results['linreg_mae'].mean():.2f}  std={spread_results['linreg_mae'].std():.2f}")
print(f"XGBoost MAE:         mean={spread_results['xgb_mae'].mean():.2f}  std={spread_results['xgb_mae'].std():.2f}")
print(f"\nLinReg direction accuracy:  mean={spread_results['linreg_direction_acc'].mean():.3f}")
print(f"XGBoost direction accuracy: mean={spread_results['xgb_direction_acc'].mean():.3f}")

Season 2020-21: naive_mae=12.09  linreg_mae=11.76 (dir_acc=0.597)  xgb_mae=11.61 (dir_acc=0.592)
Season 2021-22: naive_mae=12.22  linreg_mae=11.58 (dir_acc=0.619)  xgb_mae=11.58 (dir_acc=0.613)
Season 2022-23: naive_mae=10.98  linreg_mae=10.56 (dir_acc=0.591)  xgb_mae=10.49 (dir_acc=0.610)
Season 2023-24: naive_mae=12.49  linreg_mae=11.36 (dir_acc=0.649)  xgb_mae=11.35 (dir_acc=0.652)
Season 2024-25: naive_mae=12.55  linreg_mae=11.32 (dir_acc=0.643)  xgb_mae=11.45 (dir_acc=0.631)

=== Summary across folds ===
  test_season  naive_mae  linreg_mae    xgb_mae
0     2020-21  12.093381   11.762863  11.611818
1     2021-22  12.222166   11.583436  11.577372
2     2022-23  10.978495   10.560578  10.493629
3     2023-24  12.487119   11.364824  11.345631
4     2024-25  12.552947   11.317718  11.448970

Naive baseline MAE:  mean=12.07  std=0.64
LinReg MAE:          mean=11.32  std=0.46
XGBoost MAE:         mean=11.30  std=0.46

LinReg direction accuracy:  mean=0.620
XGBoost direction accuracy: me

In [26]:
"""
Phase 3g: Totals (TOTAL_PTS) regression, same walk-forward structure
as Spread. Naive baseline here = training set's average total points.
"""

def run_walk_forward_totals(df: pd.DataFrame, feature_cols: list, target_col: str = 'TOTAL_PTS'):
    results = []

    for train_idx, test_idx, test_season, train_seasons in walk_forward_season_splits(df):
        X_train = df.loc[train_idx, feature_cols]
        y_train = df.loc[train_idx, target_col]
        X_test = df.loc[test_idx, feature_cols]
        y_test = df.loc[test_idx, target_col]

        train_medians = X_train.median()
        X_train = X_train.fillna(train_medians)
        X_test = X_test.fillna(train_medians)

        naive_pred_value = y_train.mean()
        naive_mae = mean_absolute_error(y_test, np.full(len(y_test), naive_pred_value))

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        linreg = LinearRegression()
        linreg.fit(X_train_scaled, y_train)
        linreg_preds = linreg.predict(X_test_scaled)

        xgb_reg = XGBRegressor(
            n_estimators=150, max_depth=3, learning_rate=0.03,
            min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
            random_state=42
        )
        xgb_reg.fit(X_train, y_train)
        xgb_preds = xgb_reg.predict(X_test)

        fold_result = {
            'test_season': test_season,
            'naive_mae': naive_mae,
            'linreg_mae': mean_absolute_error(y_test, linreg_preds),
            'linreg_rmse': np.sqrt(mean_squared_error(y_test, linreg_preds)),
            'xgb_mae': mean_absolute_error(y_test, xgb_preds),
            'xgb_rmse': np.sqrt(mean_squared_error(y_test, xgb_preds)),
        }
        results.append(fold_result)

        print(f"Season {test_season}: naive_mae={naive_mae:.2f}  "
              f"linreg_mae={fold_result['linreg_mae']:.2f}  xgb_mae={fold_result['xgb_mae']:.2f}")

    return pd.DataFrame(results)


# --------------------------------------------------------------------------
# EXECUTION
# --------------------------------------------------------------------------
totals_results = run_walk_forward_totals(model_df, feature_cols)

print("\n=== Summary across folds ===")
print(totals_results[['test_season', 'naive_mae', 'linreg_mae', 'xgb_mae']])
print(f"\nNaive baseline MAE:  mean={totals_results['naive_mae'].mean():.2f}  std={totals_results['naive_mae'].std():.2f}")
print(f"LinReg MAE:          mean={totals_results['linreg_mae'].mean():.2f}  std={totals_results['linreg_mae'].std():.2f}")
print(f"XGBoost MAE:         mean={totals_results['xgb_mae'].mean():.2f}  std={totals_results['xgb_mae'].std():.2f}")

Season 2020-21: naive_mae=15.77  linreg_mae=15.40  xgb_mae=15.23
Season 2021-22: naive_mae=16.00  linreg_mae=14.94  xgb_mae=14.94
Season 2022-23: naive_mae=16.11  linreg_mae=14.89  xgb_mae=14.87
Season 2023-24: naive_mae=16.31  linreg_mae=14.83  xgb_mae=14.92
Season 2024-25: naive_mae=15.97  linreg_mae=15.02  xgb_mae=15.03

=== Summary across folds ===
  test_season  naive_mae  linreg_mae    xgb_mae
0     2020-21  15.772857   15.397647  15.232068
1     2021-22  16.000578   14.943979  14.940505
2     2022-23  16.112924   14.892035  14.872355
3     2023-24  16.311031   14.830874  14.919827
4     2024-25  15.971471   15.021465  15.032388

Naive baseline MAE:  mean=16.03  std=0.20
LinReg MAE:          mean=15.02  std=0.22
XGBoost MAE:         mean=15.00  std=0.14


In [27]:
"""
Phase 4a: Fetch a given date's schedule, then compute each team's
live rolling features by pulling their recent game logs fresh from
nba_api -- mirroring the exact same leakage-safe logic from Phase 2,
but computed on-demand rather than from our pre-built historical table.
"""

from nba_api.stats.endpoints import scoreboardv2, teamgamelog
from nba_api.stats.static import teams as nba_teams_static

TEAM_ID_TO_ABBR = {t['id']: t['abbreviation'] for t in nba_teams_static.get_teams()}
TEAM_ABBR_TO_ID = {t['abbreviation']: t['id'] for t in nba_teams_static.get_teams()}


def fetch_schedule_for_date(game_date: str) -> pd.DataFrame:
    """
    Fetches the scheduled games for a given date (format: 'YYYY-MM-DD').
    Returns home/away team abbreviations for each matchup.
    """
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            sb = scoreboardv2.ScoreboardV2(game_date=game_date, timeout=API_TIMEOUT)
            games = sb.get_data_frames()[0]  # GameHeader
            time.sleep(REQUEST_DELAY)

            games['HOME_TEAM_ABBR'] = games['HOME_TEAM_ID'].map(TEAM_ID_TO_ABBR)
            games['AWAY_TEAM_ABBR'] = games['VISITOR_TEAM_ID'].map(TEAM_ID_TO_ABBR)

            return games[['GAME_ID', 'GAME_DATE_EST', 'HOME_TEAM_ABBR', 'AWAY_TEAM_ABBR']]
        except Exception as e:
            print(f"  [Attempt {attempt}/{MAX_RETRIES}] Schedule fetch failed: {e}")
            time.sleep(REQUEST_DELAY * (2 ** (attempt - 1)))
    raise RuntimeError(f"Failed to fetch schedule for {game_date}")


def fetch_team_recent_games(team_abbr: str, as_of_date: str, season: str, lookback: int = 15) -> pd.DataFrame:
    """
    Fetches a team's game log for the given season, filtered to games
    STRICTLY BEFORE as_of_date -- this is the live-inference equivalent
    of the shift(1) we used historically. We only ever use games that
    already happened relative to the game we're predicting.
    """
    team_id = TEAM_ABBR_TO_ID.get(team_abbr)
    if team_id is None:
        raise ValueError(f"Unknown team abbreviation: {team_abbr}")

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            log = teamgamelog.TeamGameLog(team_id=team_id, season=season, timeout=API_TIMEOUT)
            df = log.get_data_frames()[0]
            time.sleep(REQUEST_DELAY)

            df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
            cutoff = pd.to_datetime(as_of_date)

            # STRICT inequality -- excludes the target game itself, mirrors shift(1)
            df = df[df['GAME_DATE'] < cutoff].sort_values('GAME_DATE', ascending=False)

            return df.head(lookback)
        except Exception as e:
            print(f"  [Attempt {attempt}/{MAX_RETRIES}] Game log fetch failed for {team_abbr}: {e}")
            time.sleep(REQUEST_DELAY * (2 ** (attempt - 1)))
    raise RuntimeError(f"Failed to fetch recent games for {team_abbr}")


def compute_live_rolling_features(recent_games: pd.DataFrame, windows=(5, 10)) -> dict:
    """
    Computes the SAME rolling features as Phase 2, but from a team's
    fetched recent-games log rather than our historical team_games table.
    recent_games is already sorted most-recent-first and already
    excludes the target game (via the strict date filter above), so
    NO additional shift is needed here -- the exclusion already happened
    at the fetch stage.
    """
    features = {}

    # nba_api's TeamGameLog uses slightly different column names than
    # LeagueGameLog -- map them to match our training feature names
    col_map = {'PTS': 'PTS', 'REB': 'REB', 'AST': 'AST', 'TOV': 'TOV',
               'STL': 'STL', 'BLK': 'BLK', 'FG_PCT': 'FG_PCT',
               'FG3_PCT': 'FG3_PCT', 'FT_PCT': 'FT_PCT'}

    for window in windows:
        window_games = recent_games.head(window)
        for train_col, live_col in col_map.items():
            if live_col in window_games.columns and len(window_games) > 0:
                features[f'{train_col}_ROLL{window}'] = window_games[live_col].mean()
            else:
                features[f'{train_col}_ROLL{window}'] = np.nan

        # WON_ROLLx
        if 'WL' in window_games.columns and len(window_games) > 0:
            features[f'WON_ROLL{window}'] = (window_games['WL'] == 'W').mean()
        else:
            features[f'WON_ROLL{window}'] = np.nan

        # OPPONENT_PTS_ROLLx requires opponent score, which TeamGameLog doesn't
        # give directly -- approximate via PLUS_MINUS: OPP_PTS = PTS - PLUS_MINUS
        if 'PLUS_MINUS' in window_games.columns and len(window_games) > 0:
            opp_pts = window_games['PTS'] - window_games['PLUS_MINUS']
            features[f'OPPONENT_PTS_ROLL{window}'] = opp_pts.mean()
        else:
            features[f'OPPONENT_PTS_ROLL{window}'] = np.nan

    # Rest days: days since most recent game in the fetched log
    if len(recent_games) > 0:
        last_game_date = recent_games['GAME_DATE'].max()
        # NOTE: as_of_date is injected by the caller via closure/param in practice;
        # here we just compute days since last known game as a placeholder,
        # actual REST_DAYS is finalized in the calling function where as_of_date is known.
        features['_LAST_GAME_DATE'] = last_game_date
    else:
        features['_LAST_GAME_DATE'] = None

    return features


# --------------------------------------------------------------------------
# TEST: fetch a past date's schedule as a validation run
# --------------------------------------------------------------------------
TEST_DATE = '2024-01-15'  # a past mid-season date, safely within our historical data
TEST_SEASON = '2023-24'

schedule = fetch_schedule_for_date(TEST_DATE)
print(f"Games scheduled on {TEST_DATE}:")
print(schedule)

Games scheduled on 2024-01-15:
       GAME_ID        GAME_DATE_EST HOME_TEAM_ABBR AWAY_TEAM_ABBR
0   0022300555  2024-01-15T00:00:00            PHI            HOU
1   0022300556  2024-01-15T00:00:00            DAL            NOP
2   0022300557  2024-01-15T00:00:00            NYK            ORL
3   0022300558  2024-01-15T00:00:00            WAS            DET
4   0022300559  2024-01-15T00:00:00            ATL            SAS
5   0022300560  2024-01-15T00:00:00            MEM            GSW
6   0022300561  2024-01-15T00:00:00            CLE            CHI
7   0022300562  2024-01-15T00:00:00            BKN            MIA
8   0022300563  2024-01-15T00:00:00            TOR            BOS
9   0022300564  2024-01-15T00:00:00            UTA            IND
10  0022300565  2024-01-15T00:00:00            LAL            OKC


In [28]:
"""
Phase 4b: Assemble live features for every scheduled game, then
generate Moneyline, Spread, and Totals predictions using our
trained models.
"""

def build_live_game_features(schedule: pd.DataFrame, as_of_date: str, season: str) -> pd.DataFrame:
    """
    For every scheduled game, fetches both teams' recent games,
    computes rolling features, and assembles a HOME/AWAY feature
    row matching the training feature schema.
    """
    all_rows = []

    for _, game in schedule.iterrows():
        home_abbr = game['HOME_TEAM_ABBR']
        away_abbr = game['AWAY_TEAM_ABBR']

        print(f"Fetching recent games: {home_abbr} vs {away_abbr}...")
        home_recent = fetch_team_recent_games(home_abbr, as_of_date, season)
        away_recent = fetch_team_recent_games(away_abbr, as_of_date, season)

        home_feats = compute_live_rolling_features(home_recent)
        away_feats = compute_live_rolling_features(away_recent)

        # Rest days: days since each team's last game before as_of_date.
        # Missing -> season-opener fallback of 3, consistent with training logic.
        cutoff = pd.to_datetime(as_of_date)

        def rest_days_from(last_date):
            if last_date is None or pd.isna(last_date):
                return 3.0
            return float((cutoff - last_date).days)

        home_rest = rest_days_from(home_feats.pop('_LAST_GAME_DATE'))
        away_rest = rest_days_from(away_feats.pop('_LAST_GAME_DATE'))

        row = {'GAME_ID': game['GAME_ID'], 'HOME_TEAM': home_abbr, 'AWAY_TEAM': away_abbr}

        for k, v in home_feats.items():
            row[f'{k}_HOME'] = v
        for k, v in away_feats.items():
            row[f'{k}_AWAY'] = v

        row['REST_DAYS_HOME'] = home_rest
        row['REST_DAYS_AWAY'] = away_rest
        row['IS_BACK_TO_BACK_HOME'] = int(home_rest <= 1)
        row['IS_BACK_TO_BACK_AWAY'] = int(away_rest <= 1)

        all_rows.append(row)

    return pd.DataFrame(all_rows)


def generate_predictions(live_features: pd.DataFrame, feature_cols: list,
                          moneyline_model, spread_model=None, totals_model=None) -> pd.DataFrame:
    """
    Applies the trained models to live features. Any feature the live
    pipeline couldn't compute (missing/NaN) is filled with the TRAINING
    set's median -- same imputation strategy as backtesting, applied
    consistently rather than inventing a new rule for live data.
    """
    X_live = pd.DataFrame(index=live_features.index)
    missing_cols = []

    for col in feature_cols:
        if col in live_features.columns:
            X_live[col] = live_features[col]
        else:
            X_live[col] = np.nan
            missing_cols.append(col)

    if missing_cols:
        print(f"\nNOTE: {len(missing_cols)} training features not available live "
              f"(filled with training median): {missing_cols[:5]}{'...' if len(missing_cols) > 5 else ''}")

    train_medians = model_df[feature_cols].median()
    X_live = X_live.fillna(train_medians)

    results = live_features[['GAME_ID', 'HOME_TEAM', 'AWAY_TEAM']].copy()
    results['HOME_WIN_PROB'] = moneyline_model.predict_proba(X_live)[:, 1]
    results['PREDICTED_WINNER'] = np.where(results['HOME_WIN_PROB'] >= 0.5,
                                            results['HOME_TEAM'], results['AWAY_TEAM'])

    if spread_model is not None:
        results['PREDICTED_SPREAD'] = spread_model.predict(X_live)  # positive = home favored

    if totals_model is not None:
        results['PREDICTED_TOTAL'] = totals_model.predict(X_live)

    return results


# --------------------------------------------------------------------------
# EXECUTION: assemble live features + predict for our test date
# --------------------------------------------------------------------------
live_features = build_live_game_features(schedule, as_of_date=TEST_DATE, season=TEST_SEASON)
print(f"\nLive features shape: {live_features.shape}")

predictions = generate_predictions(
    live_features, feature_cols,
    moneyline_model=final_model  # our tuned XGBoost from Phase 3e
)

print("\n=== Predictions for", TEST_DATE, "===")
print(predictions[['HOME_TEAM', 'AWAY_TEAM', 'HOME_WIN_PROB', 'PREDICTED_WINNER']].to_string(index=False))

Fetching recent games: PHI vs HOU...
Fetching recent games: DAL vs NOP...
Fetching recent games: NYK vs ORL...
Fetching recent games: WAS vs DET...
Fetching recent games: ATL vs SAS...
Fetching recent games: MEM vs GSW...
Fetching recent games: CLE vs CHI...
Fetching recent games: BKN vs MIA...
Fetching recent games: TOR vs BOS...
Fetching recent games: UTA vs IND...
Fetching recent games: LAL vs OKC...

Live features shape: (11, 51)

NOTE: 22 training features not available live (filled with training median): ['OFF_RATING_ROLL5_HOME', 'DEF_RATING_ROLL5_HOME', 'POSSESSIONS_ROLL5_HOME', 'OFF_RATING_ROLL10_HOME', 'DEF_RATING_ROLL10_HOME']...

=== Predictions for 2024-01-15 ===
HOME_TEAM AWAY_TEAM  HOME_WIN_PROB PREDICTED_WINNER
      PHI       HOU       0.546585              PHI
      DAL       NOP       0.507102              DAL
      NYK       ORL       0.612307              NYK
      WAS       DET       0.542872              WAS
      ATL       SAS       0.471402              SAS
    

In [29]:
def compute_live_rolling_features(recent_games: pd.DataFrame, windows=(5, 10)) -> dict:
    """
    UPDATED: now also computes OFF_RATING / DEF_RATING / POSSESSIONS
    live, using the same formula as Phase 2b, instead of leaving them
    to be median-filled.
    """
    features = {}

    col_map = {'PTS': 'PTS', 'REB': 'REB', 'AST': 'AST', 'TOV': 'TOV',
               'STL': 'STL', 'BLK': 'BLK', 'FG_PCT': 'FG_PCT',
               'FG3_PCT': 'FG3_PCT', 'FT_PCT': 'FT_PCT'}

    # --- Per-game possessions/ratings computed FIRST, across the full fetched log ---
    games = recent_games.copy()
    if len(games) > 0 and 'PLUS_MINUS' in games.columns:
        games['OPPONENT_PTS'] = games['PTS'] - games['PLUS_MINUS']
        games['POSSESSIONS'] = games['FGA'] - games['OREB'] + games['TOV'] + (0.4 * games['FTA'])
        safe_poss = games['POSSESSIONS'].replace(0, np.nan)
        games['OFF_RATING'] = 100 * games['PTS'] / safe_poss
        games['DEF_RATING'] = 100 * games['OPPONENT_PTS'] / safe_poss
    else:
        games['OPPONENT_PTS'] = np.nan
        games['POSSESSIONS'] = np.nan
        games['OFF_RATING'] = np.nan
        games['DEF_RATING'] = np.nan

    for window in windows:
        window_games = games.head(window)

        for train_col, live_col in col_map.items():
            features[f'{train_col}_ROLL{window}'] = (
                window_games[live_col].mean() if live_col in window_games.columns and len(window_games) > 0 else np.nan
            )

        features[f'WON_ROLL{window}'] = (
            (window_games['WL'] == 'W').mean() if 'WL' in window_games.columns and len(window_games) > 0 else np.nan
        )
        features[f'OPPONENT_PTS_ROLL{window}'] = window_games['OPPONENT_PTS'].mean() if len(window_games) > 0 else np.nan
        features[f'POSSESSIONS_ROLL{window}'] = window_games['POSSESSIONS'].mean() if len(window_games) > 0 else np.nan
        features[f'OFF_RATING_ROLL{window}'] = window_games['OFF_RATING'].mean() if len(window_games) > 0 else np.nan
        features[f'DEF_RATING_ROLL{window}'] = window_games['DEF_RATING'].mean() if len(window_games) > 0 else np.nan

    features['_LAST_GAME_DATE'] = games['GAME_DATE'].max() if len(games) > 0 else None

    return features


# Re-run with the fixed function
live_features = build_live_game_features(schedule, as_of_date=TEST_DATE, season=TEST_SEASON)
predictions = generate_predictions(live_features, feature_cols, moneyline_model=final_model)

print("\n=== Predictions (with ratings now computed live) ===")
print(predictions[['HOME_TEAM', 'AWAY_TEAM', 'HOME_WIN_PROB', 'PREDICTED_WINNER']].to_string(index=False))

Fetching recent games: PHI vs HOU...
Fetching recent games: DAL vs NOP...
Fetching recent games: NYK vs ORL...
Fetching recent games: WAS vs DET...
Fetching recent games: ATL vs SAS...
Fetching recent games: MEM vs GSW...
Fetching recent games: CLE vs CHI...
Fetching recent games: BKN vs MIA...
Fetching recent games: TOR vs BOS...
Fetching recent games: UTA vs IND...
Fetching recent games: LAL vs OKC...

NOTE: 10 training features not available live (filled with training median): ['TRAVEL_DISTANCE_HOME', 'STREAK_HOME', 'PTS_ROLL10_SPLIT_HOME', 'OPPONENT_PTS_ROLL10_SPLIT_HOME', 'WON_ROLL10_SPLIT_HOME']...

=== Predictions (with ratings now computed live) ===
HOME_TEAM AWAY_TEAM  HOME_WIN_PROB PREDICTED_WINNER
      PHI       HOU       0.546585              PHI
      DAL       NOP       0.507102              DAL
      NYK       ORL       0.612307              NYK
      WAS       DET       0.542872              WAS
      ATL       SAS       0.471402              SAS
      MEM       GSW   

In [30]:
# Quick isolated test: fetch ONE team's recent games and inspect
# whether OFF_RATING/DEF_RATING actually get computed
test_recent = fetch_team_recent_games('PHI', TEST_DATE, TEST_SEASON)
print("Columns in fetched game log:", test_recent.columns.tolist())
print(f"\nHas FGA: {'FGA' in test_recent.columns}, Has OREB: {'OREB' in test_recent.columns}, Has FTA: {'FTA' in test_recent.columns}")

test_features = compute_live_rolling_features(test_recent)
print(f"\nOFF_RATING_ROLL5: {test_features.get('OFF_RATING_ROLL5')}")
print(f"DEF_RATING_ROLL5: {test_features.get('DEF_RATING_ROLL5')}")
print(f"POSSESSIONS_ROLL5: {test_features.get('POSSESSIONS_ROLL5')}")

Columns in fetched game log: ['Team_ID', 'Game_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS']

Has FGA: True, Has OREB: True, Has FTA: True

OFF_RATING_ROLL5: nan
DEF_RATING_ROLL5: nan
POSSESSIONS_ROLL5: nan


In [31]:
def compute_live_rolling_features(recent_games: pd.DataFrame, windows=(5, 10)) -> dict:
    """
    FIXED: POSSESSIONS and OFF_RATING no longer depend on PLUS_MINUS
    (which TeamGameLog doesn't provide) -- only OPPONENT_PTS/DEF_RATING
    genuinely need it, and we derive OPPONENT_PTS differently instead.
    """
    features = {}

    col_map = {'PTS': 'PTS', 'REB': 'REB', 'AST': 'AST', 'TOV': 'TOV',
               'STL': 'STL', 'BLK': 'BLK', 'FG_PCT': 'FG_PCT',
               'FG3_PCT': 'FG3_PCT', 'FT_PCT': 'FT_PCT'}

    games = recent_games.copy()

    if len(games) > 0:
        # POSSESSIONS and OFF_RATING only need FGA/OREB/TOV/FTA/PTS --
        # all confirmed present in TeamGameLog, no PLUS_MINUS needed.
        games['POSSESSIONS'] = games['FGA'] - games['OREB'] + games['TOV'] + (0.4 * games['FTA'])
        safe_poss = games['POSSESSIONS'].replace(0, np.nan)
        games['OFF_RATING'] = 100 * games['PTS'] / safe_poss

        # OPPONENT_PTS: TeamGameLog has no PLUS_MINUS, so we derive it
        # by parsing the opponent abbreviation out of MATCHUP and looking
        # up THAT team's PTS from the same GAME_ID via a fresh small fetch.
        # -- Simpler, more robust alternative: use PTS +/- an estimate is
        # not reliable without PLUS_MINUS, so instead we leave DEF_RATING
        # and OPPONENT_PTS as NaN (median-filled) for now rather than
        # guess incorrectly -- flagged below for your decision.
        games['OPPONENT_PTS'] = np.nan
        games['DEF_RATING'] = np.nan
    else:
        games['POSSESSIONS'] = np.nan
        games['OFF_RATING'] = np.nan
        games['OPPONENT_PTS'] = np.nan
        games['DEF_RATING'] = np.nan

    for window in windows:
        window_games = games.head(window)

        for train_col, live_col in col_map.items():
            features[f'{train_col}_ROLL{window}'] = (
                window_games[live_col].mean() if live_col in window_games.columns and len(window_games) > 0 else np.nan
            )

        features[f'WON_ROLL{window}'] = (
            (window_games['WL'] == 'W').mean() if 'WL' in window_games.columns and len(window_games) > 0 else np.nan
        )
        features[f'POSSESSIONS_ROLL{window}'] = window_games['POSSESSIONS'].mean() if len(window_games) > 0 else np.nan
        features[f'OFF_RATING_ROLL{window}'] = window_games['OFF_RATING'].mean() if len(window_games) > 0 else np.nan
        features[f'OPPONENT_PTS_ROLL{window}'] = window_games['OPPONENT_PTS'].mean() if len(window_games) > 0 else np.nan
        features[f'DEF_RATING_ROLL{window}'] = window_games['DEF_RATING'].mean() if len(window_games) > 0 else np.nan

    features['_LAST_GAME_DATE'] = games['GAME_DATE'].max() if len(games) > 0 else None

    return features

In [32]:
"""
Phase 4b (revised): Replace per-team TeamGameLog fetching with a
single season-wide LeagueGameLog fetch, then reuse our EXISTING,
already-validated Phase 2 functions (build_game_level_df,
build_team_games_long, add_possessions_and_ratings) to get correct
OPPONENT_PTS/OFF_RATING/DEF_RATING -- no schema mismatch, no guessing.
"""

def fetch_live_team_games(season: str, as_of_date: str) -> pd.DataFrame:
    """
    Fetches ALL teams' games for the season in ONE call, reuses our
    trusted Phase 2 pipeline to compute OPPONENT_PTS/ratings correctly,
    then filters to games strictly before as_of_date.
    """
    season_long_df = fetch_season_game_log(season)  # 1 API call, all teams
    season_game_level = build_game_level_df(season_long_df)  # reuses Phase 1 logic
    season_team_games = build_team_games_long(season_game_level)  # reuses Phase 2a logic
    season_team_games = add_possessions_and_ratings(season_team_games)  # reuses Phase 2b logic

    cutoff = pd.to_datetime(as_of_date)
    season_team_games = season_team_games[season_team_games['GAME_DATE'] < cutoff]

    return season_team_games.sort_values(['TEAM_ABBREVIATION', 'GAME_DATE'])


def compute_live_rolling_features_v2(team_recent: pd.DataFrame, windows=(5, 10)) -> dict:
    """
    Computes rolling features from a team's filtered slice of
    season_team_games (already has correct PTS, OPPONENT_PTS,
    OFF_RATING, DEF_RATING, POSSESSIONS columns from Phase 2 logic).
    """
    features = {}
    team_recent = team_recent.sort_values('GAME_DATE', ascending=False)

    roll_metrics = ['PTS', 'OPPONENT_PTS', 'FG_PCT', 'FG3_PCT', 'FT_PCT',
                     'REB', 'AST', 'TOV', 'STL', 'BLK', 'WON',
                     'OFF_RATING', 'DEF_RATING', 'POSSESSIONS']

    for window in windows:
        window_games = team_recent.head(window)
        for metric in roll_metrics:
            col_name = f'{metric}_ROLL{window}'
            features[col_name] = window_games[metric].mean() if len(window_games) > 0 and metric in window_games.columns else np.nan

    features['_LAST_GAME_DATE'] = team_recent['GAME_DATE'].max() if len(team_recent) > 0 else None
    return features


def build_live_game_features_v2(schedule: pd.DataFrame, as_of_date: str, season: str) -> pd.DataFrame:
    """
    Updated version: fetches season data ONCE (not per-team), then
    slices per team for feature computation. Dramatically fewer API
    calls than the original per-team TeamGameLog approach.
    """
    print(f"Fetching full season data for {season} (single API call)...")
    season_team_games = fetch_live_team_games(season, as_of_date)

    all_rows = []
    for _, game in schedule.iterrows():
        home_abbr, away_abbr = game['HOME_TEAM_ABBR'], game['AWAY_TEAM_ABBR']

        home_recent = season_team_games[season_team_games['TEAM_ABBREVIATION'] == home_abbr]
        away_recent = season_team_games[season_team_games['TEAM_ABBREVIATION'] == away_abbr]

        home_feats = compute_live_rolling_features_v2(home_recent)
        away_feats = compute_live_rolling_features_v2(away_recent)

        cutoff = pd.to_datetime(as_of_date)
        def rest_days_from(last_date):
            return 3.0 if last_date is None or pd.isna(last_date) else float((cutoff - last_date).days)

        home_rest = rest_days_from(home_feats.pop('_LAST_GAME_DATE'))
        away_rest = rest_days_from(away_feats.pop('_LAST_GAME_DATE'))

        row = {'GAME_ID': game['GAME_ID'], 'HOME_TEAM': home_abbr, 'AWAY_TEAM': away_abbr}
        row.update({f'{k}_HOME': v for k, v in home_feats.items()})
        row.update({f'{k}_AWAY': v for k, v in away_feats.items()})
        row['REST_DAYS_HOME'], row['REST_DAYS_AWAY'] = home_rest, away_rest
        row['IS_BACK_TO_BACK_HOME'] = int(home_rest <= 1)
        row['IS_BACK_TO_BACK_AWAY'] = int(away_rest <= 1)

        all_rows.append(row)

    return pd.DataFrame(all_rows)


# --------------------------------------------------------------------------
# EXECUTION
# --------------------------------------------------------------------------
live_features = build_live_game_features_v2(schedule, as_of_date=TEST_DATE, season=TEST_SEASON)
predictions = generate_predictions(live_features, feature_cols, moneyline_model=final_model)

print(f"\nMissing feature count check:")
missing_check = [c for c in feature_cols if c not in live_features.columns]
print(f"{len(missing_check)} missing: {missing_check}")

print("\n=== Predictions (with correct OFF_RATING/DEF_RATING) ===")
print(predictions[['HOME_TEAM', 'AWAY_TEAM', 'HOME_WIN_PROB', 'PREDICTED_WINNER']].to_string(index=False))

Fetching full season data for 2023-24 (single API call)...

NOTE: 10 training features not available live (filled with training median): ['TRAVEL_DISTANCE_HOME', 'STREAK_HOME', 'PTS_ROLL10_SPLIT_HOME', 'OPPONENT_PTS_ROLL10_SPLIT_HOME', 'WON_ROLL10_SPLIT_HOME']...

Missing feature count check:
10 missing: ['TRAVEL_DISTANCE_HOME', 'STREAK_HOME', 'PTS_ROLL10_SPLIT_HOME', 'OPPONENT_PTS_ROLL10_SPLIT_HOME', 'WON_ROLL10_SPLIT_HOME', 'TRAVEL_DISTANCE_AWAY', 'STREAK_AWAY', 'PTS_ROLL10_SPLIT_AWAY', 'OPPONENT_PTS_ROLL10_SPLIT_AWAY', 'WON_ROLL10_SPLIT_AWAY']

=== Predictions (with correct OFF_RATING/DEF_RATING) ===
HOME_TEAM AWAY_TEAM  HOME_WIN_PROB PREDICTED_WINNER
      PHI       HOU       0.582240              PHI
      DAL       NOP       0.499885              NOP
      NYK       ORL       0.664880              NYK
      WAS       DET       0.444086              DET
      ATL       SAS       0.438366              SAS
      MEM       GSW       0.550297              MEM
      CLE       CHI      

In [2]:
"""
Phase 4c: Wire up the remaining 10 features (STREAK, TRAVEL_DISTANCE,
home/away SPLIT form) for live inference, reusing the same Phase 2d/2e
functions we already validated -- applied to season_team_games instead
of our historical team_games table.
"""

def compute_live_rolling_features_v3(team_recent: pd.DataFrame, team_abbr: str, windows=(5, 10)) -> dict:
    """
    Extended version: also computes STREAK, TRAVEL_DISTANCE, and
    home/away SPLIT rolling form, using the same logic as
    add_streak_features / add_travel_distance / add_home_away_split_form
    from Phase 2, applied to this team's slice of live season data.
    """
    features = {}
    team_recent = team_recent.sort_values('GAME_DATE', ascending=False)

    roll_metrics = ['PTS', 'OPPONENT_PTS', 'FG_PCT', 'FG3_PCT', 'FT_PCT',
                     'REB', 'AST', 'TOV', 'STL', 'BLK', 'WON',
                     'OFF_RATING', 'DEF_RATING', 'POSSESSIONS']

    for window in windows:
        window_games = team_recent.head(window)
        for metric in roll_metrics:
            features[f'{metric}_ROLL{window}'] = (
                window_games[metric].mean() if len(window_games) > 0 and metric in window_games.columns else np.nan
            )

    # --- STREAK: walk backward from most recent game, count consecutive same-result games ---
    if len(team_recent) > 0:
        results = team_recent['WON'].values  # already sorted most-recent-first
        streak_len = 0
        streak_type = results[0]
        for r in results:
            if r == streak_type:
                streak_len += 1
            else:
                break
        features['STREAK'] = float(streak_len if streak_type == 1 else -streak_len)
    else:
        features['STREAK'] = 0.0

    # --- TRAVEL_DISTANCE: distance from most recent game's location to THIS game's location ---
    # Note: this needs to know where the LIVE game is being played, computed by the caller
    # since this function only has access to past games, not the upcoming one.
    if len(team_recent) > 0:
        last_game = team_recent.iloc[0]
        last_opponent = last_game['OPPONENT']
        last_was_home = last_game['IS_HOME'] == 1
        last_location_team = team_abbr if last_was_home else last_opponent
        features['_LAST_LOCATION'] = last_location_team
    else:
        features['_LAST_LOCATION'] = None

    # --- Home/away SPLIT form: filter to same IS_HOME type as the team's own status ---
    for is_home_val, suffix in [(1, 'home_split'), (0, 'away_split')]:
        split_games = team_recent[team_recent['IS_HOME'] == is_home_val].head(windows[-1])
        for metric in ['PTS', 'OPPONENT_PTS', 'WON']:
            features[f'{metric}_ROLL{windows[-1]}_SPLIT_{suffix}'] = (
                split_games[metric].mean() if len(split_games) > 0 else np.nan
            )

    features['_LAST_GAME_DATE'] = team_recent['GAME_DATE'].max() if len(team_recent) > 0 else None

    return features


def build_live_game_features_v3(schedule: pd.DataFrame, as_of_date: str, season: str) -> pd.DataFrame:
    """
    Final version: full feature parity with training, including
    STREAK, TRAVEL_DISTANCE, and home/away split form.
    """
    print(f"Fetching full season data for {season} (single API call)...")
    season_team_games = fetch_live_team_games(season, as_of_date)

    all_rows = []
    for _, game in schedule.iterrows():
        home_abbr, away_abbr = game['HOME_TEAM_ABBR'], game['AWAY_TEAM_ABBR']

        home_recent = season_team_games[season_team_games['TEAM_ABBREVIATION'] == home_abbr]
        away_recent = season_team_games[season_team_games['TEAM_ABBREVIATION'] == away_abbr]

        home_feats = compute_live_rolling_features_v3(home_recent, home_abbr)
        away_feats = compute_live_rolling_features_v3(away_recent, away_abbr)

        cutoff = pd.to_datetime(as_of_date)
        def rest_days_from(last_date):
            return 3.0 if last_date is None or pd.isna(last_date) else float((cutoff - last_date).days)

        home_rest = rest_days_from(home_feats.pop('_LAST_GAME_DATE'))
        away_rest = rest_days_from(away_feats.pop('_LAST_GAME_DATE'))

        # TRAVEL_DISTANCE: home team is playing AT home this game, so their
        # travel = distance from their last game location to their own arena.
        # Away team's travel = distance from their last game location to the HOME team's arena.
        home_last_loc = home_feats.pop('_LAST_LOCATION')
        away_last_loc = away_feats.pop('_LAST_LOCATION')

        def travel_to(last_loc_team, dest_team):
            if last_loc_team is None or last_loc_team not in ARENA_LOCATIONS or dest_team not in ARENA_LOCATIONS:
                return 0.0
            lat1, lon1 = ARENA_LOCATIONS[last_loc_team]
            lat2, lon2 = ARENA_LOCATIONS[dest_team]
            return float(haversine_distance(lat1, lon1, lat2, lon2))

        home_travel = travel_to(home_last_loc, home_abbr)  # dest = home team's own arena
        away_travel = travel_to(away_last_loc, home_abbr)  # dest = home team's arena (where this game is)

        row = {'GAME_ID': game['GAME_ID'], 'HOME_TEAM': home_abbr, 'AWAY_TEAM': away_abbr}

        # Rename SPLIT features to match training column names exactly:
        # training has e.g. PTS_ROLL10_SPLIT_HOME meaning "home team's HOME-game rolling form"
        for k, v in list(home_feats.items()):
            if 'SPLIT_home_split' in k:
                row[k.replace('_SPLIT_home_split', '_SPLIT') + '_HOME'] = v
            elif 'SPLIT_away_split' in k:
                pass  # not used -- home team's AWAY split isn't a training feature
            else:
                row[f'{k}_HOME'] = v

        for k, v in list(away_feats.items()):
            if 'SPLIT_away_split' in k:
                row[k.replace('_SPLIT_away_split', '_SPLIT') + '_AWAY'] = v
            elif 'SPLIT_home_split' in k:
                pass  # not used -- away team's HOME split isn't a training feature
            else:
                row[f'{k}_AWAY'] = v

        row['REST_DAYS_HOME'], row['REST_DAYS_AWAY'] = home_rest, away_rest
        row['IS_BACK_TO_BACK_HOME'] = int(home_rest <= 1)
        row['IS_BACK_TO_BACK_AWAY'] = int(away_rest <= 1)
        row['TRAVEL_DISTANCE_HOME'] = home_travel
        row['TRAVEL_DISTANCE_AWAY'] = away_travel
        row['STREAK_HOME'] = row.pop('STREAK_HOME', home_feats.get('STREAK', 0.0))
        row['STREAK_AWAY'] = row.pop('STREAK_AWAY', away_feats.get('STREAK', 0.0))

        all_rows.append(row)

    return pd.DataFrame(all_rows)


# --------------------------------------------------------------------------
# EXECUTION
# --------------------------------------------------------------------------
live_features = build_live_game_features_v3(schedule, as_of_date=TEST_DATE, season=TEST_SEASON)

missing_check = [c for c in feature_cols if c not in live_features.columns]
print(f"\nMissing feature count: {len(missing_check)}")
print(f"Missing: {missing_check}")

predictions = generate_predictions(live_features, feature_cols, moneyline_model=final_model)
print("\n=== Predictions (full feature parity) ===")
print(predictions[['HOME_TEAM', 'AWAY_TEAM', 'HOME_WIN_PROB', 'PREDICTED_WINNER']].to_string(index=False))

NameError: name 'schedule' is not defined

In [5]:
"""
================================================================================
NBA GAME PREDICTION PIPELINE -- CONSOLIDATED (Phases 1-4)
================================================================================
Run this top-to-bottom in a fresh kernel. Everything below has already been
validated step-by-step in our session -- this file exists so a kernel
restart doesn't force you to hunt through scattered messages again.

SECTIONS:
  Phase 1  -- Data ingestion (LeagueGameLog -> game-level wide dataframe)
  Phase 2  -- Feature engineering (rolling form, ratings, rest, travel,
              streaks, home/away splits) -- all leakage-safe, season-aware
  Phase 3  -- Modeling (walk-forward CV, Moneyline/Spread/Totals, tuning)
  Phase 4  -- Live inference (fetch a date's schedule, predict)
================================================================================
"""

import time
import warnings
import numpy as np
import pandas as pd
from itertools import product

from nba_api.stats.endpoints import leaguegamelog, scoreboardv2
from nba_api.stats.library.http import NBAStatsHTTP
from nba_api.stats.static import teams as nba_teams_static

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, log_loss,
                              brier_score_loss, mean_absolute_error, mean_squared_error)
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier, XGBRegressor

warnings.filterwarnings('ignore')

# ==============================================================================
# PHASE 1: DATA INGESTION
# ==============================================================================

SEASONS = ['2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']
SEASON_TYPE = 'Regular Season'
REQUEST_DELAY = 2.0
MAX_RETRIES = 5
API_TIMEOUT = 120

NBAStatsHTTP.headers.update({
    'Host': 'stats.nba.com',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                   '(KHTML, like Gecko) Chrome/120.0 Safari/537.36',
    'Referer': 'https://www.nba.com/',
    'Origin': 'https://www.nba.com',
    'Accept': 'application/json, text/plain, */*',
})

TEAM_ID_TO_ABBR = {t['id']: t['abbreviation'] for t in nba_teams_static.get_teams()}
TEAM_ABBR_TO_ID = {t['abbreviation']: t['id'] for t in nba_teams_static.get_teams()}


def fetch_season_game_log(season: str, season_type: str = SEASON_TYPE) -> pd.DataFrame:
    """Fetches ALL team game logs for a season in one API call."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            log = leaguegamelog.LeagueGameLog(
                season=season, season_type_all_star=season_type,
                player_or_team_abbreviation='T', timeout=API_TIMEOUT
            )
            df = log.get_data_frames()[0]
            df['SEASON'] = season
            time.sleep(REQUEST_DELAY)
            return df
        except Exception as e:
            wait = REQUEST_DELAY * (2 ** (attempt - 1))
            print(f"  [Attempt {attempt}/{MAX_RETRIES}] {season} failed: {type(e).__name__}: {e} -> retry in {wait:.0f}s")
            time.sleep(wait)
    raise RuntimeError(f"Failed to fetch season {season} after {MAX_RETRIES} attempts.")


def fetch_multi_season_logs(seasons: list) -> pd.DataFrame:
    all_logs = []
    for season in seasons:
        print(f"Fetching {season}...")
        season_df = fetch_season_game_log(season)
        print(f"  -> {len(season_df)} rows")
        all_logs.append(season_df)
    return pd.concat(all_logs, ignore_index=True)


def parse_matchup_column(df: pd.DataFrame) -> pd.DataFrame:
    """'BOS @ NYK' -> BOS away; 'BOS vs. NYK' -> BOS home."""
    df = df.copy()
    df['IS_HOME'] = df['MATCHUP'].str.contains('vs.', regex=False)
    df['OPPONENT_ABBREVIATION'] = np.where(
        df['IS_HOME'], df['MATCHUP'].str.split('vs. ').str[1], df['MATCHUP'].str.split('@ ').str[1]
    )
    return df


def build_game_level_df(long_df: pd.DataFrame) -> pd.DataFrame:
    """Reshapes team-level long data into one row per GAME_ID (HOME/AWAY paired)."""
    df = parse_matchup_column(long_df)
    home_df = df[df['IS_HOME']].copy()
    away_df = df[~df['IS_HOME']].copy()

    home_df = home_df.add_suffix('_HOME')
    away_df = away_df.add_suffix('_AWAY')

    home_df = home_df.rename(columns={'GAME_ID_HOME': 'GAME_ID', 'GAME_DATE_HOME': 'GAME_DATE', 'SEASON_HOME': 'SEASON'})
    away_df = away_df.rename(columns={'GAME_ID_AWAY': 'GAME_ID'})
    away_df = away_df.drop(columns=[c for c in away_df.columns if c.replace('_AWAY', '') in ['GAME_DATE', 'SEASON']])

    game_df = pd.merge(home_df, away_df, on='GAME_ID', how='inner')

    game_df['HOME_WIN'] = (game_df['WL_HOME'] == 'W').astype(int)
    game_df['POINT_DIFF'] = game_df['PTS_HOME'] - game_df['PTS_AWAY']
    game_df['TOTAL_PTS'] = game_df['PTS_HOME'] + game_df['PTS_AWAY']

    game_df['GAME_DATE'] = pd.to_datetime(game_df['GAME_DATE'])
    game_df = game_df.sort_values('GAME_DATE').reset_index(drop=True)
    return game_df


# ==============================================================================
# PHASE 2: FEATURE ENGINEERING
# ==============================================================================

def build_team_games_long(game_df: pd.DataFrame) -> pd.DataFrame:
    """One row per TEAM per GAME (both home and away perspectives stacked)."""
    base_cols = ['GAME_ID', 'GAME_DATE', 'SEASON']
    home_cols = [c for c in game_df.columns if c.endswith('_HOME')]
    stat_names = [c.replace('_HOME', '') for c in home_cols]

    home_rows = game_df[base_cols + home_cols].copy()
    home_rows.columns = base_cols + stat_names
    home_rows['IS_HOME'] = 1
    home_rows['OPPONENT'] = game_df['TEAM_ABBREVIATION_AWAY'].values
    home_rows['OPPONENT_PTS'] = game_df['PTS_AWAY'].values

    away_cols = [c for c in game_df.columns if c.endswith('_AWAY')]
    away_rows = game_df[base_cols + away_cols].copy()
    away_rows.columns = base_cols + [c.replace('_AWAY', '') for c in away_cols]
    away_rows['IS_HOME'] = 0
    away_rows['OPPONENT'] = game_df['TEAM_ABBREVIATION_HOME'].values
    away_rows['OPPONENT_PTS'] = game_df['PTS_HOME'].values

    team_games = pd.concat([home_rows, away_rows], ignore_index=True)
    team_games['WON'] = (team_games['WL'] == 'W').astype(int)
    team_games = team_games.sort_values(['TEAM_ABBREVIATION', 'GAME_DATE']).reset_index(drop=True)
    return team_games


def add_rolling_features(team_games: pd.DataFrame, windows=(5, 10)) -> pd.DataFrame:
    """Leakage-safe rolling averages, RESET PER SEASON (shift(1) before rolling)."""
    team_games = team_games.copy()
    roll_metrics = ['PTS', 'OPPONENT_PTS', 'FG_PCT', 'FG3_PCT', 'FT_PCT',
                     'REB', 'AST', 'TOV', 'STL', 'BLK', 'WON']
    roll_metrics = [m for m in roll_metrics if m in team_games.columns]
    grouped = team_games.groupby(['TEAM_ABBREVIATION', 'SEASON'], group_keys=False)

    for window in windows:
        for metric in roll_metrics:
            team_games[f'{metric}_ROLL{window}'] = grouped[metric].transform(
                lambda s: s.shift(1).rolling(window, min_periods=1).mean()
            )
    return team_games


def add_rest_days(team_games: pd.DataFrame) -> pd.DataFrame:
    """Rest days since previous game, RESET PER SEASON. First game of season -> 3 (default)."""
    team_games = team_games.copy()
    team_games['PREV_GAME_DATE'] = team_games.groupby(['TEAM_ABBREVIATION', 'SEASON'])['GAME_DATE'].shift(1)
    team_games['REST_DAYS'] = (team_games['GAME_DATE'] - team_games['PREV_GAME_DATE']).dt.days
    team_games['REST_DAYS'] = team_games['REST_DAYS'].fillna(3)
    team_games['IS_BACK_TO_BACK'] = (team_games['REST_DAYS'] <= 1).astype(int)
    return team_games.drop(columns=['PREV_GAME_DATE'])


def add_possessions_and_ratings(team_games: pd.DataFrame) -> pd.DataFrame:
    """POSSESSIONS = FGA - OREB + TOV + 0.4*FTA; OFF/DEF_RATING per 100 possessions."""
    team_games = team_games.copy()
    required = ['FGA', 'OREB', 'TOV', 'FTA', 'PTS', 'OPPONENT_PTS']
    missing = [c for c in required if c not in team_games.columns]
    if missing:
        raise ValueError(f"Missing required columns for possession estimate: {missing}")

    team_games['POSSESSIONS'] = team_games['FGA'] - team_games['OREB'] + team_games['TOV'] + (0.4 * team_games['FTA'])
    safe_poss = team_games['POSSESSIONS'].replace(0, np.nan)
    team_games['OFF_RATING'] = 100 * team_games['PTS'] / safe_poss
    team_games['DEF_RATING'] = 100 * team_games['OPPONENT_PTS'] / safe_poss
    return team_games


def add_rating_rolling_features(team_games: pd.DataFrame, windows=(5, 10)) -> pd.DataFrame:
    """Rolling OFF/DEF_RATING/POSSESSIONS, leakage-safe, reset per season."""
    team_games = team_games.copy()
    grouped = team_games.groupby(['TEAM_ABBREVIATION', 'SEASON'], group_keys=False)
    for window in windows:
        for metric in ['OFF_RATING', 'DEF_RATING', 'POSSESSIONS']:
            team_games[f'{metric}_ROLL{window}'] = grouped[metric].transform(
                lambda s: s.shift(1).rolling(window, min_periods=1).mean()
            )
    return team_games


ARENA_LOCATIONS = {
    'ATL': (33.7573, -84.3963), 'BOS': (42.3662, -71.0621), 'BKN': (40.6826, -73.9754),
    'CHA': (35.2251, -80.8392), 'CHI': (41.8807, -87.6742), 'CLE': (41.4965, -81.6882),
    'DAL': (32.7905, -96.8103), 'DEN': (39.7487, -105.0077), 'DET': (42.3410, -83.0550),
    'GSW': (37.7680, -122.3877), 'HOU': (29.7508, -95.3621), 'IND': (39.7640, -86.1555),
    'LAC': (34.0430, -118.2673), 'LAL': (34.0430, -118.2673), 'MEM': (35.1382, -90.0505),
    'MIA': (25.7814, -80.1870), 'MIL': (43.0451, -87.9172), 'MIN': (44.9795, -93.2760),
    'NOP': (29.9490, -90.0821), 'NYK': (40.7505, -73.9934), 'OKC': (35.4634, -97.5151),
    'ORL': (28.5392, -81.3839), 'PHI': (39.9012, -75.1720), 'PHX': (33.4457, -112.0712),
    'POR': (45.5316, -122.6668), 'SAC': (38.5802, -121.4997), 'SAS': (29.4269, -98.4375),
    'TOR': (43.6435, -79.3791), 'UTA': (40.7683, -111.9011), 'WAS': (38.8981, -77.0209),
}


def haversine_distance(lat1, lon1, lat2, lon2):
    """Great-circle distance in miles. Vectorized."""
    R = 3958.8
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return R * 2 * np.arcsin(np.sqrt(a))


def add_travel_distance(team_games: pd.DataFrame) -> pd.DataFrame:
    """Distance traveled since previous game, leakage-safe, reset per season."""
    team_games = team_games.copy()

    def get_game_location(row):
        team = row['TEAM_ABBREVIATION'] if row['IS_HOME'] == 1 else row['OPPONENT']
        return ARENA_LOCATIONS.get(team, (np.nan, np.nan))

    locations = team_games.apply(get_game_location, axis=1)
    team_games['GAME_LAT'] = [loc[0] for loc in locations]
    team_games['GAME_LON'] = [loc[1] for loc in locations]

    grouped = team_games.groupby(['TEAM_ABBREVIATION', 'SEASON'])
    team_games['PREV_LAT'] = grouped['GAME_LAT'].shift(1)
    team_games['PREV_LON'] = grouped['GAME_LON'].shift(1)

    team_games['TRAVEL_DISTANCE'] = haversine_distance(
        team_games['PREV_LAT'], team_games['PREV_LON'], team_games['GAME_LAT'], team_games['GAME_LON']
    )
    team_games['TRAVEL_DISTANCE'] = team_games['TRAVEL_DISTANCE'].fillna(0)
    return team_games.drop(columns=['GAME_LAT', 'GAME_LON', 'PREV_LAT', 'PREV_LON'])


def add_streak_features(team_games: pd.DataFrame) -> pd.DataFrame:
    """Signed win/loss streak length going INTO each game (excludes that game's own result)."""
    team_games = team_games.copy()
    team_games = team_games.sort_values(['TEAM_ABBREVIATION', 'SEASON', 'GAME_DATE']).reset_index(drop=True)

    def compute_streak(group):
        prior_won = group['WON'].shift(1)
        change = (prior_won != prior_won.shift(1)).cumsum()
        streak_len = prior_won.groupby(change).cumcount() + 1
        signed = np.where(prior_won == 1, streak_len, np.where(prior_won == 0, -streak_len, np.nan))
        return pd.Series(signed, index=group.index)

    team_games['STREAK'] = team_games.groupby(['TEAM_ABBREVIATION', 'SEASON'], group_keys=False).apply(compute_streak)
    team_games['STREAK'] = team_games['STREAK'].fillna(0)
    return team_games


def add_home_away_split_form(team_games: pd.DataFrame, window: int = 10) -> pd.DataFrame:
    """Rolling form computed separately for home games vs away games."""
    team_games = team_games.copy()
    split_metrics = ['PTS', 'OPPONENT_PTS', 'WON']
    grouped = team_games.groupby(['TEAM_ABBREVIATION', 'SEASON', 'IS_HOME'], group_keys=False)
    for metric in split_metrics:
        team_games[f'{metric}_ROLL{window}_SPLIT'] = grouped[metric].transform(
            lambda s: s.shift(1).rolling(window, min_periods=1).mean()
        )
    return team_games


def merge_features_to_game_level(game_df: pd.DataFrame, team_games: pd.DataFrame) -> pd.DataFrame:
    """Joins engineered team_games features back onto game_df as HOME_*/AWAY_* columns."""
    feature_cols_ = [c for c in team_games.columns if
                      'ROLL' in c or 'STREAK' in c or 'TRAVEL' in c or c in [
                          'REST_DAYS', 'IS_BACK_TO_BACK', 'OFF_RATING', 'DEF_RATING', 'POSSESSIONS']]

    merge_keys = ['GAME_ID', 'TEAM_ABBREVIATION']
    feature_slice = team_games[merge_keys + feature_cols_].copy()

    home_features = feature_slice.add_suffix('_HOME')
    home_features = home_features.rename(columns={'GAME_ID_HOME': 'GAME_ID', 'TEAM_ABBREVIATION_HOME': 'TEAM_ABBREVIATION_HOME'})
    merged = pd.merge(game_df, home_features, on=['GAME_ID', 'TEAM_ABBREVIATION_HOME'], how='left')

    away_features = feature_slice.add_suffix('_AWAY')
    away_features = away_features.rename(columns={'GAME_ID_AWAY': 'GAME_ID', 'TEAM_ABBREVIATION_AWAY': 'TEAM_ABBREVIATION_AWAY'})
    merged = pd.merge(merged, away_features, on=['GAME_ID', 'TEAM_ABBREVIATION_AWAY'], how='left')

    return merged


def run_full_feature_pipeline(game_level_df: pd.DataFrame) -> tuple:
    """Runs the entire Phase 2 sequence in the correct order. Returns (team_games, model_ready_df)."""
    tg = build_team_games_long(game_level_df)
    tg = add_rolling_features(tg, windows=(5, 10))
    tg = add_rest_days(tg)
    tg = add_possessions_and_ratings(tg)
    tg = add_rating_rolling_features(tg, windows=(5, 10))
    tg = add_travel_distance(tg)
    tg = add_streak_features(tg)
    tg = add_home_away_split_form(tg, window=10)

    model_ready = merge_features_to_game_level(game_level_df, tg)
    return tg, model_ready


def prepare_modeling_data(df: pd.DataFrame) -> pd.DataFrame:
    """Drops rows with incomplete rolling features (early-season games)."""
    df = df.copy()
    critical_cols = ['OFF_RATING_ROLL5_HOME', 'DEF_RATING_ROLL5_HOME',
                      'OFF_RATING_ROLL5_AWAY', 'DEF_RATING_ROLL5_AWAY']
    before = len(df)
    df = df.dropna(subset=critical_cols).reset_index(drop=True)
    print(f"Dropped {before - len(df)} rows with incomplete rolling features ({before} -> {len(df)})")
    return df


def get_feature_columns(df: pd.DataFrame) -> list:
    """Whitelist: only leakage-safe, pre-game-known feature columns."""
    allowed_patterns = ['_ROLL5', '_ROLL10', 'REST_DAYS', 'IS_BACK_TO_BACK',
                         'STREAK', 'TRAVEL_DISTANCE', '_SPLIT']
    feature_cols_ = [c for c in df.columns if any(p in c for p in allowed_patterns)]
    print(f"Selected {len(feature_cols_)} leakage-safe feature columns.")
    return feature_cols_


def walk_forward_season_splits(df: pd.DataFrame, season_col: str = 'SEASON'):
    """Yields (train_idx, test_idx, test_season, train_seasons) -- expanding window by season."""
    seasons_sorted = sorted(df[season_col].unique())
    for i in range(1, len(seasons_sorted)):
        train_seasons = seasons_sorted[:i]
        test_season = seasons_sorted[i]
        train_idx = df[df[season_col].isin(train_seasons)].index
        test_idx = df[df[season_col] == test_season].index
        yield train_idx, test_idx, test_season, train_seasons


# ==============================================================================
# PHASE 3: MODELING & BACKTESTING
# ==============================================================================

def run_walk_forward_moneyline(df: pd.DataFrame, feature_cols_: list, target_col: str = 'HOME_WIN'):
    """Walk-forward LogReg + XGBoost classification, with naive baseline."""
    results = []
    for train_idx, test_idx, test_season, train_seasons in walk_forward_season_splits(df):
        X_train, y_train = df.loc[train_idx, feature_cols_], df.loc[train_idx, target_col]
        X_test, y_test = df.loc[test_idx, feature_cols_], df.loc[test_idx, target_col]

        train_medians = X_train.median()
        X_train, X_test = X_train.fillna(train_medians), X_test.fillna(train_medians)

        naive_acc = accuracy_score(y_test, np.ones(len(y_test)))

        scaler = StandardScaler()
        X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)
        logreg = LogisticRegression(max_iter=1000, C=1.0)
        logreg.fit(X_train_s, y_train)
        logreg_proba = logreg.predict_proba(X_test_s)[:, 1]

        xgb = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42)
        xgb.fit(X_train, y_train)
        xgb_proba = xgb.predict_proba(X_test)[:, 1]

        fold_result = {
            'test_season': test_season, 'naive_accuracy': naive_acc,
            'logreg_accuracy': accuracy_score(y_test, (logreg_proba >= 0.5).astype(int)),
            'logreg_logloss': log_loss(y_test, logreg_proba),
            'xgb_accuracy': accuracy_score(y_test, (xgb_proba >= 0.5).astype(int)),
            'xgb_logloss': log_loss(y_test, xgb_proba),
        }
        results.append(fold_result)
        print(f"Season {test_season}: naive={naive_acc:.3f}  logreg={fold_result['logreg_accuracy']:.3f}  xgb={fold_result['xgb_accuracy']:.3f}")

    return pd.DataFrame(results)


def tune_xgboost_walk_forward(df: pd.DataFrame, feature_cols_: list, target_col: str = 'HOME_WIN', holdout_season: str = '2024-25'):
    """Grid search excluding holdout_season entirely (never touches final test season)."""
    param_grid = {'max_depth': [3, 4, 5], 'learning_rate': [0.03, 0.05, 0.08],
                  'n_estimators': [150, 250], 'min_child_weight': [1, 5]}
    keys = list(param_grid.keys())
    combos = list(product(*param_grid.values()))
    tune_df = df[df['SEASON'] != holdout_season].reset_index(drop=True)

    results = []
    for combo in combos:
        params = dict(zip(keys, combo))
        accs, lls = [], []
        for train_idx, test_idx, test_season, train_seasons in walk_forward_season_splits(tune_df):
            X_train, y_train = tune_df.loc[train_idx, feature_cols_], tune_df.loc[train_idx, target_col]
            X_test, y_test = tune_df.loc[test_idx, feature_cols_], tune_df.loc[test_idx, target_col]
            train_medians = X_train.median()
            X_train, X_test = X_train.fillna(train_medians), X_test.fillna(train_medians)

            model = XGBClassifier(**params, subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42)
            model.fit(X_train, y_train)
            proba = model.predict_proba(X_test)[:, 1]
            lls.append(log_loss(y_test, proba))
            accs.append(accuracy_score(y_test, (proba >= 0.5).astype(int)))

        results.append({**params, 'avg_logloss': np.mean(lls), 'avg_accuracy': np.mean(accs), 'std_accuracy': np.std(accs)})

    return pd.DataFrame(results).sort_values('avg_logloss')


def final_holdout_evaluation(df: pd.DataFrame, feature_cols_: list, best_params: dict,
                               target_col: str = 'HOME_WIN', holdout_season: str = '2024-25'):
    """Trains on all non-holdout seasons, evaluates ONCE on the untouched holdout season."""
    train_df = df[df['SEASON'] != holdout_season]
    test_df = df[df['SEASON'] == holdout_season]
    X_train, y_train = train_df[feature_cols_], train_df[target_col]
    X_test, y_test = test_df[feature_cols_], test_df[target_col]

    train_medians = X_train.median()
    X_train, X_test = X_train.fillna(train_medians), X_test.fillna(train_medians)

    model = XGBClassifier(**best_params, subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42)
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    preds = (proba >= 0.5).astype(int)

    print(f"=== FINAL HOLDOUT EVALUATION: {holdout_season} ===")
    print(f"Naive baseline accuracy: {accuracy_score(y_test, np.ones(len(y_test))):.3f}")
    print(f"Tuned XGBoost accuracy:  {accuracy_score(y_test, preds):.3f}")
    print(f"Tuned XGBoost log loss:  {log_loss(y_test, proba):.3f}")
    print(f"Tuned XGBoost precision: {precision_score(y_test, preds):.3f}")
    print(f"Tuned XGBoost recall:    {recall_score(y_test, preds):.3f}")
    print(f"Tuned XGBoost Brier:     {brier_score_loss(y_test, proba):.3f}")

    return model, X_test, y_test, proba


def run_walk_forward_spread(df: pd.DataFrame, feature_cols_: list, target_col: str = 'POINT_DIFF'):
    """Walk-forward LinReg + XGBoost regression for point spread."""
    results = []
    for train_idx, test_idx, test_season, train_seasons in walk_forward_season_splits(df):
        X_train, y_train = df.loc[train_idx, feature_cols_], df.loc[train_idx, target_col]
        X_test, y_test = df.loc[test_idx, feature_cols_], df.loc[test_idx, target_col]
        train_medians = X_train.median()
        X_train, X_test = X_train.fillna(train_medians), X_test.fillna(train_medians)

        naive_mae = mean_absolute_error(y_test, np.full(len(y_test), y_train.mean()))

        scaler = StandardScaler()
        X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)
        linreg = LinearRegression().fit(X_train_s, y_train)
        linreg_preds = linreg.predict(X_test_s)

        xgb_reg = XGBRegressor(n_estimators=150, max_depth=3, learning_rate=0.03,
                                min_child_weight=1, subsample=0.8, colsample_bytree=0.8, random_state=42)
        xgb_reg.fit(X_train, y_train)
        xgb_preds = xgb_reg.predict(X_test)

        actual_home_win = (y_test > 0).astype(int)
        fold_result = {
            'test_season': test_season, 'naive_mae': naive_mae,
            'linreg_mae': mean_absolute_error(y_test, linreg_preds),
            'linreg_direction_acc': accuracy_score(actual_home_win, (linreg_preds > 0).astype(int)),
            'xgb_mae': mean_absolute_error(y_test, xgb_preds),
            'xgb_direction_acc': accuracy_score(actual_home_win, (xgb_preds > 0).astype(int)),
        }
        results.append(fold_result)
        print(f"Season {test_season}: naive_mae={naive_mae:.2f}  linreg_mae={fold_result['linreg_mae']:.2f}  xgb_mae={fold_result['xgb_mae']:.2f}")

    return pd.DataFrame(results)


def run_walk_forward_totals(df: pd.DataFrame, feature_cols_: list, target_col: str = 'TOTAL_PTS'):
    """Walk-forward LinReg + XGBoost regression for game totals."""
    results = []
    for train_idx, test_idx, test_season, train_seasons in walk_forward_season_splits(df):
        X_train, y_train = df.loc[train_idx, feature_cols_], df.loc[train_idx, target_col]
        X_test, y_test = df.loc[test_idx, feature_cols_], df.loc[test_idx, target_col]
        train_medians = X_train.median()
        X_train, X_test = X_train.fillna(train_medians), X_test.fillna(train_medians)

        naive_mae = mean_absolute_error(y_test, np.full(len(y_test), y_train.mean()))

        scaler = StandardScaler()
        X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)
        linreg = LinearRegression().fit(X_train_s, y_train)
        linreg_preds = linreg.predict(X_test_s)

        xgb_reg = XGBRegressor(n_estimators=150, max_depth=3, learning_rate=0.03,
                                min_child_weight=1, subsample=0.8, colsample_bytree=0.8, random_state=42)
        xgb_reg.fit(X_train, y_train)
        xgb_preds = xgb_reg.predict(X_test)

        fold_result = {
            'test_season': test_season, 'naive_mae': naive_mae,
            'linreg_mae': mean_absolute_error(y_test, linreg_preds),
            'xgb_mae': mean_absolute_error(y_test, xgb_preds),
        }
        results.append(fold_result)
        print(f"Season {test_season}: naive_mae={naive_mae:.2f}  linreg_mae={fold_result['linreg_mae']:.2f}  xgb_mae={fold_result['xgb_mae']:.2f}")

    return pd.DataFrame(results)


# ==============================================================================
# PHASE 4: LIVE INFERENCE
# ==============================================================================

def fetch_schedule_for_date(game_date: str) -> pd.DataFrame:
    """Fetches scheduled games for a given date (format: 'YYYY-MM-DD')."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            sb = scoreboardv2.ScoreboardV2(game_date=game_date, timeout=API_TIMEOUT)
            games = sb.get_data_frames()[0]
            time.sleep(REQUEST_DELAY)
            games['HOME_TEAM_ABBR'] = games['HOME_TEAM_ID'].map(TEAM_ID_TO_ABBR)
            games['AWAY_TEAM_ABBR'] = games['VISITOR_TEAM_ID'].map(TEAM_ID_TO_ABBR)
            return games[['GAME_ID', 'GAME_DATE_EST', 'HOME_TEAM_ABBR', 'AWAY_TEAM_ABBR']]
        except Exception as e:
            print(f"  [Attempt {attempt}/{MAX_RETRIES}] Schedule fetch failed: {e}")
            time.sleep(REQUEST_DELAY * (2 ** (attempt - 1)))
    raise RuntimeError(f"Failed to fetch schedule for {game_date}")


def fetch_live_team_games(season: str, as_of_date: str) -> pd.DataFrame:
    """
    Fetches the full season (ONE API call for all teams), reuses the exact
    same Phase 2 functions to get correct OPPONENT_PTS/ratings/travel/streaks,
    then filters to games strictly before as_of_date (leakage-safe).
    """
    season_long_df = fetch_season_game_log(season)
    season_game_level = build_game_level_df(season_long_df)
    tg = build_team_games_long(season_game_level)
    tg = add_possessions_and_ratings(tg)
    tg = add_travel_distance(tg)
    tg = add_streak_features(tg)

    cutoff = pd.to_datetime(as_of_date)
    tg = tg[tg['GAME_DATE'] < cutoff]
    return tg.sort_values(['TEAM_ABBREVIATION', 'GAME_DATE'])


def compute_live_rolling_features(team_recent: pd.DataFrame, windows=(5, 10)) -> dict:
    """Computes rolling features (incl. ratings) from a team's pre-filtered game slice."""
    features = {}
    team_recent = team_recent.sort_values('GAME_DATE', ascending=False)

    roll_metrics = ['PTS', 'OPPONENT_PTS', 'FG_PCT', 'FG3_PCT', 'FT_PCT', 'REB', 'AST',
                     'TOV', 'STL', 'BLK', 'WON', 'OFF_RATING', 'DEF_RATING', 'POSSESSIONS']

    for window in windows:
        window_games = team_recent.head(window)
        for metric in roll_metrics:
            features[f'{metric}_ROLL{window}'] = (
                window_games[metric].mean() if len(window_games) > 0 and metric in window_games.columns else np.nan
            )

    # Home/away split (10-game window only, matching training)
    for is_home_val, tag in [(1, 'HOME'), (0, 'AWAY')]:
        split_games = team_recent[team_recent['IS_HOME'] == is_home_val].head(10)
        for metric in ['PTS', 'OPPONENT_PTS', 'WON']:
            features[f'_SPLIT_{tag}_{metric}'] = split_games[metric].mean() if len(split_games) > 0 else np.nan

    # Streak: most recent games, walked backward
    if len(team_recent) > 0:
        results_arr = team_recent['WON'].values
        streak_len, streak_type = 0, results_arr[0]
        for r in results_arr:
            if r == streak_type:
                streak_len += 1
            else:
                break
        features['STREAK'] = float(streak_len if streak_type == 1 else -streak_len)
        last_game = team_recent.iloc[0]
        features['_LAST_LOCATION_TEAM'] = last_game['TEAM_ABBREVIATION'] if last_game['IS_HOME'] == 1 else last_game['OPPONENT']
    else:
        features['STREAK'] = 0.0
        features['_LAST_LOCATION_TEAM'] = None

    features['_LAST_GAME_DATE'] = team_recent['GAME_DATE'].max() if len(team_recent) > 0 else None
    return features


def build_live_game_features(schedule: pd.DataFrame, as_of_date: str, season: str) -> pd.DataFrame:
    """Assembles the full HOME/AWAY feature row for every scheduled game."""
    print(f"Fetching full season data for {season} (single API call)...")
    season_team_games = fetch_live_team_games(season, as_of_date)
    cutoff = pd.to_datetime(as_of_date)

    all_rows = []
    for _, game in schedule.iterrows():
        home_abbr, away_abbr = game['HOME_TEAM_ABBR'], game['AWAY_TEAM_ABBR']
        home_recent = season_team_games[season_team_games['TEAM_ABBREVIATION'] == home_abbr]
        away_recent = season_team_games[season_team_games['TEAM_ABBREVIATION'] == away_abbr]

        home_feats = compute_live_rolling_features(home_recent)
        away_feats = compute_live_rolling_features(away_recent)

        def rest_days_from(last_date):
            return 3.0 if last_date is None or pd.isna(last_date) else float((cutoff - last_date).days)

        home_rest = rest_days_from(home_feats.pop('_LAST_GAME_DATE'))
        away_rest = rest_days_from(away_feats.pop('_LAST_GAME_DATE'))
        home_last_loc = home_feats.pop('_LAST_LOCATION_TEAM')
        away_last_loc = away_feats.pop('_LAST_LOCATION_TEAM')

        def travel_to(last_loc_team, dest_team):
            if last_loc_team is None or last_loc_team not in ARENA_LOCATIONS or dest_team not in ARENA_LOCATIONS:
                return 0.0
            lat1, lon1 = ARENA_LOCATIONS[last_loc_team]
            lat2, lon2 = ARENA_LOCATIONS[dest_team]
            return float(haversine_distance(lat1, lon1, lat2, lon2))

        row = {'GAME_ID': game['GAME_ID'], 'HOME_TEAM': home_abbr, 'AWAY_TEAM': away_abbr}

        for k, v in home_feats.items():
            if k.startswith('_SPLIT_HOME_'):
                row[k.replace('_SPLIT_HOME_', '') + '_ROLL10_SPLIT_HOME'] = v
            elif k.startswith('_SPLIT_AWAY_'):
                pass
            else:
                row[f'{k}_HOME'] = v

        for k, v in away_feats.items():
            if k.startswith('_SPLIT_AWAY_'):
                row[k.replace('_SPLIT_AWAY_', '') + '_ROLL10_SPLIT_AWAY'] = v
            elif k.startswith('_SPLIT_HOME_'):
                pass
            else:
                row[f'{k}_AWAY'] = v

        row['REST_DAYS_HOME'], row['REST_DAYS_AWAY'] = home_rest, away_rest
        row['IS_BACK_TO_BACK_HOME'] = int(home_rest <= 1)
        row['IS_BACK_TO_BACK_AWAY'] = int(away_rest <= 1)
        row['TRAVEL_DISTANCE_HOME'] = travel_to(home_last_loc, home_abbr)
        row['TRAVEL_DISTANCE_AWAY'] = travel_to(away_last_loc, home_abbr)

        all_rows.append(row)

    return pd.DataFrame(all_rows)


def generate_predictions(live_features: pd.DataFrame, feature_cols_: list, train_medians: pd.Series,
                          moneyline_model, spread_model=None, totals_model=None) -> pd.DataFrame:
    """Applies trained models to live features. Missing/absent features -> training median fill."""
    X_live = pd.DataFrame(index=live_features.index)
    missing_cols = []
    for col in feature_cols_:
        if col in live_features.columns:
            X_live[col] = live_features[col]
        else:
            X_live[col] = np.nan
            missing_cols.append(col)

    if missing_cols:
        print(f"NOTE: {len(missing_cols)} training features not available live (median-filled): {missing_cols}")

    X_live = X_live.fillna(train_medians)

    results = live_features[['GAME_ID', 'HOME_TEAM', 'AWAY_TEAM']].copy()
    results['HOME_WIN_PROB'] = moneyline_model.predict_proba(X_live)[:, 1]
    results['PREDICTED_WINNER'] = np.where(results['HOME_WIN_PROB'] >= 0.5, results['HOME_TEAM'], results['AWAY_TEAM'])

    if spread_model is not None:
        results['PREDICTED_SPREAD'] = spread_model.predict(X_live)
    if totals_model is not None:
        results['PREDICTED_TOTAL'] = totals_model.predict(X_live)

    return results


# ==============================================================================
# EXECUTION -- run this section to build everything from scratch
# ==============================================================================

if __name__ == '__main__':
    # --- Phase 1 ---
    raw_long_df = fetch_multi_season_logs(SEASONS)
    game_level_df = build_game_level_df(raw_long_df)
    game_level_df.to_csv('nba_games_raw.csv', index=False)
    print(f"Phase 1 done: {len(game_level_df)} games")

    # --- Phase 2 ---
    team_games, model_ready_df = run_full_feature_pipeline(game_level_df)
    model_ready_df.to_csv('nba_model_ready.csv', index=False)
    print(f"Phase 2 done: {model_ready_df.shape}")

    # --- Phase 3 ---
    model_df = prepare_modeling_data(model_ready_df)
    feature_cols = get_feature_columns(model_df)

    print("\n--- Moneyline (untuned baseline) ---")
    moneyline_results = run_walk_forward_moneyline(model_df, feature_cols)

    print("\n--- Tuning XGBoost (excludes 2024-25 holdout) ---")
    tuning_results = tune_xgboost_walk_forward(model_df, feature_cols)
    best_params = tuning_results.iloc[0][['max_depth', 'learning_rate', 'n_estimators', 'min_child_weight']].to_dict()
    best_params = {k: int(v) for k, v in best_params.items()}
    print(f"Best params: {best_params}")

    print("\n--- Final holdout evaluation (2024-25, untouched) ---")
    final_model, X_test_final, y_test_final, proba_final = final_holdout_evaluation(model_df, feature_cols, best_params)

    print("\n--- Spread model ---")
    spread_results = run_walk_forward_spread(model_df, feature_cols)

    print("\n--- Totals model ---")
    totals_results = run_walk_forward_totals(model_df, feature_cols)

    # Train final spread/totals models on all non-holdout data for live use
    train_df = model_df[model_df['SEASON'] != '2024-25']
    X_train_full = train_df[feature_cols].fillna(train_df[feature_cols].median())
    spread_model_final = XGBRegressor(n_estimators=150, max_depth=3, learning_rate=0.03,
                                       min_child_weight=1, subsample=0.8, colsample_bytree=0.8, random_state=42)
    spread_model_final.fit(X_train_full, train_df['POINT_DIFF'])
    totals_model_final = XGBRegressor(n_estimators=150, max_depth=3, learning_rate=0.03,
                                       min_child_weight=1, subsample=0.8, colsample_bytree=0.8, random_state=42)
    totals_model_final.fit(X_train_full, train_df['TOTAL_PTS'])

    train_medians_full = model_df[feature_cols].median()

    # --- Phase 4 (test against a past date) ---
    TEST_DATE = '2024-01-15'
    TEST_SEASON = '2023-24'
    schedule = fetch_schedule_for_date(TEST_DATE)
    live_features = build_live_game_features(schedule, as_of_date=TEST_DATE, season=TEST_SEASON)
    predictions = generate_predictions(live_features, feature_cols, train_medians_full,
                                        moneyline_model=final_model,
                                        spread_model=spread_model_final,
                                        totals_model=totals_model_final)
    print("\n=== Predictions ===")
    print(predictions.to_string(index=False))

Fetching 2019-20...
  -> 2118 rows
Fetching 2020-21...
  -> 2160 rows
Fetching 2021-22...
  -> 2460 rows
Fetching 2022-23...
  -> 2460 rows
Fetching 2023-24...
  -> 2460 rows
Fetching 2024-25...
  -> 2460 rows
Phase 1 done: 7054 games
Phase 2 done: (7054, 140)
Dropped 95 rows with incomplete rolling features (7054 -> 6959)
Selected 70 leakage-safe feature columns.

--- Moneyline (untuned baseline) ---
Season 2020-21: naive=0.543  logreg=0.570  xgb=0.587
Season 2021-22: naive=0.543  logreg=0.617  xgb=0.609
Season 2022-23: naive=0.581  logreg=0.602  xgb=0.602
Season 2023-24: naive=0.543  logreg=0.645  xgb=0.635
Season 2024-25: naive=0.548  logreg=0.639  xgb=0.626

--- Tuning XGBoost (excludes 2024-25 holdout) ---
Best params: {'max_depth': 3, 'learning_rate': 0, 'n_estimators': 150, 'min_child_weight': 1}

--- Final holdout evaluation (2024-25, untouched) ---
=== FINAL HOLDOUT EVALUATION: 2024-25 ===
Naive baseline accuracy: 0.548
Tuned XGBoost accuracy:  0.548
Tuned XGBoost log loss:  0

In [6]:
# 1. Confirm whether HOME_WIN_PROB is genuinely constant or just looked that way in the truncated view
print("=== HOME_WIN_PROB diagnostic ===")
print(f"Unique values: {predictions['HOME_WIN_PROB'].nunique()} out of {len(predictions)} games")
print(predictions[['HOME_TEAM', 'AWAY_TEAM', 'HOME_WIN_PROB']].to_string(index=False))

# 2. Which features were actually missing for this live run
missing_check = [c for c in feature_cols if c not in live_features.columns]
print(f"\n=== Missing features: {len(missing_check)} ===")
print(missing_check)

# 3. Tuning results (untruncated)
print("\n=== Top 10 tuning configs ===")
print(tuning_results.head(10).to_string(index=False))
print(f"\nBest params used: {best_params}")

# 4. Final holdout metrics (recomputed fresh from the returned objects, not the earlier truncated print)
final_preds = (proba_final >= 0.5).astype(int)
print("\n=== Final holdout (2024-25) metrics ===")
print(f"Accuracy:  {accuracy_score(y_test_final, final_preds):.3f}")
print(f"Log loss:  {log_loss(y_test_final, proba_final):.3f}")
print(f"Precision: {precision_score(y_test_final, final_preds):.3f}")
print(f"Recall:    {recall_score(y_test_final, final_preds):.3f}")

=== HOME_WIN_PROB diagnostic ===
Unique values: 1 out of 11 games
HOME_TEAM AWAY_TEAM  HOME_WIN_PROB
      PHI       HOU       0.552348
      DAL       NOP       0.552348
      NYK       ORL       0.552348
      WAS       DET       0.552348
      ATL       SAS       0.552348
      MEM       GSW       0.552348
      CLE       CHI       0.552348
      BKN       MIA       0.552348
      TOR       BOS       0.552348
      UTA       IND       0.552348
      LAL       OKC       0.552348

=== Missing features: 0 ===
[]

=== Top 10 tuning configs ===
 max_depth  learning_rate  n_estimators  min_child_weight  avg_logloss  avg_accuracy  std_accuracy
         3           0.03           150                 1     0.657016      0.616564      0.018142
         3           0.03           150                 5     0.657032      0.614007      0.017718
         4           0.03           150                 5     0.660412      0.616302      0.021712
         3           0.03           250                

In [7]:
print("=== 1. Class balance check ===")
print(model_df['HOME_WIN'].value_counts(normalize=True))

print("\n=== 2. Prediction distribution on holdout test set ===")
print(f"proba_final min={proba_final.min():.4f}  max={proba_final.max():.4f}  "
      f"std={proba_final.std():.4f}  unique={len(np.unique(proba_final))}")
print(f"Sample of first 15 probabilities: {proba_final[:15]}")

print("\n=== 3. Feature matrix sanity check (holdout test set) ===")
print(f"X_test_final shape: {X_test_final.shape}")
print(f"NaN count per column (top 10 worst):")
print(X_test_final.isna().sum().sort_values(ascending=False).head(10))
print(f"\nAll-zero or all-identical columns (top 10):")
print(X_test_final.nunique().sort_values().head(10))

print("\n=== 4. Feature importance on the FINAL model ===")
final_importances = pd.Series(final_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(final_importances.head(10))
print(f"\nNumber of features with ZERO importance: {(final_importances == 0).sum()} out of {len(final_importances)}")

=== 1. Class balance check ===
HOME_WIN
1    0.551516
0    0.448484
Name: proportion, dtype: float64

=== 2. Prediction distribution on holdout test set ===
proba_final min=0.5523  max=0.5523  std=0.0000  unique=1
Sample of first 15 probabilities: [0.55234784 0.55234784 0.55234784 0.55234784 0.55234784 0.55234784
 0.55234784 0.55234784 0.55234784 0.55234784 0.55234784 0.55234784
 0.55234784 0.55234784 0.55234784]

=== 3. Feature matrix sanity check (holdout test set) ===
X_test_final shape: (1209, 70)
NaN count per column (top 10 worst):
PTS_ROLL5_HOME             0
OPPONENT_PTS_ROLL5_HOME    0
FG_PCT_ROLL5_HOME          0
FG3_PCT_ROLL5_HOME         0
FT_PCT_ROLL5_HOME          0
REB_ROLL5_HOME             0
AST_ROLL5_HOME             0
TOV_ROLL5_HOME             0
STL_ROLL5_HOME             0
BLK_ROLL5_HOME             0
dtype: int64

All-zero or all-identical columns (top 10):
IS_BACK_TO_BACK_HOME     2
IS_BACK_TO_BACK_AWAY     2
REST_DAYS_AWAY           9
WON_ROLL5_HOME          11


In [8]:
best_params = tuning_results.iloc[0][['max_depth', 'learning_rate', 'n_estimators', 'min_child_weight']].to_dict()
best_params['max_depth'] = int(best_params['max_depth'])
best_params['n_estimators'] = int(best_params['n_estimators'])
best_params['min_child_weight'] = int(best_params['min_child_weight'])
print(f"Best params: {best_params}")

final_model, X_test_final, y_test_final, proba_final = final_holdout_evaluation(model_df, feature_cols, best_params)

Best params: {'max_depth': 3, 'learning_rate': 0.03, 'n_estimators': 150, 'min_child_weight': 1}
=== FINAL HOLDOUT EVALUATION: 2024-25 ===
Naive baseline accuracy: 0.548
Tuned XGBoost accuracy:  0.642
Tuned XGBoost log loss:  0.633
Tuned XGBoost precision: 0.643
Tuned XGBoost recall:    0.779
Tuned XGBoost Brier:     0.221


In [9]:
print(f"Unique probabilities: {len(np.unique(proba_final))}")
print(f"Std: {proba_final.std():.4f}")
print(f"Sample: {proba_final[:10]}")

Unique probabilities: 1209
Std: 0.1439
Sample: [0.44949743 0.59748983 0.7016699  0.4161157  0.78628904 0.7769881
 0.60513705 0.42942452 0.29993534 0.39117226]


In [10]:
"""
================================================================================
NBA GAME PREDICTION PIPELINE -- CONSOLIDATED (Phases 1-4)
================================================================================
Run this top-to-bottom in a fresh kernel. Everything below has already been
validated step-by-step in our session -- this file exists so a kernel
restart doesn't force you to hunt through scattered messages again.

SECTIONS:
  Phase 1  -- Data ingestion (LeagueGameLog -> game-level wide dataframe)
  Phase 2  -- Feature engineering (rolling form, ratings, rest, travel,
              streaks, home/away splits) -- all leakage-safe, season-aware
  Phase 3  -- Modeling (walk-forward CV, Moneyline/Spread/Totals, tuning)
  Phase 4  -- Live inference (fetch a date's schedule, predict)
================================================================================
"""

import time
import warnings
import numpy as np
import pandas as pd
from itertools import product

from nba_api.stats.endpoints import leaguegamelog, scoreboardv2
from nba_api.stats.library.http import NBAStatsHTTP
from nba_api.stats.static import teams as nba_teams_static

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, log_loss,
                              brier_score_loss, mean_absolute_error, mean_squared_error)
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier, XGBRegressor

warnings.filterwarnings('ignore')

# ==============================================================================
# PHASE 1: DATA INGESTION
# ==============================================================================

SEASONS = ['2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']
SEASON_TYPE = 'Regular Season'
REQUEST_DELAY = 2.0
MAX_RETRIES = 5
API_TIMEOUT = 120

NBAStatsHTTP.headers.update({
    'Host': 'stats.nba.com',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                   '(KHTML, like Gecko) Chrome/120.0 Safari/537.36',
    'Referer': 'https://www.nba.com/',
    'Origin': 'https://www.nba.com',
    'Accept': 'application/json, text/plain, */*',
})

TEAM_ID_TO_ABBR = {t['id']: t['abbreviation'] for t in nba_teams_static.get_teams()}
TEAM_ABBR_TO_ID = {t['abbreviation']: t['id'] for t in nba_teams_static.get_teams()}


def fetch_season_game_log(season: str, season_type: str = SEASON_TYPE) -> pd.DataFrame:
    """Fetches ALL team game logs for a season in one API call."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            log = leaguegamelog.LeagueGameLog(
                season=season, season_type_all_star=season_type,
                player_or_team_abbreviation='T', timeout=API_TIMEOUT
            )
            df = log.get_data_frames()[0]
            df['SEASON'] = season
            time.sleep(REQUEST_DELAY)
            return df
        except Exception as e:
            wait = REQUEST_DELAY * (2 ** (attempt - 1))
            print(f"  [Attempt {attempt}/{MAX_RETRIES}] {season} failed: {type(e).__name__}: {e} -> retry in {wait:.0f}s")
            time.sleep(wait)
    raise RuntimeError(f"Failed to fetch season {season} after {MAX_RETRIES} attempts.")


def fetch_multi_season_logs(seasons: list) -> pd.DataFrame:
    all_logs = []
    for season in seasons:
        print(f"Fetching {season}...")
        season_df = fetch_season_game_log(season)
        print(f"  -> {len(season_df)} rows")
        all_logs.append(season_df)
    return pd.concat(all_logs, ignore_index=True)


def parse_matchup_column(df: pd.DataFrame) -> pd.DataFrame:
    """'BOS @ NYK' -> BOS away; 'BOS vs. NYK' -> BOS home."""
    df = df.copy()
    df['IS_HOME'] = df['MATCHUP'].str.contains('vs.', regex=False)
    df['OPPONENT_ABBREVIATION'] = np.where(
        df['IS_HOME'], df['MATCHUP'].str.split('vs. ').str[1], df['MATCHUP'].str.split('@ ').str[1]
    )
    return df


def build_game_level_df(long_df: pd.DataFrame) -> pd.DataFrame:
    """Reshapes team-level long data into one row per GAME_ID (HOME/AWAY paired)."""
    df = parse_matchup_column(long_df)
    home_df = df[df['IS_HOME']].copy()
    away_df = df[~df['IS_HOME']].copy()

    home_df = home_df.add_suffix('_HOME')
    away_df = away_df.add_suffix('_AWAY')

    home_df = home_df.rename(columns={'GAME_ID_HOME': 'GAME_ID', 'GAME_DATE_HOME': 'GAME_DATE', 'SEASON_HOME': 'SEASON'})
    away_df = away_df.rename(columns={'GAME_ID_AWAY': 'GAME_ID'})
    away_df = away_df.drop(columns=[c for c in away_df.columns if c.replace('_AWAY', '') in ['GAME_DATE', 'SEASON']])

    game_df = pd.merge(home_df, away_df, on='GAME_ID', how='inner')

    game_df['HOME_WIN'] = (game_df['WL_HOME'] == 'W').astype(int)
    game_df['POINT_DIFF'] = game_df['PTS_HOME'] - game_df['PTS_AWAY']
    game_df['TOTAL_PTS'] = game_df['PTS_HOME'] + game_df['PTS_AWAY']

    game_df['GAME_DATE'] = pd.to_datetime(game_df['GAME_DATE'])
    game_df = game_df.sort_values('GAME_DATE').reset_index(drop=True)
    return game_df


# ==============================================================================
# PHASE 2: FEATURE ENGINEERING
# ==============================================================================

def build_team_games_long(game_df: pd.DataFrame) -> pd.DataFrame:
    """One row per TEAM per GAME (both home and away perspectives stacked)."""
    base_cols = ['GAME_ID', 'GAME_DATE', 'SEASON']
    home_cols = [c for c in game_df.columns if c.endswith('_HOME')]
    stat_names = [c.replace('_HOME', '') for c in home_cols]

    home_rows = game_df[base_cols + home_cols].copy()
    home_rows.columns = base_cols + stat_names
    home_rows['IS_HOME'] = 1
    home_rows['OPPONENT'] = game_df['TEAM_ABBREVIATION_AWAY'].values
    home_rows['OPPONENT_PTS'] = game_df['PTS_AWAY'].values

    away_cols = [c for c in game_df.columns if c.endswith('_AWAY')]
    away_rows = game_df[base_cols + away_cols].copy()
    away_rows.columns = base_cols + [c.replace('_AWAY', '') for c in away_cols]
    away_rows['IS_HOME'] = 0
    away_rows['OPPONENT'] = game_df['TEAM_ABBREVIATION_HOME'].values
    away_rows['OPPONENT_PTS'] = game_df['PTS_HOME'].values

    team_games = pd.concat([home_rows, away_rows], ignore_index=True)
    team_games['WON'] = (team_games['WL'] == 'W').astype(int)
    team_games = team_games.sort_values(['TEAM_ABBREVIATION', 'GAME_DATE']).reset_index(drop=True)
    return team_games


def add_rolling_features(team_games: pd.DataFrame, windows=(5, 10)) -> pd.DataFrame:
    """Leakage-safe rolling averages, RESET PER SEASON (shift(1) before rolling)."""
    team_games = team_games.copy()
    roll_metrics = ['PTS', 'OPPONENT_PTS', 'FG_PCT', 'FG3_PCT', 'FT_PCT',
                     'REB', 'AST', 'TOV', 'STL', 'BLK', 'WON']
    roll_metrics = [m for m in roll_metrics if m in team_games.columns]
    grouped = team_games.groupby(['TEAM_ABBREVIATION', 'SEASON'], group_keys=False)

    for window in windows:
        for metric in roll_metrics:
            team_games[f'{metric}_ROLL{window}'] = grouped[metric].transform(
                lambda s: s.shift(1).rolling(window, min_periods=1).mean()
            )
    return team_games


def add_rest_days(team_games: pd.DataFrame) -> pd.DataFrame:
    """Rest days since previous game, RESET PER SEASON. First game of season -> 3 (default)."""
    team_games = team_games.copy()
    team_games['PREV_GAME_DATE'] = team_games.groupby(['TEAM_ABBREVIATION', 'SEASON'])['GAME_DATE'].shift(1)
    team_games['REST_DAYS'] = (team_games['GAME_DATE'] - team_games['PREV_GAME_DATE']).dt.days
    team_games['REST_DAYS'] = team_games['REST_DAYS'].fillna(3)
    team_games['IS_BACK_TO_BACK'] = (team_games['REST_DAYS'] <= 1).astype(int)
    return team_games.drop(columns=['PREV_GAME_DATE'])


def add_possessions_and_ratings(team_games: pd.DataFrame) -> pd.DataFrame:
    """POSSESSIONS = FGA - OREB + TOV + 0.4*FTA; OFF/DEF_RATING per 100 possessions."""
    team_games = team_games.copy()
    required = ['FGA', 'OREB', 'TOV', 'FTA', 'PTS', 'OPPONENT_PTS']
    missing = [c for c in required if c not in team_games.columns]
    if missing:
        raise ValueError(f"Missing required columns for possession estimate: {missing}")

    team_games['POSSESSIONS'] = team_games['FGA'] - team_games['OREB'] + team_games['TOV'] + (0.4 * team_games['FTA'])
    safe_poss = team_games['POSSESSIONS'].replace(0, np.nan)
    team_games['OFF_RATING'] = 100 * team_games['PTS'] / safe_poss
    team_games['DEF_RATING'] = 100 * team_games['OPPONENT_PTS'] / safe_poss
    return team_games


def add_rating_rolling_features(team_games: pd.DataFrame, windows=(5, 10)) -> pd.DataFrame:
    """Rolling OFF/DEF_RATING/POSSESSIONS, leakage-safe, reset per season."""
    team_games = team_games.copy()
    grouped = team_games.groupby(['TEAM_ABBREVIATION', 'SEASON'], group_keys=False)
    for window in windows:
        for metric in ['OFF_RATING', 'DEF_RATING', 'POSSESSIONS']:
            team_games[f'{metric}_ROLL{window}'] = grouped[metric].transform(
                lambda s: s.shift(1).rolling(window, min_periods=1).mean()
            )
    return team_games


ARENA_LOCATIONS = {
    'ATL': (33.7573, -84.3963), 'BOS': (42.3662, -71.0621), 'BKN': (40.6826, -73.9754),
    'CHA': (35.2251, -80.8392), 'CHI': (41.8807, -87.6742), 'CLE': (41.4965, -81.6882),
    'DAL': (32.7905, -96.8103), 'DEN': (39.7487, -105.0077), 'DET': (42.3410, -83.0550),
    'GSW': (37.7680, -122.3877), 'HOU': (29.7508, -95.3621), 'IND': (39.7640, -86.1555),
    'LAC': (34.0430, -118.2673), 'LAL': (34.0430, -118.2673), 'MEM': (35.1382, -90.0505),
    'MIA': (25.7814, -80.1870), 'MIL': (43.0451, -87.9172), 'MIN': (44.9795, -93.2760),
    'NOP': (29.9490, -90.0821), 'NYK': (40.7505, -73.9934), 'OKC': (35.4634, -97.5151),
    'ORL': (28.5392, -81.3839), 'PHI': (39.9012, -75.1720), 'PHX': (33.4457, -112.0712),
    'POR': (45.5316, -122.6668), 'SAC': (38.5802, -121.4997), 'SAS': (29.4269, -98.4375),
    'TOR': (43.6435, -79.3791), 'UTA': (40.7683, -111.9011), 'WAS': (38.8981, -77.0209),
}


def haversine_distance(lat1, lon1, lat2, lon2):
    """Great-circle distance in miles. Vectorized."""
    R = 3958.8
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return R * 2 * np.arcsin(np.sqrt(a))


def add_travel_distance(team_games: pd.DataFrame) -> pd.DataFrame:
    """Distance traveled since previous game, leakage-safe, reset per season."""
    team_games = team_games.copy()

    def get_game_location(row):
        team = row['TEAM_ABBREVIATION'] if row['IS_HOME'] == 1 else row['OPPONENT']
        return ARENA_LOCATIONS.get(team, (np.nan, np.nan))

    locations = team_games.apply(get_game_location, axis=1)
    team_games['GAME_LAT'] = [loc[0] for loc in locations]
    team_games['GAME_LON'] = [loc[1] for loc in locations]

    grouped = team_games.groupby(['TEAM_ABBREVIATION', 'SEASON'])
    team_games['PREV_LAT'] = grouped['GAME_LAT'].shift(1)
    team_games['PREV_LON'] = grouped['GAME_LON'].shift(1)

    team_games['TRAVEL_DISTANCE'] = haversine_distance(
        team_games['PREV_LAT'], team_games['PREV_LON'], team_games['GAME_LAT'], team_games['GAME_LON']
    )
    team_games['TRAVEL_DISTANCE'] = team_games['TRAVEL_DISTANCE'].fillna(0)
    return team_games.drop(columns=['GAME_LAT', 'GAME_LON', 'PREV_LAT', 'PREV_LON'])


def add_streak_features(team_games: pd.DataFrame) -> pd.DataFrame:
    """Signed win/loss streak length going INTO each game (excludes that game's own result)."""
    team_games = team_games.copy()
    team_games = team_games.sort_values(['TEAM_ABBREVIATION', 'SEASON', 'GAME_DATE']).reset_index(drop=True)

    def compute_streak(group):
        prior_won = group['WON'].shift(1)
        change = (prior_won != prior_won.shift(1)).cumsum()
        streak_len = prior_won.groupby(change).cumcount() + 1
        signed = np.where(prior_won == 1, streak_len, np.where(prior_won == 0, -streak_len, np.nan))
        return pd.Series(signed, index=group.index)

    team_games['STREAK'] = team_games.groupby(['TEAM_ABBREVIATION', 'SEASON'], group_keys=False).apply(compute_streak)
    team_games['STREAK'] = team_games['STREAK'].fillna(0)
    return team_games


def add_home_away_split_form(team_games: pd.DataFrame, window: int = 10) -> pd.DataFrame:
    """Rolling form computed separately for home games vs away games."""
    team_games = team_games.copy()
    split_metrics = ['PTS', 'OPPONENT_PTS', 'WON']
    grouped = team_games.groupby(['TEAM_ABBREVIATION', 'SEASON', 'IS_HOME'], group_keys=False)
    for metric in split_metrics:
        team_games[f'{metric}_ROLL{window}_SPLIT'] = grouped[metric].transform(
            lambda s: s.shift(1).rolling(window, min_periods=1).mean()
        )
    return team_games


def merge_features_to_game_level(game_df: pd.DataFrame, team_games: pd.DataFrame) -> pd.DataFrame:
    """Joins engineered team_games features back onto game_df as HOME_*/AWAY_* columns."""
    feature_cols_ = [c for c in team_games.columns if
                      'ROLL' in c or 'STREAK' in c or 'TRAVEL' in c or c in [
                          'REST_DAYS', 'IS_BACK_TO_BACK', 'OFF_RATING', 'DEF_RATING', 'POSSESSIONS']]

    merge_keys = ['GAME_ID', 'TEAM_ABBREVIATION']
    feature_slice = team_games[merge_keys + feature_cols_].copy()

    home_features = feature_slice.add_suffix('_HOME')
    home_features = home_features.rename(columns={'GAME_ID_HOME': 'GAME_ID', 'TEAM_ABBREVIATION_HOME': 'TEAM_ABBREVIATION_HOME'})
    merged = pd.merge(game_df, home_features, on=['GAME_ID', 'TEAM_ABBREVIATION_HOME'], how='left')

    away_features = feature_slice.add_suffix('_AWAY')
    away_features = away_features.rename(columns={'GAME_ID_AWAY': 'GAME_ID', 'TEAM_ABBREVIATION_AWAY': 'TEAM_ABBREVIATION_AWAY'})
    merged = pd.merge(merged, away_features, on=['GAME_ID', 'TEAM_ABBREVIATION_AWAY'], how='left')

    return merged


def run_full_feature_pipeline(game_level_df: pd.DataFrame) -> tuple:
    """Runs the entire Phase 2 sequence in the correct order. Returns (team_games, model_ready_df)."""
    tg = build_team_games_long(game_level_df)
    tg = add_rolling_features(tg, windows=(5, 10))
    tg = add_rest_days(tg)
    tg = add_possessions_and_ratings(tg)
    tg = add_rating_rolling_features(tg, windows=(5, 10))
    tg = add_travel_distance(tg)
    tg = add_streak_features(tg)
    tg = add_home_away_split_form(tg, window=10)

    model_ready = merge_features_to_game_level(game_level_df, tg)
    return tg, model_ready


def prepare_modeling_data(df: pd.DataFrame) -> pd.DataFrame:
    """Drops rows with incomplete rolling features (early-season games)."""
    df = df.copy()
    critical_cols = ['OFF_RATING_ROLL5_HOME', 'DEF_RATING_ROLL5_HOME',
                      'OFF_RATING_ROLL5_AWAY', 'DEF_RATING_ROLL5_AWAY']
    before = len(df)
    df = df.dropna(subset=critical_cols).reset_index(drop=True)
    print(f"Dropped {before - len(df)} rows with incomplete rolling features ({before} -> {len(df)})")
    return df


def get_feature_columns(df: pd.DataFrame) -> list:
    """Whitelist: only leakage-safe, pre-game-known feature columns."""
    allowed_patterns = ['_ROLL5', '_ROLL10', 'REST_DAYS', 'IS_BACK_TO_BACK',
                         'STREAK', 'TRAVEL_DISTANCE', '_SPLIT']
    feature_cols_ = [c for c in df.columns if any(p in c for p in allowed_patterns)]
    print(f"Selected {len(feature_cols_)} leakage-safe feature columns.")
    return feature_cols_


def walk_forward_season_splits(df: pd.DataFrame, season_col: str = 'SEASON'):
    """Yields (train_idx, test_idx, test_season, train_seasons) -- expanding window by season."""
    seasons_sorted = sorted(df[season_col].unique())
    for i in range(1, len(seasons_sorted)):
        train_seasons = seasons_sorted[:i]
        test_season = seasons_sorted[i]
        train_idx = df[df[season_col].isin(train_seasons)].index
        test_idx = df[df[season_col] == test_season].index
        yield train_idx, test_idx, test_season, train_seasons


# ==============================================================================
# PHASE 3: MODELING & BACKTESTING
# ==============================================================================

def run_walk_forward_moneyline(df: pd.DataFrame, feature_cols_: list, target_col: str = 'HOME_WIN'):
    """Walk-forward LogReg + XGBoost classification, with naive baseline."""
    results = []
    for train_idx, test_idx, test_season, train_seasons in walk_forward_season_splits(df):
        X_train, y_train = df.loc[train_idx, feature_cols_], df.loc[train_idx, target_col]
        X_test, y_test = df.loc[test_idx, feature_cols_], df.loc[test_idx, target_col]

        train_medians = X_train.median()
        X_train, X_test = X_train.fillna(train_medians), X_test.fillna(train_medians)

        naive_acc = accuracy_score(y_test, np.ones(len(y_test)))

        scaler = StandardScaler()
        X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)
        logreg = LogisticRegression(max_iter=1000, C=1.0)
        logreg.fit(X_train_s, y_train)
        logreg_proba = logreg.predict_proba(X_test_s)[:, 1]

        xgb = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42)
        xgb.fit(X_train, y_train)
        xgb_proba = xgb.predict_proba(X_test)[:, 1]

        fold_result = {
            'test_season': test_season, 'naive_accuracy': naive_acc,
            'logreg_accuracy': accuracy_score(y_test, (logreg_proba >= 0.5).astype(int)),
            'logreg_logloss': log_loss(y_test, logreg_proba),
            'xgb_accuracy': accuracy_score(y_test, (xgb_proba >= 0.5).astype(int)),
            'xgb_logloss': log_loss(y_test, xgb_proba),
        }
        results.append(fold_result)
        print(f"Season {test_season}: naive={naive_acc:.3f}  logreg={fold_result['logreg_accuracy']:.3f}  xgb={fold_result['xgb_accuracy']:.3f}")

    return pd.DataFrame(results)


def tune_xgboost_walk_forward(df: pd.DataFrame, feature_cols_: list, target_col: str = 'HOME_WIN', holdout_season: str = '2024-25'):
    """Grid search excluding holdout_season entirely (never touches final test season)."""
    param_grid = {'max_depth': [3, 4, 5], 'learning_rate': [0.03, 0.05, 0.08],
                  'n_estimators': [150, 250], 'min_child_weight': [1, 5]}
    keys = list(param_grid.keys())
    combos = list(product(*param_grid.values()))
    tune_df = df[df['SEASON'] != holdout_season].reset_index(drop=True)

    results = []
    for combo in combos:
        params = dict(zip(keys, combo))
        accs, lls = [], []
        for train_idx, test_idx, test_season, train_seasons in walk_forward_season_splits(tune_df):
            X_train, y_train = tune_df.loc[train_idx, feature_cols_], tune_df.loc[train_idx, target_col]
            X_test, y_test = tune_df.loc[test_idx, feature_cols_], tune_df.loc[test_idx, target_col]
            train_medians = X_train.median()
            X_train, X_test = X_train.fillna(train_medians), X_test.fillna(train_medians)

            model = XGBClassifier(**params, subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42)
            model.fit(X_train, y_train)
            proba = model.predict_proba(X_test)[:, 1]
            lls.append(log_loss(y_test, proba))
            accs.append(accuracy_score(y_test, (proba >= 0.5).astype(int)))

        results.append({**params, 'avg_logloss': np.mean(lls), 'avg_accuracy': np.mean(accs), 'std_accuracy': np.std(accs)})

    return pd.DataFrame(results).sort_values('avg_logloss')


def final_holdout_evaluation(df: pd.DataFrame, feature_cols_: list, best_params: dict,
                               target_col: str = 'HOME_WIN', holdout_season: str = '2024-25'):
    """Trains on all non-holdout seasons, evaluates ONCE on the untouched holdout season."""
    train_df = df[df['SEASON'] != holdout_season]
    test_df = df[df['SEASON'] == holdout_season]
    X_train, y_train = train_df[feature_cols_], train_df[target_col]
    X_test, y_test = test_df[feature_cols_], test_df[target_col]

    train_medians = X_train.median()
    X_train, X_test = X_train.fillna(train_medians), X_test.fillna(train_medians)

    model = XGBClassifier(**best_params, subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42)
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    preds = (proba >= 0.5).astype(int)

    print(f"=== FINAL HOLDOUT EVALUATION: {holdout_season} ===")
    print(f"Naive baseline accuracy: {accuracy_score(y_test, np.ones(len(y_test))):.3f}")
    print(f"Tuned XGBoost accuracy:  {accuracy_score(y_test, preds):.3f}")
    print(f"Tuned XGBoost log loss:  {log_loss(y_test, proba):.3f}")
    print(f"Tuned XGBoost precision: {precision_score(y_test, preds):.3f}")
    print(f"Tuned XGBoost recall:    {recall_score(y_test, preds):.3f}")
    print(f"Tuned XGBoost Brier:     {brier_score_loss(y_test, proba):.3f}")

    return model, X_test, y_test, proba


def run_walk_forward_spread(df: pd.DataFrame, feature_cols_: list, target_col: str = 'POINT_DIFF'):
    """Walk-forward LinReg + XGBoost regression for point spread."""
    results = []
    for train_idx, test_idx, test_season, train_seasons in walk_forward_season_splits(df):
        X_train, y_train = df.loc[train_idx, feature_cols_], df.loc[train_idx, target_col]
        X_test, y_test = df.loc[test_idx, feature_cols_], df.loc[test_idx, target_col]
        train_medians = X_train.median()
        X_train, X_test = X_train.fillna(train_medians), X_test.fillna(train_medians)

        naive_mae = mean_absolute_error(y_test, np.full(len(y_test), y_train.mean()))

        scaler = StandardScaler()
        X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)
        linreg = LinearRegression().fit(X_train_s, y_train)
        linreg_preds = linreg.predict(X_test_s)

        xgb_reg = XGBRegressor(n_estimators=150, max_depth=3, learning_rate=0.03,
                                min_child_weight=1, subsample=0.8, colsample_bytree=0.8, random_state=42)
        xgb_reg.fit(X_train, y_train)
        xgb_preds = xgb_reg.predict(X_test)

        actual_home_win = (y_test > 0).astype(int)
        fold_result = {
            'test_season': test_season, 'naive_mae': naive_mae,
            'linreg_mae': mean_absolute_error(y_test, linreg_preds),
            'linreg_direction_acc': accuracy_score(actual_home_win, (linreg_preds > 0).astype(int)),
            'xgb_mae': mean_absolute_error(y_test, xgb_preds),
            'xgb_direction_acc': accuracy_score(actual_home_win, (xgb_preds > 0).astype(int)),
        }
        results.append(fold_result)
        print(f"Season {test_season}: naive_mae={naive_mae:.2f}  linreg_mae={fold_result['linreg_mae']:.2f}  xgb_mae={fold_result['xgb_mae']:.2f}")

    return pd.DataFrame(results)


def run_walk_forward_totals(df: pd.DataFrame, feature_cols_: list, target_col: str = 'TOTAL_PTS'):
    """Walk-forward LinReg + XGBoost regression for game totals."""
    results = []
    for train_idx, test_idx, test_season, train_seasons in walk_forward_season_splits(df):
        X_train, y_train = df.loc[train_idx, feature_cols_], df.loc[train_idx, target_col]
        X_test, y_test = df.loc[test_idx, feature_cols_], df.loc[test_idx, target_col]
        train_medians = X_train.median()
        X_train, X_test = X_train.fillna(train_medians), X_test.fillna(train_medians)

        naive_mae = mean_absolute_error(y_test, np.full(len(y_test), y_train.mean()))

        scaler = StandardScaler()
        X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)
        linreg = LinearRegression().fit(X_train_s, y_train)
        linreg_preds = linreg.predict(X_test_s)

        xgb_reg = XGBRegressor(n_estimators=150, max_depth=3, learning_rate=0.03,
                                min_child_weight=1, subsample=0.8, colsample_bytree=0.8, random_state=42)
        xgb_reg.fit(X_train, y_train)
        xgb_preds = xgb_reg.predict(X_test)

        fold_result = {
            'test_season': test_season, 'naive_mae': naive_mae,
            'linreg_mae': mean_absolute_error(y_test, linreg_preds),
            'xgb_mae': mean_absolute_error(y_test, xgb_preds),
        }
        results.append(fold_result)
        print(f"Season {test_season}: naive_mae={naive_mae:.2f}  linreg_mae={fold_result['linreg_mae']:.2f}  xgb_mae={fold_result['xgb_mae']:.2f}")

    return pd.DataFrame(results)


# ==============================================================================
# PHASE 4: LIVE INFERENCE
# ==============================================================================

def fetch_schedule_for_date(game_date: str) -> pd.DataFrame:
    """Fetches scheduled games for a given date (format: 'YYYY-MM-DD')."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            sb = scoreboardv2.ScoreboardV2(game_date=game_date, timeout=API_TIMEOUT)
            games = sb.get_data_frames()[0]
            time.sleep(REQUEST_DELAY)
            games['HOME_TEAM_ABBR'] = games['HOME_TEAM_ID'].map(TEAM_ID_TO_ABBR)
            games['AWAY_TEAM_ABBR'] = games['VISITOR_TEAM_ID'].map(TEAM_ID_TO_ABBR)
            return games[['GAME_ID', 'GAME_DATE_EST', 'HOME_TEAM_ABBR', 'AWAY_TEAM_ABBR']]
        except Exception as e:
            print(f"  [Attempt {attempt}/{MAX_RETRIES}] Schedule fetch failed: {e}")
            time.sleep(REQUEST_DELAY * (2 ** (attempt - 1)))
    raise RuntimeError(f"Failed to fetch schedule for {game_date}")


def fetch_live_team_games(season: str, as_of_date: str) -> pd.DataFrame:
    """
    Fetches the full season (ONE API call for all teams), reuses the exact
    same Phase 2 functions to get correct OPPONENT_PTS/ratings/travel/streaks,
    then filters to games strictly before as_of_date (leakage-safe).
    """
    season_long_df = fetch_season_game_log(season)
    season_game_level = build_game_level_df(season_long_df)
    tg = build_team_games_long(season_game_level)
    tg = add_possessions_and_ratings(tg)
    tg = add_travel_distance(tg)
    tg = add_streak_features(tg)

    cutoff = pd.to_datetime(as_of_date)
    tg = tg[tg['GAME_DATE'] < cutoff]
    return tg.sort_values(['TEAM_ABBREVIATION', 'GAME_DATE'])


def compute_live_rolling_features(team_recent: pd.DataFrame, windows=(5, 10)) -> dict:
    """Computes rolling features (incl. ratings) from a team's pre-filtered game slice."""
    features = {}
    team_recent = team_recent.sort_values('GAME_DATE', ascending=False)

    roll_metrics = ['PTS', 'OPPONENT_PTS', 'FG_PCT', 'FG3_PCT', 'FT_PCT', 'REB', 'AST',
                     'TOV', 'STL', 'BLK', 'WON', 'OFF_RATING', 'DEF_RATING', 'POSSESSIONS']

    for window in windows:
        window_games = team_recent.head(window)
        for metric in roll_metrics:
            features[f'{metric}_ROLL{window}'] = (
                window_games[metric].mean() if len(window_games) > 0 and metric in window_games.columns else np.nan
            )

    # Home/away split (10-game window only, matching training)
    for is_home_val, tag in [(1, 'HOME'), (0, 'AWAY')]:
        split_games = team_recent[team_recent['IS_HOME'] == is_home_val].head(10)
        for metric in ['PTS', 'OPPONENT_PTS', 'WON']:
            features[f'_SPLIT_{tag}_{metric}'] = split_games[metric].mean() if len(split_games) > 0 else np.nan

    # Streak: most recent games, walked backward
    if len(team_recent) > 0:
        results_arr = team_recent['WON'].values
        streak_len, streak_type = 0, results_arr[0]
        for r in results_arr:
            if r == streak_type:
                streak_len += 1
            else:
                break
        features['STREAK'] = float(streak_len if streak_type == 1 else -streak_len)
        last_game = team_recent.iloc[0]
        features['_LAST_LOCATION_TEAM'] = last_game['TEAM_ABBREVIATION'] if last_game['IS_HOME'] == 1 else last_game['OPPONENT']
    else:
        features['STREAK'] = 0.0
        features['_LAST_LOCATION_TEAM'] = None

    features['_LAST_GAME_DATE'] = team_recent['GAME_DATE'].max() if len(team_recent) > 0 else None
    return features


def build_live_game_features(schedule: pd.DataFrame, as_of_date: str, season: str) -> pd.DataFrame:
    """Assembles the full HOME/AWAY feature row for every scheduled game."""
    print(f"Fetching full season data for {season} (single API call)...")
    season_team_games = fetch_live_team_games(season, as_of_date)
    cutoff = pd.to_datetime(as_of_date)

    all_rows = []
    for _, game in schedule.iterrows():
        home_abbr, away_abbr = game['HOME_TEAM_ABBR'], game['AWAY_TEAM_ABBR']
        home_recent = season_team_games[season_team_games['TEAM_ABBREVIATION'] == home_abbr]
        away_recent = season_team_games[season_team_games['TEAM_ABBREVIATION'] == away_abbr]

        home_feats = compute_live_rolling_features(home_recent)
        away_feats = compute_live_rolling_features(away_recent)

        def rest_days_from(last_date):
            return 3.0 if last_date is None or pd.isna(last_date) else float((cutoff - last_date).days)

        home_rest = rest_days_from(home_feats.pop('_LAST_GAME_DATE'))
        away_rest = rest_days_from(away_feats.pop('_LAST_GAME_DATE'))
        home_last_loc = home_feats.pop('_LAST_LOCATION_TEAM')
        away_last_loc = away_feats.pop('_LAST_LOCATION_TEAM')

        def travel_to(last_loc_team, dest_team):
            if last_loc_team is None or last_loc_team not in ARENA_LOCATIONS or dest_team not in ARENA_LOCATIONS:
                return 0.0
            lat1, lon1 = ARENA_LOCATIONS[last_loc_team]
            lat2, lon2 = ARENA_LOCATIONS[dest_team]
            return float(haversine_distance(lat1, lon1, lat2, lon2))

        row = {'GAME_ID': game['GAME_ID'], 'HOME_TEAM': home_abbr, 'AWAY_TEAM': away_abbr}

        for k, v in home_feats.items():
            if k.startswith('_SPLIT_HOME_'):
                row[k.replace('_SPLIT_HOME_', '') + '_ROLL10_SPLIT_HOME'] = v
            elif k.startswith('_SPLIT_AWAY_'):
                pass
            else:
                row[f'{k}_HOME'] = v

        for k, v in away_feats.items():
            if k.startswith('_SPLIT_AWAY_'):
                row[k.replace('_SPLIT_AWAY_', '') + '_ROLL10_SPLIT_AWAY'] = v
            elif k.startswith('_SPLIT_HOME_'):
                pass
            else:
                row[f'{k}_AWAY'] = v

        row['REST_DAYS_HOME'], row['REST_DAYS_AWAY'] = home_rest, away_rest
        row['IS_BACK_TO_BACK_HOME'] = int(home_rest <= 1)
        row['IS_BACK_TO_BACK_AWAY'] = int(away_rest <= 1)
        row['TRAVEL_DISTANCE_HOME'] = travel_to(home_last_loc, home_abbr)
        row['TRAVEL_DISTANCE_AWAY'] = travel_to(away_last_loc, home_abbr)

        all_rows.append(row)

    return pd.DataFrame(all_rows)


def generate_predictions(live_features: pd.DataFrame, feature_cols_: list, train_medians: pd.Series,
                          moneyline_model, spread_model=None, totals_model=None) -> pd.DataFrame:
    """Applies trained models to live features. Missing/absent features -> training median fill."""
    X_live = pd.DataFrame(index=live_features.index)
    missing_cols = []
    for col in feature_cols_:
        if col in live_features.columns:
            X_live[col] = live_features[col]
        else:
            X_live[col] = np.nan
            missing_cols.append(col)

    if missing_cols:
        print(f"NOTE: {len(missing_cols)} training features not available live (median-filled): {missing_cols}")

    X_live = X_live.fillna(train_medians)

    results = live_features[['GAME_ID', 'HOME_TEAM', 'AWAY_TEAM']].copy()
    results['HOME_WIN_PROB'] = moneyline_model.predict_proba(X_live)[:, 1]
    results['PREDICTED_WINNER'] = np.where(results['HOME_WIN_PROB'] >= 0.5, results['HOME_TEAM'], results['AWAY_TEAM'])

    if spread_model is not None:
        results['PREDICTED_SPREAD'] = spread_model.predict(X_live)
    if totals_model is not None:
        results['PREDICTED_TOTAL'] = totals_model.predict(X_live)

    return results


# ==============================================================================
# EXECUTION -- run this section to build everything from scratch
# ==============================================================================

if __name__ == '__main__':
    # --- Phase 1 ---
    raw_long_df = fetch_multi_season_logs(SEASONS)
    game_level_df = build_game_level_df(raw_long_df)
    game_level_df.to_csv('nba_games_raw.csv', index=False)
    print(f"Phase 1 done: {len(game_level_df)} games")

    # --- Phase 2 ---
    team_games, model_ready_df = run_full_feature_pipeline(game_level_df)
    model_ready_df.to_csv('nba_model_ready.csv', index=False)
    print(f"Phase 2 done: {model_ready_df.shape}")

    # --- Phase 3 ---
    model_df = prepare_modeling_data(model_ready_df)
    feature_cols = get_feature_columns(model_df)

    print("\n--- Moneyline (untuned baseline) ---")
    moneyline_results = run_walk_forward_moneyline(model_df, feature_cols)

    print("\n--- Tuning XGBoost (excludes 2024-25 holdout) ---")
    tuning_results = tune_xgboost_walk_forward(model_df, feature_cols)
    best_params = tuning_results.iloc[0][['max_depth', 'learning_rate', 'n_estimators', 'min_child_weight']].to_dict()
    # Only cast the genuinely integer hyperparameters -- learning_rate MUST
    # stay a float. Blanket int()-casting every value here previously
    # truncated learning_rate (e.g. 0.03) down to 0, which silently produces
    # a degenerate model that outputs a constant prediction for every input.
    best_params['max_depth'] = int(best_params['max_depth'])
    best_params['n_estimators'] = int(best_params['n_estimators'])
    best_params['min_child_weight'] = int(best_params['min_child_weight'])
    print(f"Best params: {best_params}")

    print("\n--- Final holdout evaluation (2024-25, untouched) ---")
    final_model, X_test_final, y_test_final, proba_final = final_holdout_evaluation(model_df, feature_cols, best_params)

    print("\n--- Spread model ---")
    spread_results = run_walk_forward_spread(model_df, feature_cols)

    print("\n--- Totals model ---")
    totals_results = run_walk_forward_totals(model_df, feature_cols)

    # Train final spread/totals models on all non-holdout data for live use
    train_df = model_df[model_df['SEASON'] != '2024-25']
    X_train_full = train_df[feature_cols].fillna(train_df[feature_cols].median())
    spread_model_final = XGBRegressor(n_estimators=150, max_depth=3, learning_rate=0.03,
                                       min_child_weight=1, subsample=0.8, colsample_bytree=0.8, random_state=42)
    spread_model_final.fit(X_train_full, train_df['POINT_DIFF'])
    totals_model_final = XGBRegressor(n_estimators=150, max_depth=3, learning_rate=0.03,
                                       min_child_weight=1, subsample=0.8, colsample_bytree=0.8, random_state=42)
    totals_model_final.fit(X_train_full, train_df['TOTAL_PTS'])

    train_medians_full = model_df[feature_cols].median()

    # --- Phase 4 (test against a past date) ---
    TEST_DATE = '2024-01-15'
    TEST_SEASON = '2023-24'
    schedule = fetch_schedule_for_date(TEST_DATE)
    live_features = build_live_game_features(schedule, as_of_date=TEST_DATE, season=TEST_SEASON)
    predictions = generate_predictions(live_features, feature_cols, train_medians_full,
                                        moneyline_model=final_model,
                                        spread_model=spread_model_final,
                                        totals_model=totals_model_final)
    print("\n=== Predictions ===")
    print(predictions.to_string(index=False))

Fetching 2019-20...
  -> 2118 rows
Fetching 2020-21...
  -> 2160 rows
Fetching 2021-22...
  -> 2460 rows
Fetching 2022-23...
  -> 2460 rows
Fetching 2023-24...
  -> 2460 rows
Fetching 2024-25...
  -> 2460 rows
Phase 1 done: 7054 games
Phase 2 done: (7054, 140)
Dropped 95 rows with incomplete rolling features (7054 -> 6959)
Selected 70 leakage-safe feature columns.

--- Moneyline (untuned baseline) ---
Season 2020-21: naive=0.543  logreg=0.570  xgb=0.587
Season 2021-22: naive=0.543  logreg=0.617  xgb=0.609
Season 2022-23: naive=0.581  logreg=0.602  xgb=0.602
Season 2023-24: naive=0.543  logreg=0.645  xgb=0.635
Season 2024-25: naive=0.548  logreg=0.639  xgb=0.626

--- Tuning XGBoost (excludes 2024-25 holdout) ---
Best params: {'max_depth': 3, 'learning_rate': 0.03, 'n_estimators': 150, 'min_child_weight': 1}

--- Final holdout evaluation (2024-25, untouched) ---
=== FINAL HOLDOUT EVALUATION: 2024-25 ===
Naive baseline accuracy: 0.548
Tuned XGBoost accuracy:  0.642
Tuned XGBoost log loss:

In [11]:
missing_check = [c for c in feature_cols if c not in live_features.columns]
print(f"Missing feature count: {len(missing_check)}")
print(missing_check)

Missing feature count: 0
[]


In [12]:
predictions.to_csv('live_predictions_check.csv', index=False)
print(f"Predictions shape: {predictions.shape}")
print(predictions[['HOME_TEAM', 'AWAY_TEAM', 'HOME_WIN_PROB', 'PREDICTED_WINNER']].to_string(index=False))

Predictions shape: (11, 7)
HOME_TEAM AWAY_TEAM  HOME_WIN_PROB PREDICTED_WINNER
      PHI       HOU       0.722901              PHI
      DAL       NOP       0.499169              NOP
      NYK       ORL       0.726077              NYK
      WAS       DET       0.428519              DET
      ATL       SAS       0.421500              SAS
      MEM       GSW       0.469415              GSW
      CLE       CHI       0.642268              CLE
      BKN       MIA       0.448075              MIA
      TOR       BOS       0.455338              BOS
      UTA       IND       0.674985              UTA
      LAL       OKC       0.467968              OKC


In [13]:
import pandas as pd

# Adjust path to wherever you downloaded/extracted the Kaggle CSV
odds_df = pd.read_csv('path/to/nba_betting_data.csv')

print("Shape:", odds_df.shape)
print("\nColumns:", odds_df.columns.tolist())
print("\nFirst 5 rows:")
print(odds_df.head())
print("\nDtypes:")
print(odds_df.dtypes)

# Specifically check team name formatting, since this is our likely pain point
team_col_candidates = [c for c in odds_df.columns if 'team' in c.lower()]
print(f"\nTeam-related columns: {team_col_candidates}")
for col in team_col_candidates:
    print(f"\nUnique values in '{col}' (first 20):")
    print(sorted(odds_df[col].dropna().unique())[:20])

# Check date range and format
date_col_candidates = [c for c in odds_df.columns if 'date' in c.lower()]
print(f"\nDate-related columns: {date_col_candidates}")
for col in date_col_candidates:
    print(f"'{col}' sample values: {odds_df[col].head(3).tolist()}")
    print(f"'{col}' range: {odds_df[col].min()} to {odds_df[col].max()}")

FileNotFoundError: [Errno 2] No such file or directory: 'path/to/nba_betting_data.csv'

In [4]:
"""
================================================================================
NBA GAME PREDICTION PIPELINE -- CONSOLIDATED (Phases 1-4)
================================================================================
Run this top-to-bottom in a fresh kernel. Everything below has already been
validated step-by-step in our session -- this file exists so a kernel
restart doesn't force you to hunt through scattered messages again.

SECTIONS:
  Phase 1  -- Data ingestion (LeagueGameLog -> game-level wide dataframe)
  Phase 2  -- Feature engineering (rolling form, ratings, rest, travel,
              streaks, home/away splits) -- all leakage-safe, season-aware
  Phase 3  -- Modeling (walk-forward CV, Moneyline/Spread/Totals, tuning)
  Phase 4  -- Live inference (fetch a date's schedule, predict)
================================================================================
"""

import time
import warnings
import numpy as np
import pandas as pd
from itertools import product

from nba_api.stats.endpoints import leaguegamelog, scoreboardv2
from nba_api.stats.library.http import NBAStatsHTTP
from nba_api.stats.static import teams as nba_teams_static

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, log_loss,
                              brier_score_loss, mean_absolute_error, mean_squared_error)
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier, XGBRegressor

warnings.filterwarnings('ignore')

# ==============================================================================
# PHASE 1: DATA INGESTION
# ==============================================================================

SEASONS = ['2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']
SEASON_TYPE = 'Regular Season'
REQUEST_DELAY = 2.0
MAX_RETRIES = 5
API_TIMEOUT = 120

NBAStatsHTTP.headers.update({
    'Host': 'stats.nba.com',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                   '(KHTML, like Gecko) Chrome/120.0 Safari/537.36',
    'Referer': 'https://www.nba.com/',
    'Origin': 'https://www.nba.com',
    'Accept': 'application/json, text/plain, */*',
})

TEAM_ID_TO_ABBR = {t['id']: t['abbreviation'] for t in nba_teams_static.get_teams()}
TEAM_ABBR_TO_ID = {t['abbreviation']: t['id'] for t in nba_teams_static.get_teams()}


def fetch_season_game_log(season: str, season_type: str = SEASON_TYPE) -> pd.DataFrame:
    """Fetches ALL team game logs for a season in one API call."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            log = leaguegamelog.LeagueGameLog(
                season=season, season_type_all_star=season_type,
                player_or_team_abbreviation='T', timeout=API_TIMEOUT
            )
            df = log.get_data_frames()[0]
            df['SEASON'] = season
            time.sleep(REQUEST_DELAY)
            return df
        except Exception as e:
            wait = REQUEST_DELAY * (2 ** (attempt - 1))
            print(f"  [Attempt {attempt}/{MAX_RETRIES}] {season} failed: {type(e).__name__}: {e} -> retry in {wait:.0f}s")
            time.sleep(wait)
    raise RuntimeError(f"Failed to fetch season {season} after {MAX_RETRIES} attempts.")


def fetch_multi_season_logs(seasons: list) -> pd.DataFrame:
    all_logs = []
    for season in seasons:
        print(f"Fetching {season}...")
        season_df = fetch_season_game_log(season)
        print(f"  -> {len(season_df)} rows")
        all_logs.append(season_df)
    return pd.concat(all_logs, ignore_index=True)


def parse_matchup_column(df: pd.DataFrame) -> pd.DataFrame:
    """'BOS @ NYK' -> BOS away; 'BOS vs. NYK' -> BOS home."""
    df = df.copy()
    df['IS_HOME'] = df['MATCHUP'].str.contains('vs.', regex=False)
    df['OPPONENT_ABBREVIATION'] = np.where(
        df['IS_HOME'], df['MATCHUP'].str.split('vs. ').str[1], df['MATCHUP'].str.split('@ ').str[1]
    )
    return df


def build_game_level_df(long_df: pd.DataFrame) -> pd.DataFrame:
    """Reshapes team-level long data into one row per GAME_ID (HOME/AWAY paired)."""
    df = parse_matchup_column(long_df)
    home_df = df[df['IS_HOME']].copy()
    away_df = df[~df['IS_HOME']].copy()

    home_df = home_df.add_suffix('_HOME')
    away_df = away_df.add_suffix('_AWAY')

    home_df = home_df.rename(columns={'GAME_ID_HOME': 'GAME_ID', 'GAME_DATE_HOME': 'GAME_DATE', 'SEASON_HOME': 'SEASON'})
    away_df = away_df.rename(columns={'GAME_ID_AWAY': 'GAME_ID'})
    away_df = away_df.drop(columns=[c for c in away_df.columns if c.replace('_AWAY', '') in ['GAME_DATE', 'SEASON']])

    game_df = pd.merge(home_df, away_df, on='GAME_ID', how='inner')

    game_df['HOME_WIN'] = (game_df['WL_HOME'] == 'W').astype(int)
    game_df['POINT_DIFF'] = game_df['PTS_HOME'] - game_df['PTS_AWAY']
    game_df['TOTAL_PTS'] = game_df['PTS_HOME'] + game_df['PTS_AWAY']

    game_df['GAME_DATE'] = pd.to_datetime(game_df['GAME_DATE'])
    game_df = game_df.sort_values('GAME_DATE').reset_index(drop=True)
    return game_df


# ==============================================================================
# PHASE 2: FEATURE ENGINEERING
# ==============================================================================

def build_team_games_long(game_df: pd.DataFrame) -> pd.DataFrame:
    """One row per TEAM per GAME (both home and away perspectives stacked)."""
    base_cols = ['GAME_ID', 'GAME_DATE', 'SEASON']
    home_cols = [c for c in game_df.columns if c.endswith('_HOME')]
    stat_names = [c.replace('_HOME', '') for c in home_cols]

    home_rows = game_df[base_cols + home_cols].copy()
    home_rows.columns = base_cols + stat_names
    home_rows['IS_HOME'] = 1
    home_rows['OPPONENT'] = game_df['TEAM_ABBREVIATION_AWAY'].values
    home_rows['OPPONENT_PTS'] = game_df['PTS_AWAY'].values

    away_cols = [c for c in game_df.columns if c.endswith('_AWAY')]
    away_rows = game_df[base_cols + away_cols].copy()
    away_rows.columns = base_cols + [c.replace('_AWAY', '') for c in away_cols]
    away_rows['IS_HOME'] = 0
    away_rows['OPPONENT'] = game_df['TEAM_ABBREVIATION_HOME'].values
    away_rows['OPPONENT_PTS'] = game_df['PTS_HOME'].values

    team_games = pd.concat([home_rows, away_rows], ignore_index=True)
    team_games['WON'] = (team_games['WL'] == 'W').astype(int)
    team_games = team_games.sort_values(['TEAM_ABBREVIATION', 'GAME_DATE']).reset_index(drop=True)
    return team_games


def add_rolling_features(team_games: pd.DataFrame, windows=(5, 10)) -> pd.DataFrame:
    """Leakage-safe rolling averages, RESET PER SEASON (shift(1) before rolling)."""
    team_games = team_games.copy()
    roll_metrics = ['PTS', 'OPPONENT_PTS', 'FG_PCT', 'FG3_PCT', 'FT_PCT',
                     'REB', 'AST', 'TOV', 'STL', 'BLK', 'WON']
    roll_metrics = [m for m in roll_metrics if m in team_games.columns]
    grouped = team_games.groupby(['TEAM_ABBREVIATION', 'SEASON'], group_keys=False)

    for window in windows:
        for metric in roll_metrics:
            team_games[f'{metric}_ROLL{window}'] = grouped[metric].transform(
                lambda s: s.shift(1).rolling(window, min_periods=1).mean()
            )
    return team_games


def add_rest_days(team_games: pd.DataFrame) -> pd.DataFrame:
    """Rest days since previous game, RESET PER SEASON. First game of season -> 3 (default)."""
    team_games = team_games.copy()
    team_games['PREV_GAME_DATE'] = team_games.groupby(['TEAM_ABBREVIATION', 'SEASON'])['GAME_DATE'].shift(1)
    team_games['REST_DAYS'] = (team_games['GAME_DATE'] - team_games['PREV_GAME_DATE']).dt.days
    team_games['REST_DAYS'] = team_games['REST_DAYS'].fillna(3)
    team_games['IS_BACK_TO_BACK'] = (team_games['REST_DAYS'] <= 1).astype(int)
    return team_games.drop(columns=['PREV_GAME_DATE'])


def add_possessions_and_ratings(team_games: pd.DataFrame) -> pd.DataFrame:
    """POSSESSIONS = FGA - OREB + TOV + 0.4*FTA; OFF/DEF_RATING per 100 possessions."""
    team_games = team_games.copy()
    required = ['FGA', 'OREB', 'TOV', 'FTA', 'PTS', 'OPPONENT_PTS']
    missing = [c for c in required if c not in team_games.columns]
    if missing:
        raise ValueError(f"Missing required columns for possession estimate: {missing}")

    team_games['POSSESSIONS'] = team_games['FGA'] - team_games['OREB'] + team_games['TOV'] + (0.4 * team_games['FTA'])
    safe_poss = team_games['POSSESSIONS'].replace(0, np.nan)
    team_games['OFF_RATING'] = 100 * team_games['PTS'] / safe_poss
    team_games['DEF_RATING'] = 100 * team_games['OPPONENT_PTS'] / safe_poss
    return team_games


def add_rating_rolling_features(team_games: pd.DataFrame, windows=(5, 10)) -> pd.DataFrame:
    """Rolling OFF/DEF_RATING/POSSESSIONS, leakage-safe, reset per season."""
    team_games = team_games.copy()
    grouped = team_games.groupby(['TEAM_ABBREVIATION', 'SEASON'], group_keys=False)
    for window in windows:
        for metric in ['OFF_RATING', 'DEF_RATING', 'POSSESSIONS']:
            team_games[f'{metric}_ROLL{window}'] = grouped[metric].transform(
                lambda s: s.shift(1).rolling(window, min_periods=1).mean()
            )
    return team_games


ARENA_LOCATIONS = {
    'ATL': (33.7573, -84.3963), 'BOS': (42.3662, -71.0621), 'BKN': (40.6826, -73.9754),
    'CHA': (35.2251, -80.8392), 'CHI': (41.8807, -87.6742), 'CLE': (41.4965, -81.6882),
    'DAL': (32.7905, -96.8103), 'DEN': (39.7487, -105.0077), 'DET': (42.3410, -83.0550),
    'GSW': (37.7680, -122.3877), 'HOU': (29.7508, -95.3621), 'IND': (39.7640, -86.1555),
    'LAC': (34.0430, -118.2673), 'LAL': (34.0430, -118.2673), 'MEM': (35.1382, -90.0505),
    'MIA': (25.7814, -80.1870), 'MIL': (43.0451, -87.9172), 'MIN': (44.9795, -93.2760),
    'NOP': (29.9490, -90.0821), 'NYK': (40.7505, -73.9934), 'OKC': (35.4634, -97.5151),
    'ORL': (28.5392, -81.3839), 'PHI': (39.9012, -75.1720), 'PHX': (33.4457, -112.0712),
    'POR': (45.5316, -122.6668), 'SAC': (38.5802, -121.4997), 'SAS': (29.4269, -98.4375),
    'TOR': (43.6435, -79.3791), 'UTA': (40.7683, -111.9011), 'WAS': (38.8981, -77.0209),
}


def haversine_distance(lat1, lon1, lat2, lon2):
    """Great-circle distance in miles. Vectorized."""
    R = 3958.8
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return R * 2 * np.arcsin(np.sqrt(a))


def add_travel_distance(team_games: pd.DataFrame) -> pd.DataFrame:
    """Distance traveled since previous game, leakage-safe, reset per season."""
    team_games = team_games.copy()

    def get_game_location(row):
        team = row['TEAM_ABBREVIATION'] if row['IS_HOME'] == 1 else row['OPPONENT']
        return ARENA_LOCATIONS.get(team, (np.nan, np.nan))

    locations = team_games.apply(get_game_location, axis=1)
    team_games['GAME_LAT'] = [loc[0] for loc in locations]
    team_games['GAME_LON'] = [loc[1] for loc in locations]

    grouped = team_games.groupby(['TEAM_ABBREVIATION', 'SEASON'])
    team_games['PREV_LAT'] = grouped['GAME_LAT'].shift(1)
    team_games['PREV_LON'] = grouped['GAME_LON'].shift(1)

    team_games['TRAVEL_DISTANCE'] = haversine_distance(
        team_games['PREV_LAT'], team_games['PREV_LON'], team_games['GAME_LAT'], team_games['GAME_LON']
    )
    team_games['TRAVEL_DISTANCE'] = team_games['TRAVEL_DISTANCE'].fillna(0)
    return team_games.drop(columns=['GAME_LAT', 'GAME_LON', 'PREV_LAT', 'PREV_LON'])


def add_streak_features(team_games: pd.DataFrame) -> pd.DataFrame:
    """Signed win/loss streak length going INTO each game (excludes that game's own result)."""
    team_games = team_games.copy()
    team_games = team_games.sort_values(['TEAM_ABBREVIATION', 'SEASON', 'GAME_DATE']).reset_index(drop=True)

    def compute_streak(group):
        prior_won = group['WON'].shift(1)
        change = (prior_won != prior_won.shift(1)).cumsum()
        streak_len = prior_won.groupby(change).cumcount() + 1
        signed = np.where(prior_won == 1, streak_len, np.where(prior_won == 0, -streak_len, np.nan))
        return pd.Series(signed, index=group.index)

    team_games['STREAK'] = team_games.groupby(['TEAM_ABBREVIATION', 'SEASON'], group_keys=False).apply(compute_streak)
    team_games['STREAK'] = team_games['STREAK'].fillna(0)
    return team_games


def add_home_away_split_form(team_games: pd.DataFrame, window: int = 10) -> pd.DataFrame:
    """Rolling form computed separately for home games vs away games."""
    team_games = team_games.copy()
    split_metrics = ['PTS', 'OPPONENT_PTS', 'WON']
    grouped = team_games.groupby(['TEAM_ABBREVIATION', 'SEASON', 'IS_HOME'], group_keys=False)
    for metric in split_metrics:
        team_games[f'{metric}_ROLL{window}_SPLIT'] = grouped[metric].transform(
            lambda s: s.shift(1).rolling(window, min_periods=1).mean()
        )
    return team_games


def merge_features_to_game_level(game_df: pd.DataFrame, team_games: pd.DataFrame) -> pd.DataFrame:
    """Joins engineered team_games features back onto game_df as HOME_*/AWAY_* columns."""
    feature_cols_ = [c for c in team_games.columns if
                      'ROLL' in c or 'STREAK' in c or 'TRAVEL' in c or c in [
                          'REST_DAYS', 'IS_BACK_TO_BACK', 'OFF_RATING', 'DEF_RATING', 'POSSESSIONS']]

    merge_keys = ['GAME_ID', 'TEAM_ABBREVIATION']
    feature_slice = team_games[merge_keys + feature_cols_].copy()

    home_features = feature_slice.add_suffix('_HOME')
    home_features = home_features.rename(columns={'GAME_ID_HOME': 'GAME_ID', 'TEAM_ABBREVIATION_HOME': 'TEAM_ABBREVIATION_HOME'})
    merged = pd.merge(game_df, home_features, on=['GAME_ID', 'TEAM_ABBREVIATION_HOME'], how='left')

    away_features = feature_slice.add_suffix('_AWAY')
    away_features = away_features.rename(columns={'GAME_ID_AWAY': 'GAME_ID', 'TEAM_ABBREVIATION_AWAY': 'TEAM_ABBREVIATION_AWAY'})
    merged = pd.merge(merged, away_features, on=['GAME_ID', 'TEAM_ABBREVIATION_AWAY'], how='left')

    return merged


def run_full_feature_pipeline(game_level_df: pd.DataFrame) -> tuple:
    """Runs the entire Phase 2 sequence in the correct order. Returns (team_games, model_ready_df)."""
    tg = build_team_games_long(game_level_df)
    tg = add_rolling_features(tg, windows=(5, 10))
    tg = add_rest_days(tg)
    tg = add_possessions_and_ratings(tg)
    tg = add_rating_rolling_features(tg, windows=(5, 10))
    tg = add_travel_distance(tg)
    tg = add_streak_features(tg)
    tg = add_home_away_split_form(tg, window=10)

    model_ready = merge_features_to_game_level(game_level_df, tg)
    return tg, model_ready


def prepare_modeling_data(df: pd.DataFrame) -> pd.DataFrame:
    """Drops rows with incomplete rolling features (early-season games)."""
    df = df.copy()
    critical_cols = ['OFF_RATING_ROLL5_HOME', 'DEF_RATING_ROLL5_HOME',
                      'OFF_RATING_ROLL5_AWAY', 'DEF_RATING_ROLL5_AWAY']
    before = len(df)
    df = df.dropna(subset=critical_cols).reset_index(drop=True)
    print(f"Dropped {before - len(df)} rows with incomplete rolling features ({before} -> {len(df)})")
    return df


def get_feature_columns(df: pd.DataFrame) -> list:
    """Whitelist: only leakage-safe, pre-game-known feature columns."""
    allowed_patterns = ['_ROLL5', '_ROLL10', 'REST_DAYS', 'IS_BACK_TO_BACK',
                         'STREAK', 'TRAVEL_DISTANCE', '_SPLIT']
    feature_cols_ = [c for c in df.columns if any(p in c for p in allowed_patterns)]
    print(f"Selected {len(feature_cols_)} leakage-safe feature columns.")
    return feature_cols_


def walk_forward_season_splits(df: pd.DataFrame, season_col: str = 'SEASON'):
    """Yields (train_idx, test_idx, test_season, train_seasons) -- expanding window by season."""
    seasons_sorted = sorted(df[season_col].unique())
    for i in range(1, len(seasons_sorted)):
        train_seasons = seasons_sorted[:i]
        test_season = seasons_sorted[i]
        train_idx = df[df[season_col].isin(train_seasons)].index
        test_idx = df[df[season_col] == test_season].index
        yield train_idx, test_idx, test_season, train_seasons


# ==============================================================================
# PHASE 3: MODELING & BACKTESTING
# ==============================================================================

def run_walk_forward_moneyline(df: pd.DataFrame, feature_cols_: list, target_col: str = 'HOME_WIN'):
    """Walk-forward LogReg + XGBoost classification, with naive baseline."""
    results = []
    for train_idx, test_idx, test_season, train_seasons in walk_forward_season_splits(df):
        X_train, y_train = df.loc[train_idx, feature_cols_], df.loc[train_idx, target_col]
        X_test, y_test = df.loc[test_idx, feature_cols_], df.loc[test_idx, target_col]

        train_medians = X_train.median()
        X_train, X_test = X_train.fillna(train_medians), X_test.fillna(train_medians)

        naive_acc = accuracy_score(y_test, np.ones(len(y_test)))

        scaler = StandardScaler()
        X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)
        logreg = LogisticRegression(max_iter=1000, C=1.0)
        logreg.fit(X_train_s, y_train)
        logreg_proba = logreg.predict_proba(X_test_s)[:, 1]

        xgb = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42)
        xgb.fit(X_train, y_train)
        xgb_proba = xgb.predict_proba(X_test)[:, 1]

        fold_result = {
            'test_season': test_season, 'naive_accuracy': naive_acc,
            'logreg_accuracy': accuracy_score(y_test, (logreg_proba >= 0.5).astype(int)),
            'logreg_logloss': log_loss(y_test, logreg_proba),
            'xgb_accuracy': accuracy_score(y_test, (xgb_proba >= 0.5).astype(int)),
            'xgb_logloss': log_loss(y_test, xgb_proba),
        }
        results.append(fold_result)
        print(f"Season {test_season}: naive={naive_acc:.3f}  logreg={fold_result['logreg_accuracy']:.3f}  xgb={fold_result['xgb_accuracy']:.3f}")

    return pd.DataFrame(results)


def tune_xgboost_walk_forward(df: pd.DataFrame, feature_cols_: list, target_col: str = 'HOME_WIN', holdout_season: str = '2024-25'):
    """Grid search excluding holdout_season entirely (never touches final test season)."""
    param_grid = {'max_depth': [3, 4, 5], 'learning_rate': [0.03, 0.05, 0.08],
                  'n_estimators': [150, 250], 'min_child_weight': [1, 5]}
    keys = list(param_grid.keys())
    combos = list(product(*param_grid.values()))
    tune_df = df[df['SEASON'] != holdout_season].reset_index(drop=True)

    results = []
    for combo in combos:
        params = dict(zip(keys, combo))
        accs, lls = [], []
        for train_idx, test_idx, test_season, train_seasons in walk_forward_season_splits(tune_df):
            X_train, y_train = tune_df.loc[train_idx, feature_cols_], tune_df.loc[train_idx, target_col]
            X_test, y_test = tune_df.loc[test_idx, feature_cols_], tune_df.loc[test_idx, target_col]
            train_medians = X_train.median()
            X_train, X_test = X_train.fillna(train_medians), X_test.fillna(train_medians)

            model = XGBClassifier(**params, subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42)
            model.fit(X_train, y_train)
            proba = model.predict_proba(X_test)[:, 1]
            lls.append(log_loss(y_test, proba))
            accs.append(accuracy_score(y_test, (proba >= 0.5).astype(int)))

        results.append({**params, 'avg_logloss': np.mean(lls), 'avg_accuracy': np.mean(accs), 'std_accuracy': np.std(accs)})

    return pd.DataFrame(results).sort_values('avg_logloss')


def final_holdout_evaluation(df: pd.DataFrame, feature_cols_: list, best_params: dict,
                               target_col: str = 'HOME_WIN', holdout_season: str = '2024-25'):
    """Trains on all non-holdout seasons, evaluates ONCE on the untouched holdout season."""
    train_df = df[df['SEASON'] != holdout_season]
    test_df = df[df['SEASON'] == holdout_season]
    X_train, y_train = train_df[feature_cols_], train_df[target_col]
    X_test, y_test = test_df[feature_cols_], test_df[target_col]

    train_medians = X_train.median()
    X_train, X_test = X_train.fillna(train_medians), X_test.fillna(train_medians)

    model = XGBClassifier(**best_params, subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42)
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    preds = (proba >= 0.5).astype(int)

    print(f"=== FINAL HOLDOUT EVALUATION: {holdout_season} ===")
    print(f"Naive baseline accuracy: {accuracy_score(y_test, np.ones(len(y_test))):.3f}")
    print(f"Tuned XGBoost accuracy:  {accuracy_score(y_test, preds):.3f}")
    print(f"Tuned XGBoost log loss:  {log_loss(y_test, proba):.3f}")
    print(f"Tuned XGBoost precision: {precision_score(y_test, preds):.3f}")
    print(f"Tuned XGBoost recall:    {recall_score(y_test, preds):.3f}")
    print(f"Tuned XGBoost Brier:     {brier_score_loss(y_test, proba):.3f}")

    return model, X_test, y_test, proba


def run_walk_forward_spread(df: pd.DataFrame, feature_cols_: list, target_col: str = 'POINT_DIFF'):
    """Walk-forward LinReg + XGBoost regression for point spread."""
    results = []
    for train_idx, test_idx, test_season, train_seasons in walk_forward_season_splits(df):
        X_train, y_train = df.loc[train_idx, feature_cols_], df.loc[train_idx, target_col]
        X_test, y_test = df.loc[test_idx, feature_cols_], df.loc[test_idx, target_col]
        train_medians = X_train.median()
        X_train, X_test = X_train.fillna(train_medians), X_test.fillna(train_medians)

        naive_mae = mean_absolute_error(y_test, np.full(len(y_test), y_train.mean()))

        scaler = StandardScaler()
        X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)
        linreg = LinearRegression().fit(X_train_s, y_train)
        linreg_preds = linreg.predict(X_test_s)

        xgb_reg = XGBRegressor(n_estimators=150, max_depth=3, learning_rate=0.03,
                                min_child_weight=1, subsample=0.8, colsample_bytree=0.8, random_state=42)
        xgb_reg.fit(X_train, y_train)
        xgb_preds = xgb_reg.predict(X_test)

        actual_home_win = (y_test > 0).astype(int)
        fold_result = {
            'test_season': test_season, 'naive_mae': naive_mae,
            'linreg_mae': mean_absolute_error(y_test, linreg_preds),
            'linreg_direction_acc': accuracy_score(actual_home_win, (linreg_preds > 0).astype(int)),
            'xgb_mae': mean_absolute_error(y_test, xgb_preds),
            'xgb_direction_acc': accuracy_score(actual_home_win, (xgb_preds > 0).astype(int)),
        }
        results.append(fold_result)
        print(f"Season {test_season}: naive_mae={naive_mae:.2f}  linreg_mae={fold_result['linreg_mae']:.2f}  xgb_mae={fold_result['xgb_mae']:.2f}")

    return pd.DataFrame(results)


def run_walk_forward_totals(df: pd.DataFrame, feature_cols_: list, target_col: str = 'TOTAL_PTS'):
    """Walk-forward LinReg + XGBoost regression for game totals."""
    results = []
    for train_idx, test_idx, test_season, train_seasons in walk_forward_season_splits(df):
        X_train, y_train = df.loc[train_idx, feature_cols_], df.loc[train_idx, target_col]
        X_test, y_test = df.loc[test_idx, feature_cols_], df.loc[test_idx, target_col]
        train_medians = X_train.median()
        X_train, X_test = X_train.fillna(train_medians), X_test.fillna(train_medians)

        naive_mae = mean_absolute_error(y_test, np.full(len(y_test), y_train.mean()))

        scaler = StandardScaler()
        X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)
        linreg = LinearRegression().fit(X_train_s, y_train)
        linreg_preds = linreg.predict(X_test_s)

        xgb_reg = XGBRegressor(n_estimators=150, max_depth=3, learning_rate=0.03,
                                min_child_weight=1, subsample=0.8, colsample_bytree=0.8, random_state=42)
        xgb_reg.fit(X_train, y_train)
        xgb_preds = xgb_reg.predict(X_test)

        fold_result = {
            'test_season': test_season, 'naive_mae': naive_mae,
            'linreg_mae': mean_absolute_error(y_test, linreg_preds),
            'xgb_mae': mean_absolute_error(y_test, xgb_preds),
        }
        results.append(fold_result)
        print(f"Season {test_season}: naive_mae={naive_mae:.2f}  linreg_mae={fold_result['linreg_mae']:.2f}  xgb_mae={fold_result['xgb_mae']:.2f}")

    return pd.DataFrame(results)


# ==============================================================================
# PHASE 4: LIVE INFERENCE
# ==============================================================================

def fetch_schedule_for_date(game_date: str) -> pd.DataFrame:
    """Fetches scheduled games for a given date (format: 'YYYY-MM-DD')."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            sb = scoreboardv2.ScoreboardV2(game_date=game_date, timeout=API_TIMEOUT)
            games = sb.get_data_frames()[0]
            time.sleep(REQUEST_DELAY)
            games['HOME_TEAM_ABBR'] = games['HOME_TEAM_ID'].map(TEAM_ID_TO_ABBR)
            games['AWAY_TEAM_ABBR'] = games['VISITOR_TEAM_ID'].map(TEAM_ID_TO_ABBR)
            return games[['GAME_ID', 'GAME_DATE_EST', 'HOME_TEAM_ABBR', 'AWAY_TEAM_ABBR']]
        except Exception as e:
            print(f"  [Attempt {attempt}/{MAX_RETRIES}] Schedule fetch failed: {e}")
            time.sleep(REQUEST_DELAY * (2 ** (attempt - 1)))
    raise RuntimeError(f"Failed to fetch schedule for {game_date}")


def fetch_live_team_games(season: str, as_of_date: str) -> pd.DataFrame:
    """
    Fetches the full season (ONE API call for all teams), reuses the exact
    same Phase 2 functions to get correct OPPONENT_PTS/ratings/travel/streaks,
    then filters to games strictly before as_of_date (leakage-safe).
    """
    season_long_df = fetch_season_game_log(season)
    season_game_level = build_game_level_df(season_long_df)
    tg = build_team_games_long(season_game_level)
    tg = add_possessions_and_ratings(tg)
    tg = add_travel_distance(tg)
    tg = add_streak_features(tg)

    cutoff = pd.to_datetime(as_of_date)
    tg = tg[tg['GAME_DATE'] < cutoff]
    return tg.sort_values(['TEAM_ABBREVIATION', 'GAME_DATE'])


def compute_live_rolling_features(team_recent: pd.DataFrame, windows=(5, 10)) -> dict:
    """Computes rolling features (incl. ratings) from a team's pre-filtered game slice."""
    features = {}
    team_recent = team_recent.sort_values('GAME_DATE', ascending=False)

    roll_metrics = ['PTS', 'OPPONENT_PTS', 'FG_PCT', 'FG3_PCT', 'FT_PCT', 'REB', 'AST',
                     'TOV', 'STL', 'BLK', 'WON', 'OFF_RATING', 'DEF_RATING', 'POSSESSIONS']

    for window in windows:
        window_games = team_recent.head(window)
        for metric in roll_metrics:
            features[f'{metric}_ROLL{window}'] = (
                window_games[metric].mean() if len(window_games) > 0 and metric in window_games.columns else np.nan
            )

    # Home/away split (10-game window only, matching training)
    for is_home_val, tag in [(1, 'HOME'), (0, 'AWAY')]:
        split_games = team_recent[team_recent['IS_HOME'] == is_home_val].head(10)
        for metric in ['PTS', 'OPPONENT_PTS', 'WON']:
            features[f'_SPLIT_{tag}_{metric}'] = split_games[metric].mean() if len(split_games) > 0 else np.nan

    # Streak: most recent games, walked backward
    if len(team_recent) > 0:
        results_arr = team_recent['WON'].values
        streak_len, streak_type = 0, results_arr[0]
        for r in results_arr:
            if r == streak_type:
                streak_len += 1
            else:
                break
        features['STREAK'] = float(streak_len if streak_type == 1 else -streak_len)
        last_game = team_recent.iloc[0]
        features['_LAST_LOCATION_TEAM'] = last_game['TEAM_ABBREVIATION'] if last_game['IS_HOME'] == 1 else last_game['OPPONENT']
    else:
        features['STREAK'] = 0.0
        features['_LAST_LOCATION_TEAM'] = None

    features['_LAST_GAME_DATE'] = team_recent['GAME_DATE'].max() if len(team_recent) > 0 else None
    return features


def build_live_game_features(schedule: pd.DataFrame, as_of_date: str, season: str) -> pd.DataFrame:
    """Assembles the full HOME/AWAY feature row for every scheduled game."""
    print(f"Fetching full season data for {season} (single API call)...")
    season_team_games = fetch_live_team_games(season, as_of_date)
    cutoff = pd.to_datetime(as_of_date)

    all_rows = []
    for _, game in schedule.iterrows():
        home_abbr, away_abbr = game['HOME_TEAM_ABBR'], game['AWAY_TEAM_ABBR']
        home_recent = season_team_games[season_team_games['TEAM_ABBREVIATION'] == home_abbr]
        away_recent = season_team_games[season_team_games['TEAM_ABBREVIATION'] == away_abbr]

        home_feats = compute_live_rolling_features(home_recent)
        away_feats = compute_live_rolling_features(away_recent)

        def rest_days_from(last_date):
            return 3.0 if last_date is None or pd.isna(last_date) else float((cutoff - last_date).days)

        home_rest = rest_days_from(home_feats.pop('_LAST_GAME_DATE'))
        away_rest = rest_days_from(away_feats.pop('_LAST_GAME_DATE'))
        home_last_loc = home_feats.pop('_LAST_LOCATION_TEAM')
        away_last_loc = away_feats.pop('_LAST_LOCATION_TEAM')

        def travel_to(last_loc_team, dest_team):
            if last_loc_team is None or last_loc_team not in ARENA_LOCATIONS or dest_team not in ARENA_LOCATIONS:
                return 0.0
            lat1, lon1 = ARENA_LOCATIONS[last_loc_team]
            lat2, lon2 = ARENA_LOCATIONS[dest_team]
            return float(haversine_distance(lat1, lon1, lat2, lon2))

        row = {'GAME_ID': game['GAME_ID'], 'HOME_TEAM': home_abbr, 'AWAY_TEAM': away_abbr}

        for k, v in home_feats.items():
            if k.startswith('_SPLIT_HOME_'):
                row[k.replace('_SPLIT_HOME_', '') + '_ROLL10_SPLIT_HOME'] = v
            elif k.startswith('_SPLIT_AWAY_'):
                pass
            else:
                row[f'{k}_HOME'] = v

        for k, v in away_feats.items():
            if k.startswith('_SPLIT_AWAY_'):
                row[k.replace('_SPLIT_AWAY_', '') + '_ROLL10_SPLIT_AWAY'] = v
            elif k.startswith('_SPLIT_HOME_'):
                pass
            else:
                row[f'{k}_AWAY'] = v

        row['REST_DAYS_HOME'], row['REST_DAYS_AWAY'] = home_rest, away_rest
        row['IS_BACK_TO_BACK_HOME'] = int(home_rest <= 1)
        row['IS_BACK_TO_BACK_AWAY'] = int(away_rest <= 1)
        row['TRAVEL_DISTANCE_HOME'] = travel_to(home_last_loc, home_abbr)
        row['TRAVEL_DISTANCE_AWAY'] = travel_to(away_last_loc, home_abbr)

        all_rows.append(row)

    return pd.DataFrame(all_rows)


def generate_predictions(live_features: pd.DataFrame, feature_cols_: list, train_medians: pd.Series,
                          moneyline_model, spread_model=None, totals_model=None) -> pd.DataFrame:
    """Applies trained models to live features. Missing/absent features -> training median fill."""
    X_live = pd.DataFrame(index=live_features.index)
    missing_cols = []
    for col in feature_cols_:
        if col in live_features.columns:
            X_live[col] = live_features[col]
        else:
            X_live[col] = np.nan
            missing_cols.append(col)

    if missing_cols:
        print(f"NOTE: {len(missing_cols)} training features not available live (median-filled): {missing_cols}")

    X_live = X_live.fillna(train_medians)

    results = live_features[['GAME_ID', 'HOME_TEAM', 'AWAY_TEAM']].copy()
    results['HOME_WIN_PROB'] = moneyline_model.predict_proba(X_live)[:, 1]
    results['PREDICTED_WINNER'] = np.where(results['HOME_WIN_PROB'] >= 0.5, results['HOME_TEAM'], results['AWAY_TEAM'])

    if spread_model is not None:
        results['PREDICTED_SPREAD'] = spread_model.predict(X_live)
    if totals_model is not None:
        results['PREDICTED_TOTAL'] = totals_model.predict(X_live)

    return results


# ==============================================================================
# EXECUTION -- run this section to build everything from scratch
# ==============================================================================

if __name__ == '__main__':
    # --- Phase 1 ---
    raw_long_df = fetch_multi_season_logs(SEASONS)
    game_level_df = build_game_level_df(raw_long_df)
    game_level_df.to_csv('nba_games_raw.csv', index=False)
    print(f"Phase 1 done: {len(game_level_df)} games")

    # --- Phase 2 ---
    team_games, model_ready_df = run_full_feature_pipeline(game_level_df)
    model_ready_df.to_csv('nba_model_ready.csv', index=False)
    print(f"Phase 2 done: {model_ready_df.shape}")

    # --- Phase 3 ---
    model_df = prepare_modeling_data(model_ready_df)
    feature_cols = get_feature_columns(model_df)

    print("\n--- Moneyline (untuned baseline) ---")
    moneyline_results = run_walk_forward_moneyline(model_df, feature_cols)

    print("\n--- Tuning XGBoost (excludes 2024-25 holdout) ---")
    tuning_results = tune_xgboost_walk_forward(model_df, feature_cols)
    best_params = tuning_results.iloc[0][['max_depth', 'learning_rate', 'n_estimators', 'min_child_weight']].to_dict()
    # Only cast the genuinely integer hyperparameters -- learning_rate MUST
    # stay a float. Blanket int()-casting every value here previously
    # truncated learning_rate (e.g. 0.03) down to 0, which silently produces
    # a degenerate model that outputs a constant prediction for every input.
    best_params['max_depth'] = int(best_params['max_depth'])
    best_params['n_estimators'] = int(best_params['n_estimators'])
    best_params['min_child_weight'] = int(best_params['min_child_weight'])
    print(f"Best params: {best_params}")

    print("\n--- Final holdout evaluation (2024-25, untouched) ---")
    final_model, X_test_final, y_test_final, proba_final = final_holdout_evaluation(model_df, feature_cols, best_params)

    print("\n--- Spread model ---")
    spread_results = run_walk_forward_spread(model_df, feature_cols)

    print("\n--- Totals model ---")
    totals_results = run_walk_forward_totals(model_df, feature_cols)

    # Train final spread/totals models on all non-holdout data for live use
    train_df = model_df[model_df['SEASON'] != '2024-25']
    X_train_full = train_df[feature_cols].fillna(train_df[feature_cols].median())
    spread_model_final = XGBRegressor(n_estimators=150, max_depth=3, learning_rate=0.03,
                                       min_child_weight=1, subsample=0.8, colsample_bytree=0.8, random_state=42)
    spread_model_final.fit(X_train_full, train_df['POINT_DIFF'])
    totals_model_final = XGBRegressor(n_estimators=150, max_depth=3, learning_rate=0.03,
                                       min_child_weight=1, subsample=0.8, colsample_bytree=0.8, random_state=42)
    totals_model_final.fit(X_train_full, train_df['TOTAL_PTS'])

    train_medians_full = model_df[feature_cols].median()

    # --- Phase 4 (test against a past date) ---
    TEST_DATE = '2024-01-15'
    TEST_SEASON = '2023-24'
    schedule = fetch_schedule_for_date(TEST_DATE)
    live_features = build_live_game_features(schedule, as_of_date=TEST_DATE, season=TEST_SEASON)
    predictions = generate_predictions(live_features, feature_cols, train_medians_full,
                                        moneyline_model=final_model,
                                        spread_model=spread_model_final,
                                        totals_model=totals_model_final)
    print("\n=== Predictions ===")
    print(predictions.to_string(index=False))

Fetching 2019-20...
  -> 2118 rows
Fetching 2020-21...
  -> 2160 rows
Fetching 2021-22...
  -> 2460 rows
Fetching 2022-23...
  -> 2460 rows
Fetching 2023-24...
  -> 2460 rows
Fetching 2024-25...
  -> 2460 rows
Phase 1 done: 7054 games
Phase 2 done: (7054, 140)
Dropped 95 rows with incomplete rolling features (7054 -> 6959)
Selected 70 leakage-safe feature columns.

--- Moneyline (untuned baseline) ---
Season 2020-21: naive=0.543  logreg=0.570  xgb=0.587
Season 2021-22: naive=0.543  logreg=0.617  xgb=0.609
Season 2022-23: naive=0.581  logreg=0.602  xgb=0.597
Season 2023-24: naive=0.543  logreg=0.645  xgb=0.646
Season 2024-25: naive=0.548  logreg=0.639  xgb=0.619

--- Tuning XGBoost (excludes 2024-25 holdout) ---
Best params: {'max_depth': 3, 'learning_rate': 0.03, 'n_estimators': 150, 'min_child_weight': 5}

--- Final holdout evaluation (2024-25, untouched) ---
=== FINAL HOLDOUT EVALUATION: 2024-25 ===
Naive baseline accuracy: 0.548
Tuned XGBoost accuracy:  0.636
Tuned XGBoost log loss:

In [11]:
import subprocess
result = subprocess.run(['where', '/r', 'C:\\', 'nba_pipeline_consolidated.py'],
                         capture_output=True, text=True, timeout=120)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)

STDOUT: 
STDERR: INFO: Could not find files for the given pattern(s).



In [14]:
for name in ['model_ready_df', 'model_df', 'feature_cols', 'final_model']:
    exists = name in dir()
    print(f"{name}: {'EXISTS' if exists else 'MISSING'}")

model_ready_df: EXISTS
model_df: EXISTS
feature_cols: EXISTS
final_model: EXISTS


In [15]:
from sklearn.isotonic import IsotonicRegression

# ---- Phase 1b: Market data ----
ODDS_TEAM_MAP = {
    'atl': 'ATL', 'bkn': 'BKN', 'bos': 'BOS', 'cha': 'CHA', 'chi': 'CHI',
    'cle': 'CLE', 'dal': 'DAL', 'den': 'DEN', 'det': 'DET', 'gs': 'GSW',
    'hou': 'HOU', 'ind': 'IND', 'lac': 'LAC', 'lal': 'LAL', 'mem': 'MEM',
    'mia': 'MIA', 'mil': 'MIL', 'min': 'MIN', 'no': 'NOP', 'ny': 'NYK',
    'okc': 'OKC', 'orl': 'ORL', 'phi': 'PHI', 'phx': 'PHX', 'por': 'POR',
    'sa': 'SAS', 'sac': 'SAC', 'tor': 'TOR', 'utah': 'UTA', 'wsh': 'WAS',
}

def american_odds_to_prob(ml):
    if pd.isna(ml):
        return np.nan
    return 100 / (ml + 100) if ml > 0 else -ml / (-ml + 100)

def load_market_odds(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df['HOME_ABBR'] = df['home'].map(ODDS_TEAM_MAP)
    df['AWAY_ABBR'] = df['away'].map(ODDS_TEAM_MAP)
    unmapped = df[df['HOME_ABBR'].isna() | df['AWAY_ABBR'].isna()]
    if len(unmapped) > 0:
        print(f"WARNING: {len(unmapped)} rows have unmapped team codes -- dropping them.")
        df = df.dropna(subset=['HOME_ABBR', 'AWAY_ABBR'])
    df['CLOSING_SPREAD_HOME'] = np.where(df['whos_favored'] == 'home', df['spread'], -df['spread'])
    df['CLOSING_TOTAL'] = df['total']
    raw_prob_home = df['moneyline_home'].apply(american_odds_to_prob)
    raw_prob_away = df['moneyline_away'].apply(american_odds_to_prob)
    vig_sum = raw_prob_home + raw_prob_away
    df['IMPLIED_WIN_PROB_HOME'] = raw_prob_home / vig_sum
    df['IMPLIED_WIN_PROB_AWAY'] = raw_prob_away / vig_sum
    df['GAME_DATE'] = pd.to_datetime(df['date'])
    keep_cols = ['GAME_DATE', 'HOME_ABBR', 'AWAY_ABBR', 'CLOSING_SPREAD_HOME', 'CLOSING_TOTAL',
                 'moneyline_home', 'moneyline_away', 'IMPLIED_WIN_PROB_HOME', 'IMPLIED_WIN_PROB_AWAY']
    result = df[keep_cols].rename(columns={'moneyline_home': 'MONEYLINE_HOME', 'moneyline_away': 'MONEYLINE_AWAY'})
    dupes = result.duplicated(subset=['GAME_DATE', 'HOME_ABBR', 'AWAY_ABBR'], keep=False)
    if dupes.sum() > 0:
        print(f"WARNING: {dupes.sum()} duplicate combos -- keeping first.")
        result = result.drop_duplicates(subset=['GAME_DATE', 'HOME_ABBR', 'AWAY_ABBR'], keep='first')
    return result

def merge_market_data(game_level_df: pd.DataFrame, market_df: pd.DataFrame) -> pd.DataFrame:
    merged = pd.merge(game_level_df, market_df,
                       left_on=['GAME_DATE', 'TEAM_ABBREVIATION_HOME', 'TEAM_ABBREVIATION_AWAY'],
                       right_on=['GAME_DATE', 'HOME_ABBR', 'AWAY_ABBR'], how='left')
    merged = merged.drop(columns=['HOME_ABBR', 'AWAY_ABBR'])
    match_rate = merged['CLOSING_SPREAD_HOME'].notna().mean()
    print(f"Market data match rate: {match_rate:.1%} ({merged['CLOSING_SPREAD_HOME'].notna().sum()} / {len(merged)} games)")
    return merged

def add_market_residual_targets(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['SPREAD_RESIDUAL'] = df['POINT_DIFF'] - df['CLOSING_SPREAD_HOME']
    df['TOTALS_RESIDUAL'] = df['TOTAL_PTS'] - df['CLOSING_TOTAL']
    return df

# ---- Phase 3h: Calibration ----
def fit_isotonic_calibrator(raw_proba_calib, y_calib):
    calibrator = IsotonicRegression(out_of_bounds='clip')
    calibrator.fit(raw_proba_calib, y_calib)
    return calibrator

def apply_calibration(calibrator, raw_proba):
    return calibrator.predict(raw_proba)

# ---- Phase 3i: Market-relative evaluation ----
def evaluate_vs_market(y_true, our_proba, market_implied_prob):
    valid = market_implied_prob.notna() & y_true.notna()
    n_valid = valid.sum()
    if n_valid == 0:
        print("WARNING: no rows with market implied probability available.")
        return {}
    y_valid = y_true[valid].values
    our_valid = our_proba[valid.values] if isinstance(our_proba, np.ndarray) else our_proba[valid]
    market_valid = market_implied_prob[valid].values
    result = {
        'n_games': int(n_valid),
        'our_logloss': log_loss(y_valid, our_valid), 'market_logloss': log_loss(y_valid, market_valid),
        'our_brier': brier_score_loss(y_valid, our_valid), 'market_brier': brier_score_loss(y_valid, market_valid),
    }
    result['logloss_edge'] = result['market_logloss'] - result['our_logloss']
    result['brier_edge'] = result['market_brier'] - result['our_brier']
    print(f"=== Model vs Market ({n_valid} games) ===")
    print(f"Log Loss -- Ours: {result['our_logloss']:.4f}  Market: {result['market_logloss']:.4f}")
    print(f"Brier    -- Ours: {result['our_brier']:.4f}  Market: {result['market_brier']:.4f}")
    return result

# ---- Phase 3j: Residual walk-forward ----
def run_walk_forward_residual(df, feature_cols_, target_col, market_line_col, label):
    results = []
    df_valid = df[df[target_col].notna()].copy()
    for train_idx, test_idx, test_season, train_seasons in walk_forward_season_splits(df_valid):
        X_train, y_train = df_valid.loc[train_idx, feature_cols_], df_valid.loc[train_idx, target_col]
        X_test, y_test = df_valid.loc[test_idx, feature_cols_], df_valid.loc[test_idx, target_col]
        if len(X_train) == 0 or len(X_test) == 0:
            continue
        train_medians = X_train.median()
        X_train, X_test = X_train.fillna(train_medians), X_test.fillna(train_medians)
        market_mae = mean_absolute_error(y_test, np.zeros(len(y_test)))
        xgb_reg = XGBRegressor(n_estimators=150, max_depth=3, learning_rate=0.03,
                                min_child_weight=1, subsample=0.8, colsample_bytree=0.8, random_state=42)
        xgb_reg.fit(X_train, y_train)
        xgb_preds = xgb_reg.predict(X_test)
        fold_result = {'test_season': test_season, 'n_games': len(test_idx), 'market_mae': market_mae,
                        'model_mae': mean_absolute_error(y_test, xgb_preds)}
        fold_result['beats_market'] = fold_result['model_mae'] < fold_result['market_mae']
        results.append(fold_result)
        print(f"[{label}] {test_season}: market_mae={market_mae:.2f}  model_mae={fold_result['model_mae']:.2f}  "
              f"{'<- BEATS MARKET' if fold_result['beats_market'] else ''}")
    return pd.DataFrame(results)

# ---- Phase 4c: Kelly sizing + backtest ----
def american_to_decimal(american_odds):
    return 1 + (american_odds/100) if american_odds > 0 else 1 + (100/abs(american_odds))

def kelly_fraction(win_prob, american_odds, kelly_multiplier=0.25):
    decimal_odds = american_to_decimal(american_odds)
    b = decimal_odds - 1
    q = 1 - win_prob
    f_star = (b*win_prob - q)/b
    return max(f_star, 0.0) * kelly_multiplier

def simulate_kelly_backtest(predicted_probs, actual_outcomes, american_odds,
                              starting_bankroll=1000.0, kelly_multiplier=0.25,
                              max_bet_pct=0.05, min_edge_threshold=0.0):
    bankroll = starting_bankroll
    trajectory = [bankroll]
    bets_placed = 0
    bet_returns = []
    for p, outcome, odds in zip(predicted_probs, actual_outcomes, american_odds):
        if pd.isna(p) or pd.isna(odds):
            trajectory.append(bankroll); continue
        breakeven_prob = 1/american_to_decimal(odds)
        edge = p - breakeven_prob
        if edge <= min_edge_threshold:
            trajectory.append(bankroll); continue
        f = min(kelly_fraction(p, odds, kelly_multiplier), max_bet_pct)
        stake = bankroll * f
        if stake <= 0:
            trajectory.append(bankroll); continue
        decimal_odds = american_to_decimal(odds)
        profit = stake*(decimal_odds-1) if outcome == 1 else -stake
        bankroll += profit
        bet_returns.append(profit/(bankroll-profit))
        bets_placed += 1
        trajectory.append(bankroll)
    trajectory = np.array(trajectory)
    running_max = np.maximum.accumulate(trajectory)
    drawdowns = (trajectory - running_max)/running_max
    max_drawdown = drawdowns.min()
    bet_returns = np.array(bet_returns)
    sharpe = (bet_returns.mean()/bet_returns.std()*np.sqrt(len(bet_returns))
              if len(bet_returns) > 1 and bet_returns.std() > 0 else np.nan)
    return {'ending_bankroll': bankroll, 'total_return_pct': (bankroll/starting_bankroll-1)*100,
            'bets_placed': bets_placed, 'total_games_seen': len(predicted_probs),
            'max_drawdown_pct': max_drawdown*100, 'sharpe_per_bet': sharpe, 'trajectory': trajectory}

print("New functions loaded successfully.")

New functions loaded successfully.


In [16]:
# --------------------------------------------------------------------------
# Phase 1b/2g: Merge market data + residual targets onto your REAL model_ready_df
# --------------------------------------------------------------------------
market_df = load_market_odds('nba_2008-2026.csv')  # adjust path to wherever you placed this CSV
model_ready_df = merge_market_data(model_ready_df, market_df)
model_ready_df = add_market_residual_targets(model_ready_df)

# Re-run data prep since model_ready_df gained new columns
model_df = prepare_modeling_data(model_ready_df)
feature_cols = get_feature_columns(model_df)

print(f"\nmodel_ready_df shape: {model_ready_df.shape}")
print(f"SPREAD_RESIDUAL mean: {model_df['SPREAD_RESIDUAL'].mean():.2f} (should be near 0)")
print(f"TOTALS_RESIDUAL mean: {model_df['TOTALS_RESIDUAL'].mean():.2f} (should be near 0)")
print(f"IMPLIED_WIN_PROB_HOME non-null: {model_df['IMPLIED_WIN_PROB_HOME'].notna().sum()} / {len(model_df)}")

# Breakdown by season -- confirms the known moneyline coverage gap (2023-24/2024-25 should show ~0)
print("\nMoneyline coverage by season:")
print(model_df.groupby('SEASON')['IMPLIED_WIN_PROB_HOME'].apply(lambda x: x.notna().mean()))

Market data match rate: 100.0% (7051 / 7054 games)
Dropped 95 rows with incomplete rolling features (7054 -> 6959)
Selected 70 leakage-safe feature columns.

model_ready_df shape: (7054, 148)
SPREAD_RESIDUAL mean: -0.11 (should be near 0)
TOTALS_RESIDUAL mean: 0.59 (should be near 0)
IMPLIED_WIN_PROB_HOME non-null: 3969 / 6959

Moneyline coverage by season:
SEASON
2019-20    1.000000
2020-21    1.000000
2021-22    1.000000
2022-23    0.533773
2023-24    0.000000
2024-25    0.000000
Name: IMPLIED_WIN_PROB_HOME, dtype: float64


In [17]:
# --------------------------------------------------------------------------
# Phase 3j: Residual walk-forward -- does our model beat the market itself?
# --------------------------------------------------------------------------
print("=== SPREAD: model vs. trusting the closing line ===")
spread_residual_results = run_walk_forward_residual(
    model_df, feature_cols, target_col='SPREAD_RESIDUAL',
    market_line_col='CLOSING_SPREAD_HOME', label='SPREAD'
)

print("\n=== TOTALS: model vs. trusting the closing line ===")
totals_residual_results = run_walk_forward_residual(
    model_df, feature_cols, target_col='TOTALS_RESIDUAL',
    market_line_col='CLOSING_TOTAL', label='TOTALS'
)

print("\n=== Summary ===")
print(f"Spread -- beats market in {spread_residual_results['beats_market'].sum()}/{len(spread_residual_results)} folds")
print(f"Totals -- beats market in {totals_residual_results['beats_market'].sum()}/{len(totals_residual_results)} folds")

=== SPREAD: model vs. trusting the closing line ===
[SPREAD] 2020-21: market_mae=10.72  model_mae=10.79  
[SPREAD] 2021-22: market_mae=10.59  model_mae=10.70  
[SPREAD] 2022-23: market_mae=9.78  model_mae=9.88  
[SPREAD] 2023-24: market_mae=10.56  model_mae=10.59  
[SPREAD] 2024-25: market_mae=10.49  model_mae=10.54  

=== TOTALS: model vs. trusting the closing line ===
[TOTALS] 2020-21: market_mae=14.38  model_mae=14.68  
[TOTALS] 2021-22: market_mae=14.05  model_mae=14.25  
[TOTALS] 2022-23: market_mae=14.22  model_mae=14.25  
[TOTALS] 2023-24: market_mae=14.24  model_mae=14.44  
[TOTALS] 2024-25: market_mae=14.12  model_mae=14.18  

=== Summary ===
Spread -- beats market in 0/5 folds
Totals -- beats market in 0/5 folds


In [18]:
# --------------------------------------------------------------------------
# Phase 3h/3i: Calibrate the moneyline model, then test vs. market
# --------------------------------------------------------------------------
# Use 2019-22 (pre-gap) as the calibration set, since it has full moneyline coverage
calib_seasons = ['2019-20', '2020-21']
eval_season = '2021-22'  # has moneyline coverage AND wasn't used for calibration

calib_df = model_df[model_df['SEASON'].isin(calib_seasons)]
eval_df = model_df[model_df['SEASON'] == eval_season]

X_calib = calib_df[feature_cols].fillna(calib_df[feature_cols].median())
y_calib = calib_df['HOME_WIN']
X_eval = eval_df[feature_cols].fillna(calib_df[feature_cols].median())
y_eval = eval_df['HOME_WIN']

# Train on calibration seasons, get raw probabilities
raw_model = XGBClassifier(max_depth=3, learning_rate=0.03, n_estimators=150,
                           min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
                           eval_metric='logloss', random_state=42)
raw_model.fit(X_calib, y_calib)
raw_proba_eval = raw_model.predict_proba(X_eval)[:, 1]

# Fit isotonic calibrator on a SEPARATE split within calib_df (not eval_df, not the training data itself)
# Simple approach: use 2020-21 predictions (out-of-fold relative to a 2019-20-only model) to fit the calibrator
calib_only_model = XGBClassifier(max_depth=3, learning_rate=0.03, n_estimators=150,
                                  min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
                                  eval_metric='logloss', random_state=42)
train_2019 = model_df[model_df['SEASON'] == '2019-20']
X_2019 = train_2019[feature_cols].fillna(train_2019[feature_cols].median())
calib_only_model.fit(X_2019, train_2019['HOME_WIN'])

calib_2020 = model_df[model_df['SEASON'] == '2020-21']
X_2020 = calib_2020[feature_cols].fillna(X_2019.median())
raw_proba_2020 = calib_only_model.predict_proba(X_2020)[:, 1]

calibrator = fit_isotonic_calibrator(raw_proba_2020, calib_2020['HOME_WIN'].values)
calibrated_proba_eval = apply_calibration(calibrator, raw_proba_eval)

# Now compare BOTH raw and calibrated probabilities against the market on eval_season
print("=== RAW (uncalibrated) model vs market ===")
evaluate_vs_market(eval_df['HOME_WIN'], raw_proba_eval, eval_df['IMPLIED_WIN_PROB_HOME'])

print("\n=== CALIBRATED model vs market ===")
evaluate_vs_market(eval_df['HOME_WIN'], calibrated_proba_eval, eval_df['IMPLIED_WIN_PROB_HOME'])

=== RAW (uncalibrated) model vs market ===
=== Model vs Market (1214 games) ===
Log Loss -- Ours: 0.6588  Market: 0.6047
Brier    -- Ours: 0.2326  Market: 0.2088

=== CALIBRATED model vs market ===
=== Model vs Market (1214 games) ===
Log Loss -- Ours: 0.6739  Market: 0.6047
Brier    -- Ours: 0.2349  Market: 0.2088


{'n_games': 1214,
 'our_logloss': 0.6738913655281067,
 'market_logloss': 0.6046989816659113,
 'our_brier': 0.23490938544273376,
 'market_brier': 0.20878997164420893,
 'logloss_edge': -0.0691923838621954,
 'brier_edge': -0.026119413798524838}

In [19]:
# --------------------------------------------------------------------------
# Phase 3h (CORRECTED): One consistent model throughout -- train, calibrate, eval
# --------------------------------------------------------------------------
train_season = '2019-20'
calib_season = '2020-21'
eval_season = '2021-22'

train_df = model_df[model_df['SEASON'] == train_season]
calib_df = model_df[model_df['SEASON'] == calib_season]
eval_df = model_df[model_df['SEASON'] == eval_season]

X_train = train_df[feature_cols].fillna(train_df[feature_cols].median())
y_train = train_df['HOME_WIN']
train_medians = X_train.median()  # reused for calib/eval fillna, consistent with our leakage discipline

X_calib = calib_df[feature_cols].fillna(train_medians)
X_eval = eval_df[feature_cols].fillna(train_medians)

# ONE model, trained once
model = XGBClassifier(max_depth=3, learning_rate=0.03, n_estimators=150,
                       min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
                       eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

# That SAME model's predictions on the calibration season
raw_proba_calib = model.predict_proba(X_calib)[:, 1]
calibrator = fit_isotonic_calibrator(raw_proba_calib, calib_df['HOME_WIN'].values)

# That SAME model's predictions on eval, both raw and calibrated
raw_proba_eval = model.predict_proba(X_eval)[:, 1]
calibrated_proba_eval = apply_calibration(calibrator, raw_proba_eval)

print("=== RAW (uncalibrated) vs market ===")
evaluate_vs_market(eval_df['HOME_WIN'], raw_proba_eval, eval_df['IMPLIED_WIN_PROB_HOME'])

print("\n=== CALIBRATED vs market (properly, same base model) ===")
evaluate_vs_market(eval_df['HOME_WIN'], calibrated_proba_eval, eval_df['IMPLIED_WIN_PROB_HOME'])

=== RAW (uncalibrated) vs market ===
=== Model vs Market (1214 games) ===
Log Loss -- Ours: 0.6599  Market: 0.6047
Brier    -- Ours: 0.2323  Market: 0.2088

=== CALIBRATED vs market (properly, same base model) ===
=== Model vs Market (1214 games) ===
Log Loss -- Ours: 0.6693  Market: 0.6047
Brier    -- Ours: 0.2327  Market: 0.2088


{'n_games': 1214,
 'our_logloss': 0.6693305373191833,
 'market_logloss': 0.6046989816659113,
 'our_brier': 0.23272547125816345,
 'market_brier': 0.20878997164420893,
 'logloss_edge': -0.06463155565327205,
 'brier_edge': -0.023935499613954525}

In [20]:
# --------------------------------------------------------------------------
# Phase 3h (v2): Platt scaling instead of isotonic -- more robust on small samples
# --------------------------------------------------------------------------
def fit_platt_calibrator(raw_proba_calib, y_calib):
    """
    Platt scaling: fits a simple 2-parameter logistic regression on top of
    raw probabilities. Far less prone to overfitting than isotonic on small
    calibration sets (~1000-1200 games), since it can only learn a sigmoid
    shape, not an arbitrary step function.
    """
    platt = LogisticRegression()
    platt.fit(raw_proba_calib.reshape(-1, 1), y_calib)
    return platt

def apply_platt_calibration(platt_model, raw_proba):
    return platt_model.predict_proba(raw_proba.reshape(-1, 1))[:, 1]


train_seasons = ['2019-20', '2020-21']   # wider base -- more data for the model itself
calib_season = '2021-22'
eval_season = '2022-23'                  # 53% moneyline coverage, but genuinely untouched by training/calibration

train_df = model_df[model_df['SEASON'].isin(train_seasons)]
calib_df = model_df[model_df['SEASON'] == calib_season]
eval_df = model_df[model_df['SEASON'] == eval_season]

X_train = train_df[feature_cols].fillna(train_df[feature_cols].median())
y_train = train_df['HOME_WIN']
train_medians = X_train.median()

X_calib = calib_df[feature_cols].fillna(train_medians)
X_eval = eval_df[feature_cols].fillna(train_medians)

model = XGBClassifier(max_depth=3, learning_rate=0.03, n_estimators=150,
                       min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
                       eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

raw_proba_calib = model.predict_proba(X_calib)[:, 1]
platt = fit_platt_calibrator(raw_proba_calib, calib_df['HOME_WIN'].values)

raw_proba_eval = model.predict_proba(X_eval)[:, 1]
calibrated_proba_eval = apply_platt_calibration(platt, raw_proba_eval)

print("=== RAW vs market (2022-23, moneyline subset) ===")
evaluate_vs_market(eval_df['HOME_WIN'], raw_proba_eval, eval_df['IMPLIED_WIN_PROB_HOME'])

print("\n=== PLATT-CALIBRATED vs market ===")
evaluate_vs_market(eval_df['HOME_WIN'], calibrated_proba_eval, eval_df['IMPLIED_WIN_PROB_HOME'])

=== RAW vs market (2022-23, moneyline subset) ===
=== Model vs Market (648 games) ===
Log Loss -- Ours: 0.6690  Market: 0.6391
Brier    -- Ours: 0.2382  Market: 0.2234

=== PLATT-CALIBRATED vs market ===
=== Model vs Market (648 games) ===
Log Loss -- Ours: 0.6678  Market: 0.6391
Brier    -- Ours: 0.2376  Market: 0.2234


{'n_games': 648,
 'our_logloss': 0.6678382754325867,
 'market_logloss': 0.6391334526049275,
 'our_brier': 0.23763908445835114,
 'market_brier': 0.22344995419393485,
 'logloss_edge': -0.028704822827659182,
 'brier_edge': -0.014189130264416289}

In [22]:
# --------------------------------------------------------------------------
# Phase 4c: Kelly backtest on REAL predictions -- confirms the system correctly
# declines to bet when there's no genuine edge, rather than gambling on noise.
# --------------------------------------------------------------------------

# Reconstruct decimal-odds-compatible american odds for the eval_season subset
# (only rows where we actually have real moneylines)
bet_df = eval_df[eval_df['MONEYLINE_HOME'].notna()].copy()
bet_df['OUR_PROB'] = calibrated_proba_eval[eval_df['MONEYLINE_HOME'].notna().values]
bet_df['ACTUAL_OUTCOME'] = bet_df['HOME_WIN']  # 1 if home won (bet resolves in our favor if we bet home)

print(f"Games with real moneyline odds in eval season: {len(bet_df)}")

backtest_result = simulate_kelly_backtest(
    predicted_probs=bet_df['OUR_PROB'].values,
    actual_outcomes=bet_df['ACTUAL_OUTCOME'].values,
    american_odds=bet_df['MONEYLINE_HOME'].values,
    starting_bankroll=1000.0,
    kelly_multiplier=0.25,
    max_bet_pct=0.05,
    min_edge_threshold=0.02  # require at least a 2pt edge over breakeven before betting -- filters out noise-level "edges"
)

print(f"\n=== Kelly Backtest Results (Moneyline, {eval_season}, home-side bets only) ===")
print(f"Games seen: {backtest_result['total_games_seen']}")
print(f"Bets actually placed: {backtest_result['bets_placed']}")
print(f"Starting bankroll: ${backtest_result['starting_bankroll']:.2f}")
print(f"Ending bankroll: ${backtest_result['ending_bankroll']:.2f}")
print(f"Total return: {backtest_result['total_return_pct']:.2f}%")
print(f"Max drawdown: {backtest_result['max_drawdown_pct']:.2f}%")
print(f"Sharpe (per-bet): {backtest_result['sharpe_per_bet']}")

Games with real moneyline odds in eval season: 648

=== Kelly Backtest Results (Moneyline, 2022-23, home-side bets only) ===
Games seen: 648
Bets actually placed: 213


KeyError: 'starting_bankroll'

In [23]:
# Quick fix: starting_bankroll wasn't returned by the pasted version -- just reference it directly
print(f"\n=== Kelly Backtest Results (Moneyline, {eval_season}, home-side bets only) ===")
print(f"Games seen: {backtest_result['total_games_seen']}")
print(f"Bets actually placed: {backtest_result['bets_placed']}")
print(f"Starting bankroll: $1000.00")
print(f"Ending bankroll: ${backtest_result['ending_bankroll']:.2f}")
print(f"Total return: {backtest_result['total_return_pct']:.2f}%")
print(f"Max drawdown: {backtest_result['max_drawdown_pct']:.2f}%")
print(f"Sharpe (per-bet): {backtest_result['sharpe_per_bet']}")

# Now let's actually investigate the 213-bet figure before trusting it
print(f"\n=== Investigating why 213/648 games cleared the edge threshold ===")
bet_df['BREAKEVEN_PROB'] = 1 / bet_df['MONEYLINE_HOME'].apply(american_to_decimal)
bet_df['EDGE'] = bet_df['OUR_PROB'] - bet_df['BREAKEVEN_PROB']
print(f"Edge distribution across ALL 648 games:\n{bet_df['EDGE'].describe()}")
print(f"\nGames with edge > 0.02 (our threshold): {(bet_df['EDGE'] > 0.02).sum()}")

# Of the bets we WOULD place, what's our actual win rate vs implied breakeven?
placed = bet_df[bet_df['EDGE'] > 0.02]
print(f"\nOf the {len(placed)} games we'd bet on:")
print(f"Our actual win rate: {placed['ACTUAL_OUTCOME'].mean():.3f}")
print(f"Average breakeven required: {placed['BREAKEVEN_PROB'].mean():.3f}")
print(f"Average our_prob claimed: {placed['OUR_PROB'].mean():.3f}")


=== Kelly Backtest Results (Moneyline, 2022-23, home-side bets only) ===
Games seen: 648
Bets actually placed: 213
Starting bankroll: $1000.00
Ending bankroll: $4363.92
Total return: 336.39%
Max drawdown: -49.36%
Sharpe (per-bet): 2.083699761515436

=== Investigating why 213/648 games cleared the edge threshold ===
Edge distribution across ALL 648 games:
count    648.000000
mean      -0.044340
std        0.141673
min       -0.438517
25%       -0.145395
50%       -0.054546
75%        0.050500
max        0.367461
Name: EDGE, dtype: float64

Games with edge > 0.02 (our threshold): 213

Of the 213 games we'd bet on:
Our actual win rate: 0.474
Average breakeven required: 0.406
Average our_prob claimed: 0.523


In [24]:
# --------------------------------------------------------------------------
# Robustness check: does the same model+calibrator show a similar pattern
# on a DIFFERENT season it's never seen? (2019-20/2020-21 held out earlier)
# --------------------------------------------------------------------------
# Re-use the SAME model + platt calibrator from before, apply to 2019-20 instead
robustness_season = model_df[model_df['SEASON'] == '2019-20']
robustness_season = robustness_season[robustness_season['MONEYLINE_HOME'].notna()]

X_robust = robustness_season[feature_cols].fillna(train_medians)
raw_proba_robust = model.predict_proba(X_robust)[:, 1]
calibrated_proba_robust = apply_platt_calibration(platt, raw_proba_robust)

robust_df = robustness_season.copy()
robust_df['OUR_PROB'] = calibrated_proba_robust
robust_df['BREAKEVEN_PROB'] = 1 / robust_df['MONEYLINE_HOME'].apply(american_to_decimal)
robust_df['EDGE'] = robust_df['OUR_PROB'] - robust_df['BREAKEVEN_PROB']
robust_df['ACTUAL_OUTCOME'] = robust_df['HOME_WIN']

robust_placed = robust_df[robust_df['EDGE'] > 0.02]
print(f"NOTE: model was TRAINED partly on this season's data lineage -- this is a rough directional check, not a clean holdout")
print(f"Games clearing threshold: {len(robust_placed)} / {len(robust_df)}")
print(f"Win rate on selected subset: {robust_placed['ACTUAL_OUTCOME'].mean():.3f}")
print(f"Average breakeven required: {robust_placed['BREAKEVEN_PROB'].mean():.3f}")

NOTE: model was TRAINED partly on this season's data lineage -- this is a rough directional check, not a clean holdout
Games clearing threshold: 343 / 1043
Win rate on selected subset: 0.481
Average breakeven required: 0.373


In [25]:
# --------------------------------------------------------------------------
# Diagnostic: does our model ever predict genuinely LOW probabilities,
# or does it compress toward the middle regardless of true underdog status?
# --------------------------------------------------------------------------
combined_check = pd.concat([bet_df[['OUR_PROB', 'BREAKEVEN_PROB', 'MONEYLINE_HOME']],
                             robust_df[['OUR_PROB', 'BREAKEVEN_PROB', 'MONEYLINE_HOME']]])

print(f"OUR_PROB distribution (both seasons combined, n={len(combined_check)}):")
print(combined_check['OUR_PROB'].describe())
print(f"\nBREAKEVEN_PROB distribution (the market's implied floor):")
print(combined_check['BREAKEVEN_PROB'].describe())

print(f"\nMinimum OUR_PROB ever predicted: {combined_check['OUR_PROB'].min():.3f}")
print(f"Minimum BREAKEVEN_PROB in the data (biggest real underdogs): {combined_check['BREAKEVEN_PROB'].min():.3f}")

# The real test: for the BIGGEST underdogs (bottom 10% by breakeven), 
# does our model actually track down with them, or stay flat?
biggest_dogs = combined_check.nsmallest(int(len(combined_check)*0.1), 'BREAKEVEN_PROB')
print(f"\nBiggest underdogs (bottom 10%, n={len(biggest_dogs)}):")
print(f"  Their average BREAKEVEN_PROB (what market thinks): {biggest_dogs['BREAKEVEN_PROB'].mean():.3f}")
print(f"  Our average OUR_PROB for these SAME games: {biggest_dogs['OUR_PROB'].mean():.3f}")
print(f"  Gap: {(biggest_dogs['OUR_PROB'] - biggest_dogs['BREAKEVEN_PROB']).mean():.3f}")

OUR_PROB distribution (both seasons combined, n=1691):
count    1691.000000
mean        0.540111
std         0.108226
min         0.263472
25%         0.459198
50%         0.540251
75%         0.627798
max         0.766722
Name: OUR_PROB, dtype: float64

BREAKEVEN_PROB distribution (the market's implied floor):
count    1691.000000
mean        0.589290
std         0.198674
min         0.100000
25%         0.434783
50%         0.600000
75%         0.750000
max         0.980392
Name: BREAKEVEN_PROB, dtype: float64

Minimum OUR_PROB ever predicted: 0.263
Minimum BREAKEVEN_PROB in the data (biggest real underdogs): 0.100

Biggest underdogs (bottom 10%, n=169):
  Their average BREAKEVEN_PROB (what market thinks): 0.244
  Our average OUR_PROB for these SAME games: 0.412
  Gap: 0.168


In [26]:
import importlib
import nba_pipeline_consolidated
importlib.reload(nba_pipeline_consolidated)
from nba_pipeline_consolidated import *

print("Functions loaded. Testing availability:", 'fetch_season_player_log' in dir())

Functions loaded. Testing availability: True


In [27]:
player_log_raw = fetch_multi_season_player_logs(SEASONS)
print(f"\nTotal player-game rows: {len(player_log_raw)}")
print(player_log_raw[['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ABBREVIATION', 'GAME_DATE', 'MIN', 'PTS']].head(10))

Fetching player log 2019-20...
  -> 22393 player-game rows
Fetching player log 2020-21...
  -> 23054 player-game rows
Fetching player log 2021-22...
  -> 26039 player-game rows
Fetching player log 2022-23...
  -> 25895 player-game rows
Fetching player log 2023-24...
  -> 26401 player-game rows
Fetching player log 2024-25...
  -> 26306 player-game rows

Total player-game rows: 150088
   PLAYER_ID       PLAYER_NAME TEAM_ABBREVIATION   GAME_DATE  MIN  PTS
0     202324    Derrick Favors               NOP  2019-10-22   21    6
1    1627742    Brandon Ingram               NOP  2019-10-22   35   22
2    1628404         Josh Hart               NOP  2019-10-22   28   15
3     101150      Lou Williams               LAC  2019-10-22   37   21
4     201976  Patrick Beverley               LAC  2019-10-22   31    2
5       2544      LeBron James               LAL  2019-10-22   36   18
6     203584      Troy Daniels               LAL  2019-10-22   16    6
7     201586       Serge Ibaka               T

In [28]:
# --------------------------------------------------------------------------
# Compute rolling player minutes/impact, then missing-rotation-player features
# --------------------------------------------------------------------------
player_log_roll = compute_expected_rotation(player_log_raw, top_n=8, window=10)
print(f"Player log with rolling features: {player_log_roll.shape}")

# This uses your existing 'team_games' object from Phase 2 -- confirm it's still in memory
print(f"team_games available: {team_games.shape}")

missing_impact_df = compute_missing_player_impact(player_log_roll, team_games, top_n=8)
print(f"\nMissing-impact computed for {len(missing_impact_df)} team-games")
print(f"NaN count (season openers, no rotation history yet): {missing_impact_df['MISSING_PLAYERS_COUNT'].isna().sum()}")

print(f"\nMISSING_PLAYERS_COUNT distribution:\n{missing_impact_df['MISSING_PLAYERS_COUNT'].describe()}")
print(f"\nMISSING_PTS_IMPACT distribution:\n{missing_impact_df['MISSING_PTS_IMPACT'].describe()}")

# Sanity spot-check: find a specific real example of a game with multiple
# missing key players and eyeball whether it looks plausible
big_absence = missing_impact_df.nlargest(5, 'MISSING_PTS_IMPACT')
print(f"\nTop 5 games by missing-player impact (biggest injury/absence signal):")
print(big_absence.merge(team_games[['GAME_ID', 'TEAM_ABBREVIATION', 'GAME_DATE']],
                          on=['GAME_ID', 'TEAM_ABBREVIATION'], how='left'))

Player log with rolling features: (150088, 36)
team_games available: (14108, 74)


KeyboardInterrupt: 

In [30]:
missing_impact_df = compute_missing_player_impact(player_log_roll, team_games, top_n=8)
print(f"Missing-impact computed for {len(missing_impact_df)} team-games")
print(f"NaN count: {missing_impact_df['MISSING_PLAYERS_COUNT'].isna().sum()}")
print(f"\nMISSING_PLAYERS_COUNT distribution:\n{missing_impact_df['MISSING_PLAYERS_COUNT'].describe()}")
print(f"\nMISSING_PTS_IMPACT distribution:\n{missing_impact_df['MISSING_PTS_IMPACT'].describe()}")

big_absence = missing_impact_df.nlargest(5, 'MISSING_PTS_IMPACT')
print(f"\nTop 5 games by missing-player impact:")
print(big_absence.merge(team_games[['GAME_ID', 'TEAM_ABBREVIATION', 'GAME_DATE']], on=['GAME_ID', 'TEAM_ABBREVIATION'], how='left'))

Missing-impact computed for 14108 team-games
NaN count: 180

MISSING_PLAYERS_COUNT distribution:
count    13928.000000
mean         1.841973
std          1.454935
min          0.000000
25%          1.000000
50%          2.000000
75%          3.000000
max          8.000000
Name: MISSING_PLAYERS_COUNT, dtype: float64

MISSING_PTS_IMPACT distribution:
count    13928.000000
mean        26.998208
std         25.054757
min          0.000000
25%          8.000000
50%         21.650000
75%         39.500000
max        154.700000
Name: MISSING_PTS_IMPACT, dtype: float64

Top 5 games by missing-player impact:
      GAME_ID TEAM_ABBREVIATION  MISSING_PLAYERS_COUNT  MISSING_MINUTES_SHARE  \
0  0022401173               PHI                    8.0               1.150417   
1  0022101130               POR                    8.0               1.139583   
2  0022101146               POR                    8.0               1.139583   
3  0022101185               POR                    8.0               

In [31]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)
print(big_absence.merge(team_games[['GAME_ID', 'TEAM_ABBREVIATION', 'GAME_DATE']],
                          on=['GAME_ID', 'TEAM_ABBREVIATION'], how='left').to_string(index=False))

   GAME_ID TEAM_ABBREVIATION  MISSING_PLAYERS_COUNT  MISSING_MINUTES_SHARE  MISSING_PTS_IMPACT  GAME_DATE
0022401173               PHI                    8.0               1.150417               154.7 2025-04-11
0022101130               POR                    8.0               1.139583               150.6 2022-03-28
0022101146               POR                    8.0               1.139583               150.6 2022-03-30
0022101185               POR                    8.0               1.139583               150.6 2022-04-05
0022101230               POR                    8.0               1.142083               150.0 2022-04-10


In [32]:
# --------------------------------------------------------------------------
# Merge missing-player impact into model_ready_df (both HOME and AWAY sides)
# --------------------------------------------------------------------------
missing_impact_home = missing_impact_df.add_suffix('_HOME').rename(
    columns={'GAME_ID_HOME': 'GAME_ID', 'TEAM_ABBREVIATION_HOME': 'TEAM_ABBREVIATION_HOME'})
missing_impact_away = missing_impact_df.add_suffix('_AWAY').rename(
    columns={'GAME_ID_AWAY': 'GAME_ID', 'TEAM_ABBREVIATION_AWAY': 'TEAM_ABBREVIATION_AWAY'})

model_ready_df = pd.merge(model_ready_df, missing_impact_home, on=['GAME_ID', 'TEAM_ABBREVIATION_HOME'], how='left')
model_ready_df = pd.merge(model_ready_df, missing_impact_away, on=['GAME_ID', 'TEAM_ABBREVIATION_AWAY'], how='left')

print(f"model_ready_df shape: {model_ready_df.shape}")

# Re-run data prep -- this will also drop rows with NaN missing-impact (season openers),
# on top of the rolling-feature NaNs we already handle
model_df = prepare_modeling_data(model_ready_df)
model_df = model_df.dropna(subset=['MISSING_PLAYERS_COUNT_HOME', 'MISSING_PLAYERS_COUNT_AWAY']).reset_index(drop=True)
print(f"model_df shape after dropping missing-injury-data rows: {model_df.shape}")

feature_cols = get_feature_columns(model_df)
injury_features = [c for c in feature_cols if 'MISSING' in c]
print(f"Injury-related features now included: {injury_features}")

model_ready_df shape: (7054, 154)
Dropped 95 rows with incomplete rolling features (7054 -> 6959)
model_df shape after dropping missing-injury-data rows: (6959, 154)
Selected 70 leakage-safe feature columns.
Injury-related features now included: []


In [36]:
# Download + save the updated file into your NBA folder first, overwriting the old copy, then:
importlib.reload(nba_pipeline_consolidated)
from nba_pipeline_consolidated import *

feature_cols = get_feature_columns(model_df)
injury_features = [c for c in feature_cols if 'MISSING' in c]

print(f"Feature count: {len(feature_cols)} (should be 76, up from 70)")
print(f"Injury features: {injury_features}")
print(f"Expected: 6 columns -- MISSING_PLAYERS_COUNT/MISSING_MINUTES_SHARE/MISSING_PTS_IMPACT, each _HOME and _AWAY")

Selected 76 leakage-safe feature columns.
Feature count: 76 (should be 76, up from 70)
Injury features: ['MISSING_PLAYERS_COUNT_HOME', 'MISSING_MINUTES_SHARE_HOME', 'MISSING_PTS_IMPACT_HOME', 'MISSING_PLAYERS_COUNT_AWAY', 'MISSING_MINUTES_SHARE_AWAY', 'MISSING_PTS_IMPACT_AWAY']
Expected: 6 columns -- MISSING_PLAYERS_COUNT/MISSING_MINUTES_SHARE/MISSING_PTS_IMPACT, each _HOME and _AWAY


In [37]:
print("=== Moneyline WITH injury/availability features ===")
moneyline_results_injury = run_walk_forward_moneyline(model_df, feature_cols)

print("\n=== Feature importance check -- do the injury features matter at all? ===")
X_full = model_df[feature_cols].fillna(model_df[feature_cols].median())
y_full = model_df['HOME_WIN']
check_model = XGBClassifier(max_depth=3, learning_rate=0.03, n_estimators=150,
                             min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
                             eval_metric='logloss', random_state=42)
check_model.fit(X_full, y_full)
importances = pd.Series(check_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(f"\nInjury feature ranks (out of {len(feature_cols)} total):")
for feat in injury_features:
    rank = list(importances.index).index(feat) + 1
    print(f"  {feat}: rank {rank}, importance {importances[feat]:.4f}")
print(f"\nTop 10 overall:\n{importances.head(10)}")

=== Moneyline WITH injury/availability features ===
Season 2020-21: naive=0.543  logreg=0.585  xgb=0.598
Season 2021-22: naive=0.543  logreg=0.614  xgb=0.614
Season 2022-23: naive=0.581  logreg=0.605  xgb=0.600
Season 2023-24: naive=0.543  logreg=0.667  xgb=0.651
Season 2024-25: naive=0.548  logreg=0.661  xgb=0.634

=== Feature importance check -- do the injury features matter at all? ===

Injury feature ranks (out of 76 total):
  MISSING_PLAYERS_COUNT_HOME: rank 21, importance 0.0134
  MISSING_MINUTES_SHARE_HOME: rank 7, importance 0.0199
  MISSING_PTS_IMPACT_HOME: rank 8, importance 0.0198
  MISSING_PLAYERS_COUNT_AWAY: rank 10, importance 0.0188
  MISSING_MINUTES_SHARE_AWAY: rank 12, importance 0.0157
  MISSING_PTS_IMPACT_AWAY: rank 18, importance 0.0145

Top 10 overall:
WON_ROLL10_SPLIT_HOME         0.054797
WON_ROLL10_HOME               0.044100
WON_ROLL10_AWAY               0.037574
WON_ROLL10_SPLIT_AWAY         0.033380
IS_BACK_TO_BACK_AWAY          0.022152
OFF_RATING_ROLL10_AWA

In [38]:
# --------------------------------------------------------------------------
# The real test: does injury data close the gap vs market we found earlier?
# --------------------------------------------------------------------------
train_seasons_v2 = ['2019-20', '2020-21']
calib_season_v2 = '2021-22'
eval_season_v2 = '2022-23'

train_df_v2 = model_df[model_df['SEASON'].isin(train_seasons_v2)]
calib_df_v2 = model_df[model_df['SEASON'] == calib_season_v2]
eval_df_v2 = model_df[model_df['SEASON'] == eval_season_v2]

X_train_v2 = train_df_v2[feature_cols].fillna(train_df_v2[feature_cols].median())
y_train_v2 = train_df_v2['HOME_WIN']
train_medians_v2 = X_train_v2.median()

X_calib_v2 = calib_df_v2[feature_cols].fillna(train_medians_v2)
X_eval_v2 = eval_df_v2[feature_cols].fillna(train_medians_v2)

model_v2 = XGBClassifier(max_depth=3, learning_rate=0.03, n_estimators=150,
                          min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
                          eval_metric='logloss', random_state=42)
model_v2.fit(X_train_v2, y_train_v2)

raw_proba_calib_v2 = model_v2.predict_proba(X_calib_v2)[:, 1]
platt_v2 = fit_platt_calibrator(raw_proba_calib_v2, calib_df_v2['HOME_WIN'].values)

raw_proba_eval_v2 = model_v2.predict_proba(X_eval_v2)[:, 1]
calibrated_proba_eval_v2 = apply_platt_calibration(platt_v2, raw_proba_eval_v2)

print("=== WITH injury features: calibrated vs market (2022-23) ===")
evaluate_vs_market(eval_df_v2['HOME_WIN'], calibrated_proba_eval_v2, eval_df_v2['IMPLIED_WIN_PROB_HOME'])

print("\n--- For reference, WITHOUT injury features (from earlier): ---")
print("Our log loss: 0.6678  Market log loss: 0.6391  (we lost by 0.0287)")
print("Our Brier:    0.2376  Market Brier:    0.2234  (we lost by 0.0142)")

=== WITH injury features: calibrated vs market (2022-23) ===
=== Model vs Market (648 games with odds available) ===
Log Loss  -- Ours: 0.6645  Market: 0.6391  (market beat us by 0.0253)
Brier     -- Ours: 0.2360  Market: 0.2234  (market beat us by 0.0125)

--- For reference, WITHOUT injury features (from earlier): ---
Our log loss: 0.6678  Market log loss: 0.6391  (we lost by 0.0287)
Our Brier:    0.2376  Market Brier:    0.2234  (we lost by 0.0142)


In [39]:
# --------------------------------------------------------------------------
# Re-tune XGBoost on the CURRENT full feature set (76 features, incl. injury)
# Same discipline as before: holdout season (2024-25) completely excluded
# from tuning, so the final evaluation stays honest.
# --------------------------------------------------------------------------
tuning_results_v2 = tune_xgboost_walk_forward(model_df, feature_cols, holdout_season='2024-25')

print("=== Top 10 configs (76-feature set) ===")
print(tuning_results_v2.head(10).to_string(index=False))

best_params_v2 = tuning_results_v2.iloc[0][['max_depth', 'learning_rate', 'n_estimators', 'min_child_weight']].to_dict()
best_params_v2['max_depth'] = int(best_params_v2['max_depth'])
best_params_v2['n_estimators'] = int(best_params_v2['n_estimators'])
best_params_v2['min_child_weight'] = int(best_params_v2['min_child_weight'])
print(f"\nBest params: {best_params_v2}")

# Compare against our OLD tuned params (from the 70-feature set) for reference
print(f"\nOld best params (70-feature set, for comparison): "
      f"max_depth=3, learning_rate=0.03, n_estimators=150, min_child_weight=1")

=== Top 10 configs (76-feature set) ===
 max_depth  learning_rate  n_estimators  min_child_weight  avg_logloss  avg_accuracy  std_accuracy
         3           0.03           150                 1     0.647300      0.621593      0.024534
         3           0.03           150                 5     0.648033      0.622560      0.018135
         4           0.03           150                 5     0.649211      0.617712      0.026116
         4           0.03           150                 1     0.650160      0.620738      0.021572
         5           0.03           150                 5     0.650589      0.620352      0.022657
         3           0.03           250                 1     0.650797      0.619882      0.021097
         3           0.05           150                 5     0.651238      0.621999      0.020199
         3           0.03           250                 5     0.651264      0.620030      0.023543
         3           0.05           150                 1     0.65398

In [40]:
final_model_v2, X_test_final_v2, y_test_final_v2, proba_final_v2 = final_holdout_evaluation(
    model_df, feature_cols, best_params_v2
)

print(f"\n--- For reference, our PREVIOUS best (70 features, old tuning): ---")
print(f"Accuracy: 0.643  Precision: 0.646  Recall: 0.769  Brier: 0.222")

=== FINAL HOLDOUT EVALUATION: 2024-25 ===
Naive baseline accuracy: 0.548
Tuned XGBoost accuracy:  0.641
Tuned XGBoost log loss:  0.624
Tuned XGBoost precision: 0.647
Tuned XGBoost recall:    0.758
Tuned XGBoost Brier:     0.218

--- For reference, our PREVIOUS best (70 features, old tuning): ---
Accuracy: 0.643  Precision: 0.646  Recall: 0.769  Brier: 0.222


In [41]:
print(f"\nProbability range check: min={proba_final_v2.min():.3f}, max={proba_final_v2.max():.3f}")
print(f"(For reference, the old model's range was 0.263 to 0.767)")


Probability range check: min=0.144, max=0.865
(For reference, the old model's range was 0.263 to 0.767)


In [42]:
# --------------------------------------------------------------------------
# Does the wider probability range close more of the market gap?
# --------------------------------------------------------------------------
train_seasons_v3 = ['2019-20', '2020-21']
calib_season_v3 = '2021-22'
eval_season_v3 = '2022-23'

train_df_v3 = model_df[model_df['SEASON'].isin(train_seasons_v3)]
calib_df_v3 = model_df[model_df['SEASON'] == calib_season_v3]
eval_df_v3 = model_df[model_df['SEASON'] == eval_season_v3]

X_train_v3 = train_df_v3[feature_cols].fillna(train_df_v3[feature_cols].median())
y_train_v3 = train_df_v3['HOME_WIN']
train_medians_v3 = X_train_v3.median()

X_calib_v3 = calib_df_v3[feature_cols].fillna(train_medians_v3)
X_eval_v3 = eval_df_v3[feature_cols].fillna(train_medians_v3)

model_v3 = XGBClassifier(**best_params_v2, subsample=0.8, colsample_bytree=0.8,
                          eval_metric='logloss', random_state=42)
model_v3.fit(X_train_v3, y_train_v3)

raw_proba_calib_v3 = model_v3.predict_proba(X_calib_v3)[:, 1]
platt_v3 = fit_platt_calibrator(raw_proba_calib_v3, calib_df_v3['HOME_WIN'].values)

raw_proba_eval_v3 = model_v3.predict_proba(X_eval_v3)[:, 1]
calibrated_proba_eval_v3 = apply_platt_calibration(platt_v3, raw_proba_eval_v3)

print("=== TUNED + injury features: calibrated vs market (2022-23) ===")
evaluate_vs_market(eval_df_v3['HOME_WIN'], calibrated_proba_eval_v3, eval_df_v3['IMPLIED_WIN_PROB_HOME'])

print("\n--- Progression across this whole session ---")
print("Baseline (no injury, no tuning):  logloss=0.6690  gap=-0.0299")
print("Injury features, untuned:          logloss=0.6645  gap=-0.0253")
print(f"Injury features + tuned:           logloss={{result computed above}}")

=== TUNED + injury features: calibrated vs market (2022-23) ===
=== Model vs Market (648 games with odds available) ===
Log Loss  -- Ours: 0.6645  Market: 0.6391  (market beat us by 0.0253)
Brier     -- Ours: 0.2360  Market: 0.2234  (market beat us by 0.0125)

--- Progression across this whole session ---
Baseline (no injury, no tuning):  logloss=0.6690  gap=-0.0299
Injury features, untuned:          logloss=0.6645  gap=-0.0253
Injury features + tuned:           logloss={result computed above}


In [43]:
# --------------------------------------------------------------------------
# Add CLOSING_SPREAD_HOME / CLOSING_TOTAL as legitimate pregame input features
# NOTE: unlike MONEYLINE/IMPLIED_WIN_PROB (huge coverage gaps in 2023-25),
# spread/total have ~100% coverage across all 6 seasons -- confirmed earlier.
# --------------------------------------------------------------------------
print(f"CLOSING_SPREAD_HOME coverage: {model_df['CLOSING_SPREAD_HOME'].notna().mean():.1%}")
print(f"CLOSING_TOTAL coverage: {model_df['CLOSING_TOTAL'].notna().mean():.1%}")

feature_cols_with_market = feature_cols + ['CLOSING_SPREAD_HOME', 'CLOSING_TOTAL']
print(f"\nFeature count: {len(feature_cols_with_market)} (was {len(feature_cols)})")

print("\n=== Moneyline WITH market line as a feature (ceiling test) ===")
moneyline_results_market = run_walk_forward_moneyline(model_df, feature_cols_with_market)

CLOSING_SPREAD_HOME coverage: 100.0%
CLOSING_TOTAL coverage: 100.0%

Feature count: 78 (was 76)

=== Moneyline WITH market line as a feature (ceiling test) ===
Season 2020-21: naive=0.543  logreg=0.616  xgb=0.610
Season 2021-22: naive=0.543  logreg=0.661  xgb=0.653
Season 2022-23: naive=0.581  logreg=0.646  xgb=0.636
Season 2023-24: naive=0.543  logreg=0.694  xgb=0.679
Season 2024-25: naive=0.548  logreg=0.686  xgb=0.667


In [44]:
# --------------------------------------------------------------------------
# Does market-line-as-feature let us match/beat the market itself?
# --------------------------------------------------------------------------
train_df_v4 = model_df[model_df['SEASON'].isin(['2019-20', '2020-21'])]
calib_df_v4 = model_df[model_df['SEASON'] == '2021-22']
eval_df_v4 = model_df[model_df['SEASON'] == '2022-23']

X_train_v4 = train_df_v4[feature_cols_with_market].fillna(train_df_v4[feature_cols_with_market].median())
y_train_v4 = train_df_v4['HOME_WIN']
train_medians_v4 = X_train_v4.median()

X_calib_v4 = calib_df_v4[feature_cols_with_market].fillna(train_medians_v4)
X_eval_v4 = eval_df_v4[feature_cols_with_market].fillna(train_medians_v4)

model_v4 = XGBClassifier(max_depth=3, learning_rate=0.03, n_estimators=150,
                          min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
                          eval_metric='logloss', random_state=42)
model_v4.fit(X_train_v4, y_train_v4)

raw_proba_calib_v4 = model_v4.predict_proba(X_calib_v4)[:, 1]
platt_v4 = fit_platt_calibrator(raw_proba_calib_v4, calib_df_v4['HOME_WIN'].values)
raw_proba_eval_v4 = model_v4.predict_proba(X_eval_v4)[:, 1]
calibrated_proba_eval_v4 = apply_platt_calibration(platt_v4, raw_proba_eval_v4)

print("=== Market-line-as-feature: calibrated vs market itself (2022-23) ===")
evaluate_vs_market(eval_df_v4['HOME_WIN'], calibrated_proba_eval_v4, eval_df_v4['IMPLIED_WIN_PROB_HOME'])

print("\n--- Full progression this session ---")
print("Baseline (no injury, no market feature):     logloss=0.6690  gap=-0.0299")
print("+ Injury features:                            logloss=0.6645  gap=-0.0253")
print("+ Market line as feature:                     logloss=(above)")

=== Market-line-as-feature: calibrated vs market itself (2022-23) ===
=== Model vs Market (648 games with odds available) ===
Log Loss  -- Ours: 0.6484  Market: 0.6391  (market beat us by 0.0093)
Brier     -- Ours: 0.2283  Market: 0.2234  (market beat us by 0.0049)

--- Full progression this session ---
Baseline (no injury, no market feature):     logloss=0.6690  gap=-0.0299
+ Injury features:                            logloss=0.6645  gap=-0.0253
+ Market line as feature:                     logloss=(above)


In [1]:
pip install beautifulsoup4 requests

Note: you may need to restart the kernel to use updated packages.


In [4]:
import importlib
import nba_pipeline_consolidated
importlib.reload(nba_pipeline_consolidated)
from nba_pipeline_consolidated import *

print("fetch_espn_injury_report loaded:", 'fetch_espn_injury_report' in dir())

fetch_espn_injury_report loaded: True


In [7]:
import subprocess
result = subprocess.run(['py', '-m', 'pip', 'install', 'lxml'], capture_output=True, text=True)
print(result.stdout[-500:])
print(result.stderr[-500:])

--------------------------- 0.3/4.1 MB ? eta -:--:--
   ------- -------------------------------- 0.8/4.1 MB 2.0 MB/s eta 0:00:02
   ------------ --------------------------- 1.3/4.1 MB 2.3 MB/s eta 0:00:02
   ----------------------- ---------------- 2.4/4.1 MB 3.2 MB/s eta 0:00:01
   -------------------------------------- - 3.9/4.1 MB 4.1 MB/s eta 0:00:01
   ---------------------------------------- 4.1/4.1 MB 4.0 MB/s  0:00:01




In [10]:
import requests
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36'}
resp = requests.get('https://www.espn.com/nba/injuries', headers=headers, timeout=30)
print(f"Status code: {resp.status_code}")
print(f"Content length: {len(resp.content)}")
print(f"Content-Type header: {resp.headers.get('Content-Type')}")
print(f"Content-Encoding header: {resp.headers.get('Content-Encoding')}")
print("\n--- First 2000 chars of resp.text ---")
print(resp.text[:2000])

Status code: 202
Content length: 0
Content-Type header: text/html; charset=UTF-8
Content-Encoding header: None

--- First 2000 chars of resp.text ---



In [4]:
import subprocess
result = subprocess.run(['py', '-m', 'pip', 'install', 'pdfplumber', 'beautifulsoup4', 'lxml'], capture_output=True, text=True)
print(result.stdout[-800:])
print(result.stderr[-500:])

six]
   ------------------------ --------------- 3/5 [pdfminer.six]
   ------------------------ --------------- 3/5 [pdfminer.six]
   ------------------------ --------------- 3/5 [pdfminer.six]
   ------------------------ --------------- 3/5 [pdfminer.six]
   -------------------------------- ------- 4/5 [pdfplumber]
   -------------------------------- ------- 4/5 [pdfplumber]
   -------------------------------- ------- 4/5 [pdfplumber]
   -------------------------------- ------- 4/5 [pdfplumber]
   -------------------------------- ------- 4/5 [pdfplumber]
   -------------------------------- ------- 4/5 [pdfplumber]
   ---------------------------------------- 5/5 [pdfplumber]


The script pypdfium2.exe is installed in 'C:\Users\egbee\AppData\Local\Python\pythoncore-3.14-64\Scripts' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress

In [3]:
import importlib
import nba_pipeline_consolidated
importlib.reload(nba_pipeline_consolidated)
from nba_pipeline_consolidated import *
print("Loaded:", 'find_latest_injury_report_url' in dir())

Loaded: True


In [6]:
import subprocess
result = subprocess.run(['py', '-m', 'pip', 'install', 'pdfplumber'], capture_output=True, text=True)
print(result.stdout[-300:])

e-3.14-64\Lib\site-packages (from cryptography>=36.0.0->pdfminer.six==20260107->pdfplumber) (2.1.1)



In [7]:
url = find_latest_injury_report_url()
if url:
    raw_text = extract_injury_pdf_text(url)
    print(raw_text[:3000])  # first 3000 chars -- enough to see real structure

  Not found (403): 05_30PM
  Not found (403): 05_15PM
  Not found (403): 04_30PM
  Not found (403): 04_15PM
  Not found (403): 03_30PM
  Not found (403): 03_00PM
  Not found (403): 02_30PM
  Not found (403): 02_15PM
  Not found (403): 01_30PM
  Not found (403): 01_00PM
  Not found (403): 12_45PM
  Not found (403): 12_30PM
  Not found (403): 12_00PM
  Not found (403): 08_30AM
  Not found (403): 08_00AM


In [8]:
import requests

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36',
    'Referer': 'https://official.nba.com/',
    'Accept': 'application/pdf,*/*',
}

known_good_url = 'https://ak-static.cms.nba.com/referee/injury/Injury-Report_2026-03-11_03_30PM.pdf'
resp = requests.get(known_good_url, headers=headers, timeout=30)
print(f"Status: {resp.status_code}")
print(f"Content length: {len(resp.content)}")

Status: 200
Content length: 71006


In [9]:
raw_text = extract_injury_pdf_text(known_good_url)
print(raw_text[:3000])

PDF has 4 pages

--- PAGE 1 ---
Injury Report: 03/11/26 03:30 PM
GameDate GameTime Matchup Team PlayerName CurrentStatus Reason
03/11/2026 07:30(ET) CLE@ORL ClevelandCavaliers Allen,Jarrett Out Injury/Illness-RightKnee;Tendonitis
Injury/Illness-LeftIndexFinger;
Ellis,Keon Available
Fracture
Harden,James Available Injury/Illness-RightThumb;Fracture
Injury/Illness-RightQuadricep;
Proctor,Tyrese Out
Strain
Sarr,Olivier Out GLeague-Two-Way
Injury/Illness-LeftFoot;Surgery-
Strus,Max Out
JonesFracture
Injury/Illness-LeftLateral
OrlandoMagic Black,Anthony Out
Abdominal;Strain
Isaac,Jonathan Available Injury/Illness-LeftKnee;Strain
Richardson,Jase Available Injury/Illness-LowBack;Strain
Injury/Illness-LeftHighAnkle;Sprain
Wagner,Franz Out
injurymanagement
08:00(ET) TOR@NOP TorontoRaptors Hepburn,Chucky Out GLeague-Two-Way
Injury/Illness-RightHand;Middle
Jackson-Davis,Trayce Questionable
Finger-Dislocation
Lawson,A.J. Out GLeague-Two-Way
Murray-Boyles,Collin Out Injury/Illness-LeftThumb;Sprain


In [10]:
import pdfplumber

local_path = 'injury_report_temp.pdf' if os.name == 'nt' else '/tmp/injury_report.pdf'

with pdfplumber.open(local_path) as pdf:
    page = pdf.pages[0]

    print("=== STRATEGY 1: extract_tables() with DEFAULT settings ===")
    tables_default = page.extract_tables()
    print(f"Found {len(tables_default)} tables")
    if tables_default:
        for row in tables_default[0][:8]:
            print(row)

    print("\n=== STRATEGY 2: extract_tables() with strategy='text' ===")
    tables_text = page.extract_tables(table_settings={"vertical_strategy": "text", "horizontal_strategy": "text"})
    print(f"Found {len(tables_text)} tables")
    if tables_text:
        for row in tables_text[0][:8]:
            print(row)

    print("\n=== STRATEGY 3: raw word-level extraction (fallback, gives x/y coordinates) ===")
    words = page.extract_words()
    print(f"Found {len(words)} words. First 20 with positions:")
    for w in words[:20]:
        print(f"  '{w['text']}' at x0={w['x0']:.0f}, x1={w['x1']:.0f}, top={w['top']:.0f}")

=== STRATEGY 1: extract_tables() with DEFAULT settings ===
Found 2 tables
['Injury Report: 03/11/26 03:30 PM', 'Injury Report: 03/11/26 03:30 PM']

=== STRATEGY 2: extract_tables() with strategy='text' ===
Found 1 tables
['GameTime', 'Matchup', 'Team', 'PlayerName', 'CurrentStatus', 'Reason']
['', '', '', '', '', '']
['07:30(ET)', 'CLE@ORL', 'ClevelandCavaliers', 'Allen,Jarrett', 'Out', 'Injury/Illness-RightKnee;Tendonitis']
['', '', '', '', '', '']
['', '', '', '', '', 'Injury/Illness-LeftIndexFinger;']
['', '', '', 'Ellis,Keon', 'Available', '']
['', '', '', '', '', 'Fracture']
['', '', '', '', '', '']

=== STRATEGY 3: raw word-level extraction (fallback, gives x/y coordinates) ===
Found 69 words. First 20 with positions:
  'Injury' at x0=289, x1=349, top=46
  'Report:' at x0=356, x1=432, top=46
  '03/11/26' at x0=438, x1=534, top=46
  '03:30' at x0=540, x1=596, top=46
  'PM' at x0=602, x1=637, top=46
  'GameDate' at x0=23, x1=75, top=108
  'GameTime' at x0=120, x1=173, top=108
  'Ma

In [5]:
import importlib
import nba_pipeline_consolidated
importlib.reload(nba_pipeline_consolidated)
from nba_pipeline_consolidated import *
print("Loaded:", 'parse_injury_report_pdf' in dir())

Loaded: True


In [2]:
import importlib
import nba_pipeline_consolidated
importlib.reload(nba_pipeline_consolidated)
from nba_pipeline_consolidated import *
print("Loaded:", 'parse_injury_report_pdf' in dir())

known_good_url = 'https://ak-static.cms.nba.com/referee/injury/Injury-Report_2026-03-11_03_30PM.pdf'
injury_df = parse_injury_report_pdf(known_good_url)

print(f"Shape: {injury_df.shape}")
print(injury_df.head(20).to_string(index=False))
print(f"\nCurrentStatus value counts:\n{injury_df['CurrentStatus'].value_counts()}")
print(f"\nUnique teams found: {injury_df['Team'].nunique()}")

Loaded: True
Shape: (14, 6)
 GameTime Matchup               Team           PlayerName CurrentStatus                                                                    Reason
07:30(ET) CLE@ORL ClevelandCavaliers        Allen,Jarrett           Out                                       Injury/Illness-RightKnee;Tendonitis
07:30(ET) CLE@ORL ClevelandCavaliers           Ellis,Keon     Available                                  Injury/Illness-LeftIndexFinger; Fracture
07:30(ET) CLE@ORL ClevelandCavaliers         Harden,James     Available                                        Injury/Illness-RightThumb;Fracture
07:30(ET) CLE@ORL ClevelandCavaliers       Proctor,Tyrese           Out                                     Injury/Illness-RightQuadricep; Strain
07:30(ET) CLE@ORL ClevelandCavaliers         Sarr,Olivier           Out                                                           GLeague-Two-Way
07:30(ET) CLE@ORL ClevelandCavaliers            Strus,Max           Out Injury/Illness-LeftFoot;

In [3]:
import pdfplumber

local_path = 'injury_report_temp.pdf' if os.name == 'nt' else '/tmp/injury_report.pdf'

with pdfplumber.open(local_path) as pdf:
    print(f"Total pages: {len(pdf.pages)}")
    for i, page in enumerate(pdf.pages):
        tables = page.extract_tables(table_settings={"vertical_strategy": "text", "horizontal_strategy": "text"})
        print(f"\nPage {i+1}: {len(tables)} table(s) found")
        for j, t in enumerate(tables):
            print(f"  Table {j}: {len(t)} rows, header={t[0] if t else 'EMPTY'}")

Total pages: 4

Page 1: 1 table(s) found
  Table 0: 41 rows, header=['GameTime', 'Matchup', 'Team', 'PlayerName', 'CurrentStatus', 'Reason']

Page 2: 1 table(s) found
  Table 0: 41 rows, header=['', 'Poeltl,Jakob', 'Questionable', 'Injury/Illness-Illness;Illness']

Page 3: 1 table(s) found
  Table 0: 39 rows, header=['', 'SacramentoKings', '', '', 'NOTYETSUBMITTED']

Page 4: 1 table(s) found
  Table 0: 33 rows, header=['', '', 'DetroitPistons', 'NOTYETSUBMITTED']


In [4]:
import importlib
import nba_pipeline_consolidated
importlib.reload(nba_pipeline_consolidated)
from nba_pipeline_consolidated import *

known_good_url = 'https://ak-static.cms.nba.com/referee/injury/Injury-Report_2026-03-11_03_30PM.pdf'
injury_df = parse_injury_report_pdf(known_good_url)

print(f"Shape: {injury_df.shape}")
print(injury_df.head(30).to_string(index=False))
print(f"\nCurrentStatus value counts:\n{injury_df['CurrentStatus'].value_counts()}")
print(f"\nUnique teams found: {injury_df['Team'].nunique()}")
print(f"\nTeams found: {sorted(injury_df['Team'].unique())}")

Shape: (25, 6)
 GameTime        Matchup               Team           PlayerName                      CurrentStatus                                                                    Reason
07:30(ET)        CLE@ORL ClevelandCavaliers        Allen,Jarrett                                Out                                       Injury/Illness-RightKnee;Tendonitis
07:30(ET)        CLE@ORL ClevelandCavaliers           Ellis,Keon                          Available                                  Injury/Illness-LeftIndexFinger; Fracture
07:30(ET)        CLE@ORL ClevelandCavaliers         Harden,James                          Available                                        Injury/Illness-RightThumb;Fracture
07:30(ET)        CLE@ORL ClevelandCavaliers       Proctor,Tyrese                                Out                                     Injury/Illness-RightQuadricep; Strain
07:30(ET)        CLE@ORL ClevelandCavaliers         Sarr,Olivier                                Out                

In [5]:
import importlib
import nba_pipeline_consolidated
importlib.reload(nba_pipeline_consolidated)
from nba_pipeline_consolidated import *

known_good_url = 'https://ak-static.cms.nba.com/referee/injury/Injury-Report_2026-03-11_03_30PM.pdf'
injury_df = parse_injury_report_pdf(known_good_url)

print(f"Shape: {injury_df.shape}")
print(injury_df.head(30).to_string(index=False))
print(f"\nCurrentStatus value counts:\n{injury_df['CurrentStatus'].value_counts()}")
print(f"\nUnique teams found: {injury_df['Team'].nunique()}")
print(f"\nTeams: {sorted(injury_df['Team'].unique())}")

Shape: (43, 6)
 GameTime Matchup               Team           PlayerName CurrentStatus                                                                     Reason
07:30(ET) CLE@ORL ClevelandCavaliers        Allen,Jarrett           Out                                        Injury/Illness-RightKnee;Tendonitis
07:30(ET) CLE@ORL ClevelandCavaliers           Ellis,Keon     Available                                   Injury/Illness-LeftIndexFinger; Fracture
07:30(ET) CLE@ORL ClevelandCavaliers         Harden,James     Available                                         Injury/Illness-RightThumb;Fracture
07:30(ET) CLE@ORL ClevelandCavaliers       Proctor,Tyrese           Out                                      Injury/Illness-RightQuadricep; Strain
07:30(ET) CLE@ORL ClevelandCavaliers         Sarr,Olivier           Out                                                            GLeague-Two-Way
07:30(ET) CLE@ORL ClevelandCavaliers            Strus,Max           Out                Injury/Illness-L

TypeError: '<' not supported between instances of 'float' and 'str'

In [6]:
import importlib
import nba_pipeline_consolidated
importlib.reload(nba_pipeline_consolidated)
from nba_pipeline_consolidated import *

known_good_url = 'https://ak-static.cms.nba.com/referee/injury/Injury-Report_2026-03-11_03_30PM.pdf'
injury_df = parse_injury_report_pdf(known_good_url)

print(f"Shape: {injury_df.shape}")
print(f"Any missing Team values: {injury_df['Team'].isna().sum()}")
print(f"CurrentStatus value counts:\n{injury_df['CurrentStatus'].value_counts()}")
print(f"Unique teams found: {injury_df['Team'].nunique()}")
print(f"Teams: {sorted(injury_df['Team'].unique())}")

Shape: (43, 6)
Any missing Team values: 0
CurrentStatus value counts:
CurrentStatus
Out             33
Available        4
Questionable     3
Probable         2
Doubtful         1
Name: count, dtype: int64
Unique teams found: 9
Teams: ['ClevelandCavaliers', 'DenverNuggets', 'HoustonRockets', 'MinnesotaTimberwolves', 'NewOrleansPelicans', 'NewYorkKnicks', 'OrlandoMagic', 'TorontoRaptors', 'UtahJazz']


In [1]:
import importlib
import nba_pipeline_consolidated
importlib.reload(nba_pipeline_consolidated)
from nba_pipeline_consolidated import *
print("Loaded:", 'run_true_holdout_validation' in dir())

Loaded: True


In [2]:
try:
    print(f"model_df is in memory: {model_df.shape}")
    print(f"Seasons present: {sorted(model_df['SEASON'].unique())}")
    HAVE_EXISTING_DATA = True
except NameError:
    print("model_df not in memory -- will need a full rebuild (slower).")
    HAVE_EXISTING_DATA = False

model_df not in memory -- will need a full rebuild (slower).


In [5]:
model_locked, platt_locked, holdout_df, holdout_proba = run_true_holdout_validation(existing_model_df=None)

No existing model_df provided -- rebuilding 2019-20 through 2024-25 from scratch.
(This will take a while -- 6 seasons of game + player log fetches.)
Fetching game log for 2019-20...
Building team-game features for 2019-20...
Fetching player log for 2019-20 (injury/availability features)...
Fetching game log for 2020-21...
Building team-game features for 2020-21...
Fetching player log for 2020-21 (injury/availability features)...
Fetching game log for 2021-22...
Building team-game features for 2021-22...
Fetching player log for 2021-22 (injury/availability features)...
Fetching game log for 2022-23...
Building team-game features for 2022-23...
Fetching player log for 2022-23 (injury/availability features)...
Fetching game log for 2023-24...
Building team-game features for 2023-24...
Fetching player log for 2023-24 (injury/availability features)...
Fetching game log for 2024-25...
Building team-game features for 2024-25...
Fetching player log for 2024-25 (injury/availability features)..

NameError: name 'fit_platt_calibrator' is not defined

In [6]:
import importlib
import nba_pipeline_consolidated
importlib.reload(nba_pipeline_consolidated)
from nba_pipeline_consolidated import *
print("Loaded:", 'fit_platt_calibrator' in dir())

model_locked, platt_locked, holdout_df, holdout_proba = run_true_holdout_validation(existing_model_df=None)

Loaded: True
No existing model_df provided -- rebuilding 2019-20 through 2024-25 from scratch.
(This will take a while -- 6 seasons of game + player log fetches.)
Fetching game log for 2019-20...
Building team-game features for 2019-20...
Fetching player log for 2019-20 (injury/availability features)...
Fetching game log for 2020-21...
Building team-game features for 2020-21...
Fetching player log for 2020-21 (injury/availability features)...
Fetching game log for 2021-22...
Building team-game features for 2021-22...
Fetching player log for 2021-22 (injury/availability features)...
Fetching game log for 2022-23...
Building team-game features for 2022-23...
Fetching player log for 2022-23 (injury/availability features)...
Fetching game log for 2023-24...
Building team-game features for 2023-24...
Fetching player log for 2023-24 (injury/availability features)...
Fetching game log for 2024-25...
Building team-game features for 2024-25...
Fetching player log for 2024-25 (injury/availabilit

In [7]:
naive_acc = accuracy_score(holdout_df['HOME_WIN'], np.ones(len(holdout_df)))
preds_holdout = (holdout_proba >= 0.5).astype(int)

print("=" * 70)
print("TRUE HOLDOUT RESULTS: 2025-26 (LOCKED CONFIG, SEEN EXACTLY ONCE)")
print("=" * 70)
print(f"Games evaluated:         {len(holdout_df)}")
print(f"Naive baseline accuracy: {naive_acc:.3f}")
print(f"Model accuracy:          {accuracy_score(holdout_df['HOME_WIN'], preds_holdout):.3f}")
print(f"Model log loss:          {log_loss(holdout_df['HOME_WIN'], holdout_proba):.3f}")
print(f"Model precision:         {precision_score(holdout_df['HOME_WIN'], preds_holdout):.3f}")
print(f"Model recall:            {recall_score(holdout_df['HOME_WIN'], preds_holdout):.3f}")
print(f"Model Brier:             {brier_score_loss(holdout_df['HOME_WIN'], holdout_proba):.3f}")

print(f"\nProbability range: min={holdout_proba.min():.3f}, max={holdout_proba.max():.3f}")

TRUE HOLDOUT RESULTS: 2025-26 (LOCKED CONFIG, SEEN EXACTLY ONCE)
Games evaluated:         1209
Naive baseline accuracy: 0.553
Model accuracy:          0.662
Model log loss:          0.620
Model precision:         0.667
Model recall:            0.777
Model Brier:             0.215

Probability range: min=0.181, max=0.807


In [1]:
import importlib
import nba_pipeline_consolidated
importlib.reload(nba_pipeline_consolidated)
from nba_pipeline_consolidated import *

DRY_RUN_DATE = '2026-03-11'
DRY_RUN_SEASON = '2025-26'

schedule = fetch_schedule_for_date(DRY_RUN_DATE)
print(f"Games scheduled on {DRY_RUN_DATE}:")
print(schedule)

Games scheduled on 2026-03-11:
      GAME_ID        GAME_DATE_EST HOME_TEAM_ABBR AWAY_TEAM_ABBR
0  0022500947  2026-03-11T00:00:00            ORL            CLE
1  0022500948  2026-03-11T00:00:00            NOP            TOR
2  0022500949  2026-03-11T00:00:00            UTA            NYK
3  0022500950  2026-03-11T00:00:00            DEN            HOU
4  0022500951  2026-03-11T00:00:00            SAC            CHA
5  0022500952  2026-03-11T00:00:00            LAC            MIN


In [2]:
live_features = build_live_game_features(schedule, as_of_date=DRY_RUN_DATE, season=DRY_RUN_SEASON)

print(f"\nShape: {live_features.shape}")
injury_cols = [c for c in live_features.columns if 'MISSING' in c]
print(f"\nInjury feature columns present: {injury_cols}")
print(live_features[['HOME_TEAM', 'AWAY_TEAM'] + injury_cols].to_string(index=False))

Fetching full season data for 2025-26 (single API call)...
Fetching player log for 2025-26 (for live injury/availability features)...
Fetching live injury report for 2026-03-11...
Found: https://ak-static.cms.nba.com/referee/injury/Injury-Report_2026-03-11_05_30PM.pdf

Shape: (6, 79)

Injury feature columns present: ['MISSING_PLAYERS_COUNT_HOME', 'MISSING_MINUTES_SHARE_HOME', 'MISSING_PTS_IMPACT_HOME', 'MISSING_PLAYERS_COUNT_AWAY', 'MISSING_MINUTES_SHARE_AWAY', 'MISSING_PTS_IMPACT_AWAY']
HOME_TEAM AWAY_TEAM  MISSING_PLAYERS_COUNT_HOME  MISSING_MINUTES_SHARE_HOME  MISSING_PTS_IMPACT_HOME  MISSING_PLAYERS_COUNT_AWAY  MISSING_MINUTES_SHARE_AWAY  MISSING_PTS_IMPACT_AWAY
      ORL       CLE                         2.0                    0.243333                     33.5                         1.0                    0.126667                     21.5
      NOP       TOR                         1.0                    0.100833                      9.6                         1.0               

In [3]:
importlib.reload(nba_pipeline_consolidated)
from nba_pipeline_consolidated import *

schedule = fetch_schedule_for_date('2026-03-11')
live_features = build_live_game_features(schedule, as_of_date='2026-03-11', season='2025-26')

injury_cols = [c for c in live_features.columns if 'MISSING' in c]
print(live_features[['HOME_TEAM', 'AWAY_TEAM'] + injury_cols].to_string(index=False))

Fetching full season data for 2025-26 (single API call)...
Fetching player log for 2025-26 (for live injury/availability features)...
Fetching live injury report for 2026-03-11...
Found: https://ak-static.cms.nba.com/referee/injury/Injury-Report_2026-03-11_05_30PM.pdf
HOME_TEAM AWAY_TEAM  MISSING_PLAYERS_COUNT_HOME  MISSING_MINUTES_SHARE_HOME  MISSING_PTS_IMPACT_HOME  MISSING_PLAYERS_COUNT_AWAY  MISSING_MINUTES_SHARE_AWAY  MISSING_PTS_IMPACT_AWAY
      ORL       CLE                         2.0                    0.243333                     33.5                         1.0                    0.126667                     21.5
      NOP       TOR                         1.0                    0.100833                      9.6                         1.0                    0.107917                      8.6
      UTA       NYK                         4.0                    0.505417                     75.7                         1.0                    0.127083                     14.4
   

In [4]:
# Check whether MIN and CHA actually have ANY data in our live player log at all
player_log = fetch_season_player_log('2025-26')
player_log['GAME_DATE'] = pd.to_datetime(player_log['GAME_DATE'])
player_log = player_log[player_log['GAME_DATE'] < pd.to_datetime('2026-03-11')]

for team in ['MIN', 'CHA']:
    team_rows = player_log[player_log['TEAM_ABBREVIATION'] == team]
    print(f"{team}: {len(team_rows)} player-game rows found")
    print(f"  Sample players: {team_rows['PLAYER_NAME'].unique()[:10].tolist()}")

MIN: 713 player-game rows found
  Sample players: ['Mike Conley', 'Jaden McDaniels', 'Bones Hyland', 'Rudy Gobert', 'Donte DiVincenzo', 'Jaylen Clark', 'Anthony Edwards', 'Terrence Shannon Jr', 'Julius Randle', 'Naz Reid']
CHA: 722 player-game rows found
  Sample players: ['Miles Bridges', 'Collin Sexton', 'Tre Mann', 'Tidjane Salaün', 'Kon Knueppel', 'LaMelo Ball', 'Moussa Diabaté', 'Brandon Miller', 'Ryan Kalkbrenner', 'Sion James']


In [6]:
known_url = 'https://ak-static.cms.nba.com/referee/injury/Injury-Report_2026-03-11_05_30PM.pdf'
injury_df = parse_injury_report_pdf(known_url)

print("=== Rows attributed to MinnesotaTimberwolves ===")
print(injury_df[injury_df['Team'] == 'MinnesotaTimberwolves'][['Matchup', 'Team', 'PlayerName', 'CurrentStatus']].to_string(index=False))

print("\n=== Rows attributed to CharlotteHornets ===")
print(injury_df[injury_df['Team'] == 'CharlotteHornets'][['Matchup', 'Team', 'PlayerName', 'CurrentStatus']].to_string(index=False))

print("\n=== Full injury_df, unfiltered, original document order ===")
print(injury_df[['Matchup', 'Team', 'PlayerName', 'CurrentStatus']].to_string(index=False))

=== Rows attributed to MinnesotaTimberwolves ===
Matchup                  Team              PlayerName CurrentStatus
TOR@NOP MinnesotaTimberwolves           Beringer,Joan           Out
TOR@NOP MinnesotaTimberwolves             Dosunmu,Ayo  Questionable
TOR@NOP MinnesotaTimberwolves         Freeman,Enrique           Out
TOR@NOP MinnesotaTimberwolves             Pullin,Zyon           Out
TOR@NOP MinnesotaTimberwolves            Beal,Bradley           Out
TOR@NOP MinnesotaTimberwolves            Collins,John           Out
TOR@NOP MinnesotaTimberwolves Niederhauser,YanicKonan           Out
TOR@NOP MinnesotaTimberwolves               Bona,Adem  Questionable
TOR@NOP MinnesotaTimberwolves            Broome,Johni           Out
TOR@NOP MinnesotaTimberwolves             Embiid,Joel           Out

=== Rows attributed to CharlotteHornets ===
Matchup             Team      PlayerName CurrentStatus
TOR@NOP CharlotteHornets         Hall,PJ           Out
TOR@NOP CharlotteHornets   McNeeley,Liam        

In [7]:
importlib.reload(nba_pipeline_consolidated)
from nba_pipeline_consolidated import *

known_url = 'https://ak-static.cms.nba.com/referee/injury/Injury-Report_2026-03-11_05_30PM.pdf'
injury_df = parse_injury_report_pdf(known_url)

print("=== Rows now attributed to Philadelphia76ers (should exist and contain Embiid/Beal/Collins) ===")
print(injury_df[injury_df['Team'] == 'Philadelphia76ers'][['PlayerName', 'CurrentStatus']].to_string(index=False))

print(f"\nAny remaining rows still misattributed to MIN/CHA that shouldn't be there?")
print(injury_df[injury_df['Team'].isin(['MinnesotaTimberwolves', 'CharlotteHornets'])][['Team', 'PlayerName', 'CurrentStatus']].to_string(index=False))

=== Rows now attributed to Philadelphia76ers (should exist and contain Embiid/Beal/Collins) ===
  PlayerName CurrentStatus
   Bona,Adem  Questionable
Broome,Johni           Out
 Embiid,Joel           Out

Any remaining rows still misattributed to MIN/CHA that shouldn't be there?
                 Team              PlayerName CurrentStatus
     CharlotteHornets                 Hall,PJ           Out
     CharlotteHornets           McNeeley,Liam           Out
     CharlotteHornets          Reeves,Antonio           Out
     CharlotteHornets          Salaun,Tidjane           Out
     CharlotteHornets          Williams,Grant           Out
     CharlotteHornets          Cardwell,Dylan           Out
     CharlotteHornets            Carter,Devin           Out
     CharlotteHornets         Hunter,De'Andre           Out
     CharlotteHornets             LaVine,Zach           Out
MinnesotaTimberwolves           Beringer,Joan           Out
MinnesotaTimberwolves             Dosunmu,Ayo  Questionable


In [8]:
schedule = fetch_schedule_for_date('2026-03-11')
live_features = build_live_game_features(schedule, as_of_date='2026-03-11', season='2025-26')
injury_cols = [c for c in live_features.columns if 'MISSING' in c]
print(live_features[['HOME_TEAM', 'AWAY_TEAM'] + injury_cols].to_string(index=False))

Fetching full season data for 2025-26 (single API call)...
Fetching player log for 2025-26 (for live injury/availability features)...
Fetching live injury report for 2026-03-11...
Found: https://ak-static.cms.nba.com/referee/injury/Injury-Report_2026-03-11_05_30PM.pdf
HOME_TEAM AWAY_TEAM  MISSING_PLAYERS_COUNT_HOME  MISSING_MINUTES_SHARE_HOME  MISSING_PTS_IMPACT_HOME  MISSING_PLAYERS_COUNT_AWAY  MISSING_MINUTES_SHARE_AWAY  MISSING_PTS_IMPACT_AWAY
      ORL       CLE                         2.0                    0.243333                     33.5                         1.0                    0.126667                     21.5
      NOP       TOR                         1.0                    0.100833                      9.6                         1.0                    0.107917                      8.6
      UTA       NYK                         4.0                    0.505417                     75.7                         1.0                    0.127083                     14.4
   

In [9]:
importlib.reload(nba_pipeline_consolidated)
from nba_pipeline_consolidated import *

schedule = fetch_schedule_for_date('2026-03-11')
live_features = build_live_game_features(schedule, as_of_date='2026-03-11', season='2025-26')

injury_cols = [c for c in live_features.columns if 'MISSING' in c]
print(live_features[['HOME_TEAM', 'AWAY_TEAM'] + injury_cols].to_string(index=False))

Fetching full season data for 2025-26 (single API call)...
Fetching player log for 2025-26 (for live injury/availability features)...
Fetching live injury report for 2026-03-11...
Found: https://ak-static.cms.nba.com/referee/injury/Injury-Report_2026-03-11_05_30PM.pdf
HOME_TEAM AWAY_TEAM  MISSING_PLAYERS_COUNT_HOME  MISSING_MINUTES_SHARE_HOME  MISSING_PTS_IMPACT_HOME  MISSING_PLAYERS_COUNT_AWAY  MISSING_MINUTES_SHARE_AWAY  MISSING_PTS_IMPACT_AWAY
      ORL       CLE                         2.0                    0.243333                     33.5                         1.0                    0.126667                     21.5
      NOP       TOR                         1.0                    0.100833                      9.6                         1.0                    0.107917                      8.6
      UTA       NYK                         4.0                    0.505417                     75.7                         1.0                    0.127083                     14.4
   

In [10]:
importlib.reload(nba_pipeline_consolidated)
from nba_pipeline_consolidated import *

schedule = fetch_schedule_for_date('2026-03-11')
live_features = build_live_game_features(schedule, as_of_date='2026-03-11', season='2025-26')
injury_cols = [c for c in live_features.columns if 'MISSING' in c]
print(live_features[['HOME_TEAM', 'AWAY_TEAM'] + injury_cols].to_string(index=False))

Fetching full season data for 2025-26 (single API call)...
Fetching player log for 2025-26 (for live injury/availability features)...
Fetching live injury report for 2026-03-11...
Found: https://ak-static.cms.nba.com/referee/injury/Injury-Report_2026-03-11_05_30PM.pdf
HOME_TEAM AWAY_TEAM  MISSING_PLAYERS_COUNT_HOME  MISSING_MINUTES_SHARE_HOME  MISSING_PTS_IMPACT_HOME  MISSING_PLAYERS_COUNT_AWAY  MISSING_MINUTES_SHARE_AWAY  MISSING_PTS_IMPACT_AWAY
      ORL       CLE                         2.0                    0.243333                     33.5                         1.0                    0.126667                     21.5
      NOP       TOR                         1.0                    0.100833                      9.6                         1.0                    0.107917                      8.6
      UTA       NYK                         4.0                    0.505417                     75.7                         1.0                    0.127083                     14.4
   

In [12]:
parts = [build_season_features(s) for s in ['2019-20','2020-21','2021-22','2022-23','2023-24','2024-25']]
model_ready_df = pd.concat(parts, ignore_index=True)
market_df = load_market_odds('nba_2008-2026.csv')
model_ready_df = merge_market_data(model_ready_df, market_df)
model_ready_df = add_market_residual_targets(model_ready_df)
model_df = prepare_modeling_data(model_ready_df)
model_df = model_df.dropna(subset=['MISSING_PLAYERS_COUNT_HOME','MISSING_PLAYERS_COUNT_AWAY']).reset_index(drop=True)

feature_cols = get_feature_columns(model_df)
print(f"Feature count: {len(feature_cols)} (expect 76)")

Fetching game log for 2019-20...
Building team-game features for 2019-20...
Fetching player log for 2019-20 (injury/availability features)...
Fetching game log for 2020-21...
Building team-game features for 2020-21...
Fetching player log for 2020-21 (injury/availability features)...
Fetching game log for 2021-22...
Building team-game features for 2021-22...
Fetching player log for 2021-22 (injury/availability features)...
Fetching game log for 2022-23...
Building team-game features for 2022-23...
Fetching player log for 2022-23 (injury/availability features)...
Fetching game log for 2023-24...
Building team-game features for 2023-24...
Fetching player log for 2023-24 (injury/availability features)...
Fetching game log for 2024-25...
Building team-game features for 2024-25...
Fetching player log for 2024-25 (injury/availability features)...
Market data match rate: 100.0% (7051 / 7054 games)
Dropped 95 rows with incomplete rolling features (7054 -> 6959)
Selected 76 leakage-safe feature 

In [13]:
print("=== SPREAD RESIDUAL (with injury features) ===")
spread_results_injury = run_walk_forward_residual(
    model_df, feature_cols, target_col='SPREAD_RESIDUAL',
    market_line_col='CLOSING_SPREAD_HOME', label='SPREAD'
)

print("\n=== TOTALS RESIDUAL (with injury features) ===")
totals_results_injury = run_walk_forward_residual(
    model_df, feature_cols, target_col='TOTALS_RESIDUAL',
    market_line_col='CLOSING_TOTAL', label='TOTALS'
)

print(f"\nSpread -- beats market in {spread_results_injury['beats_market'].sum()}/{len(spread_results_injury)} folds")
print(f"Totals -- beats market in {totals_results_injury['beats_market'].sum()}/{len(totals_results_injury)} folds")

=== SPREAD RESIDUAL (with injury features) ===
[SPREAD] Season 2020-21: market_mae=10.72  model_mae=10.85  
[SPREAD] Season 2021-22: market_mae=10.59  model_mae=10.69  
[SPREAD] Season 2022-23: market_mae=9.78  model_mae=9.88  
[SPREAD] Season 2023-24: market_mae=10.56  model_mae=10.54  <- MODEL BEATS MARKET
[SPREAD] Season 2024-25: market_mae=10.49  model_mae=10.58  

=== TOTALS RESIDUAL (with injury features) ===
[TOTALS] Season 2020-21: market_mae=14.38  model_mae=14.79  
[TOTALS] Season 2021-22: market_mae=14.05  model_mae=14.21  
[TOTALS] Season 2022-23: market_mae=14.22  model_mae=14.19  <- MODEL BEATS MARKET
[TOTALS] Season 2023-24: market_mae=14.24  model_mae=14.43  
[TOTALS] Season 2024-25: market_mae=14.12  model_mae=14.15  

Spread -- beats market in 1/5 folds
Totals -- beats market in 1/5 folds


In [14]:
for test_date in ['2026-01-15', '2026-02-10']:
    print(f"\n{'='*70}\nTESTING: {test_date}\n{'='*70}")
    try:
        schedule = fetch_schedule_for_date(test_date)
        print(f"Games found: {len(schedule)}")

        live_features = build_live_game_features(schedule, as_of_date=test_date, season='2025-26')
        injury_cols = [c for c in live_features.columns if 'MISSING' in c]
        print(live_features[['HOME_TEAM', 'AWAY_TEAM'] + injury_cols].to_string(index=False))
    except Exception as e:
        print(f"ERROR on {test_date}: {type(e).__name__}: {e}")


TESTING: 2026-01-15
Games found: 9
Fetching full season data for 2025-26 (single API call)...
Fetching player log for 2025-26 (for live injury/availability features)...
Fetching live injury report for 2026-01-15...
Found: https://ak-static.cms.nba.com/referee/injury/Injury-Report_2026-01-15_05_30PM.pdf
HOME_TEAM AWAY_TEAM  MISSING_PLAYERS_COUNT_HOME  MISSING_MINUTES_SHARE_HOME  MISSING_PTS_IMPACT_HOME  MISSING_PLAYERS_COUNT_AWAY  MISSING_MINUTES_SHARE_AWAY  MISSING_PTS_IMPACT_AWAY
      ORL       MEM                         1.0                    0.118333                   17.400                         2.0                    0.222500                     32.1
      DET       PHX                         0.0                    0.000000                    0.000                         2.0                    0.234583                     53.9
      MIA       BOS                         2.0                    0.232083                   23.600                         0.0                    0

In [15]:
# Re-fetch the same live player log used in the Jan 15 test (or reuse if still in memory)
player_log_check = fetch_season_player_log('2025-26')
player_log_check['GAME_DATE'] = pd.to_datetime(player_log_check['GAME_DATE'])
player_log_check = player_log_check[player_log_check['GAME_DATE'] < pd.to_datetime('2026-01-15')]

checks = [
    ('MEM', 'Pippen'),
    ('NYK', 'Jemison'),
    ('SAS', 'Jones'),   # broad surname search since "Jones-Garcia" could split multiple ways
]

for team, surname_fragment in checks:
    team_players = player_log_check[player_log_check['TEAM_ABBREVIATION'] == team]['PLAYER_NAME'].unique()
    matches = [p for p in team_players if surname_fragment.lower() in p.lower()]
    print(f"{team} players containing '{surname_fragment}': {matches if matches else 'NONE FOUND -- genuinely absent from recent games'}")

MEM players containing 'Pippen': NONE FOUND -- genuinely absent from recent games
NYK players containing 'Jemison': ['Trey Jemison III']
SAS players containing 'Jones': ['David Jones Garcia']


In [16]:
importlib.reload(nba_pipeline_consolidated)
from nba_pipeline_consolidated import *

schedule = fetch_schedule_for_date('2026-01-15')
live_features = build_live_game_features(schedule, as_of_date='2026-01-15', season='2025-26')
injury_cols = [c for c in live_features.columns if 'MISSING' in c]
print(live_features[['HOME_TEAM', 'AWAY_TEAM'] + injury_cols].to_string(index=False))

Fetching full season data for 2025-26 (single API call)...
Fetching player log for 2025-26 (for live injury/availability features)...
Fetching live injury report for 2026-01-15...
Found: https://ak-static.cms.nba.com/referee/injury/Injury-Report_2026-01-15_05_30PM.pdf
HOME_TEAM AWAY_TEAM  MISSING_PLAYERS_COUNT_HOME  MISSING_MINUTES_SHARE_HOME  MISSING_PTS_IMPACT_HOME  MISSING_PLAYERS_COUNT_AWAY  MISSING_MINUTES_SHARE_AWAY  MISSING_PTS_IMPACT_AWAY
      ORL       MEM                         1.0                    0.118333                   17.400                         2.0                    0.222500                     32.1
      DET       PHX                         0.0                    0.000000                    0.000                         2.0                    0.234583                     53.9
      MIA       BOS                         2.0                    0.232083                   23.600                         0.0                    0.000000                      0.0
   

In [17]:
schedule = fetch_schedule_for_date('2026-02-10')
live_features = build_live_game_features(schedule, as_of_date='2026-02-10', season='2025-26')
injury_cols = [c for c in live_features.columns if 'MISSING' in c]
print(live_features[['HOME_TEAM', 'AWAY_TEAM'] + injury_cols].to_string(index=False))

Fetching full season data for 2025-26 (single API call)...
Fetching player log for 2025-26 (for live injury/availability features)...
Fetching live injury report for 2026-02-10...
Found: https://ak-static.cms.nba.com/referee/injury/Injury-Report_2026-02-10_05_30PM.pdf
HOME_TEAM AWAY_TEAM  MISSING_PLAYERS_COUNT_HOME  MISSING_MINUTES_SHARE_HOME  MISSING_PTS_IMPACT_HOME  MISSING_PLAYERS_COUNT_AWAY  MISSING_MINUTES_SHARE_AWAY  MISSING_PTS_IMPACT_AWAY
      NYK       IND                         2.0                    0.215000                     20.1                         2.0                    0.229583                     21.3
      HOU       LAC                         1.0                    0.102083                      5.6                         0.0                    0.000000                      0.0
      PHX       DAL                         1.0                    0.132500                     18.2                         0.0                    0.000000                      0.0
   

In [18]:
# Reuse the model/calibrator from the 76-feature, injury-enhanced training
# we already built for the true holdout — train strictly on 2019-24,
# calibrate on 2024-25, test on a season NEVER used for tuning decisions.
train_df_kelly = model_df[model_df['SEASON'].isin(['2019-20','2020-21','2021-22','2022-23'])]
calib_df_kelly = model_df[model_df['SEASON'] == '2023-24']
eval_df_kelly = model_df[model_df['SEASON'] == '2022-23']  # has real moneyline coverage

X_train_k = train_df_kelly[feature_cols].fillna(train_df_kelly[feature_cols].median())
y_train_k = train_df_kelly['HOME_WIN']
train_medians_k = X_train_k.median()

X_calib_k = calib_df_kelly[feature_cols].fillna(train_medians_k)
X_eval_k = eval_df_kelly[feature_cols].fillna(train_medians_k)

model_k = XGBClassifier(max_depth=3, learning_rate=0.03, n_estimators=150,
                         min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
                         eval_metric='logloss', random_state=42)
model_k.fit(X_train_k, y_train_k)

raw_proba_calib_k = model_k.predict_proba(X_calib_k)[:, 1]
platt_k = fit_platt_calibrator(raw_proba_calib_k, calib_df_kelly['HOME_WIN'].values)
raw_proba_eval_k = model_k.predict_proba(X_eval_k)[:, 1]
calibrated_proba_eval_k = apply_platt_calibration(platt_k, raw_proba_eval_k)

print(f"Probability range: min={calibrated_proba_eval_k.min():.3f}, max={calibrated_proba_eval_k.max():.3f}")
print(f"(Compressed pre-injury model was 0.263-0.767 -- checking if this genuinely widened)")

bet_df_k = eval_df_kelly[eval_df_kelly['MONEYLINE_HOME'].notna()].copy()
bet_df_k['OUR_PROB'] = calibrated_proba_eval_k[eval_df_kelly['MONEYLINE_HOME'].notna().values]
bet_df_k['ACTUAL_OUTCOME'] = bet_df_k['HOME_WIN']

backtest_result_k = simulate_kelly_backtest(
    predicted_probs=bet_df_k['OUR_PROB'].values,
    actual_outcomes=bet_df_k['ACTUAL_OUTCOME'].values,
    american_odds=bet_df_k['MONEYLINE_HOME'].values,
    starting_bankroll=1000.0, kelly_multiplier=0.25, max_bet_pct=0.05,
    min_edge_threshold=0.02
)
print(f"\nGames with real odds: {len(bet_df_k)}")
print(f"Bets placed: {backtest_result_k['bets_placed']}")
print(f"Ending bankroll: ${backtest_result_k['ending_bankroll']:.2f} (started $1000)")
print(f"Total return: {backtest_result_k['total_return_pct']:.2f}%")
print(f"Max drawdown: {backtest_result_k['max_drawdown_pct']:.2f}%")

Probability range: min=0.164, max=0.827
(Compressed pre-injury model was 0.263-0.767 -- checking if this genuinely widened)

Games with real odds: 648
Bets placed: 239
Ending bankroll: $45655.58 (started $1000)
Total return: 4465.56%
Max drawdown: -26.70%


In [19]:
# Same diagnostic as before: is this genuine, or riding a probability-range artifact?
bet_df_k['BREAKEVEN_PROB'] = 1 / bet_df_k['MONEYLINE_HOME'].apply(american_to_decimal)
bet_df_k['EDGE'] = bet_df_k['OUR_PROB'] - bet_df_k['BREAKEVEN_PROB']

placed_k = bet_df_k[bet_df_k['EDGE'] > 0.02]
print(f"Games bet on: {len(placed_k)} / {len(bet_df_k)}")
print(f"\nOur actual win rate on bet games: {placed_k['ACTUAL_OUTCOME'].mean():.3f}")
print(f"Average breakeven required: {placed_k['BREAKEVEN_PROB'].mean():.3f}")
print(f"Average OUR_PROB claimed: {placed_k['OUR_PROB'].mean():.3f}")

# Check concentration: are we betting disproportionately on extreme favorites now (not underdogs)?
print(f"\nBreakeven distribution of BET games:\n{placed_k['BREAKEVEN_PROB'].describe()}")
print(f"\nEdge distribution of BET games:\n{placed_k['EDGE'].describe()}")

# Look at the biggest individual bets -- where did the bankroll actually grow?
print(f"\nTop 10 highest-edge bets:")
print(placed_k.nlargest(10, 'EDGE')[['BREAKEVEN_PROB', 'OUR_PROB', 'EDGE', 'ACTUAL_OUTCOME', 'MONEYLINE_HOME']].to_string(index=False))

# Sanity: what's the actual win rate needed to justify this edge at these odds?
print(f"\nWin rate needed vs actual at edge>0.02 subset -- if actual << our claimed prob, that's the tell")

Games bet on: 239 / 648

Our actual win rate on bet games: 0.619
Average breakeven required: 0.461
Average OUR_PROB claimed: 0.568

Breakeven distribution of BET games:
count    239.000000
mean       0.461083
std        0.136434
min        0.111111
25%        0.357143
50%        0.454545
75%        0.565217
max        0.736842
Name: BREAKEVEN_PROB, dtype: float64

Edge distribution of BET games:
count    239.000000
mean       0.107066
std        0.067781
min        0.020901
25%        0.050030
50%        0.090838
75%        0.148113
max        0.327982
Name: EDGE, dtype: float64

Top 10 highest-edge bets:
 BREAKEVEN_PROB  OUR_PROB     EDGE  ACTUAL_OUTCOME  MONEYLINE_HOME
       0.200000  0.527982 0.327982               1           400.0
       0.357143  0.658143 0.301000               1           180.0
       0.238095  0.536603 0.298508               1           320.0
       0.454545  0.744694 0.290149               1           120.0
       0.333333  0.601267 0.267934               1  

In [4]:
import importlib
import nba_pipeline_consolidated
importlib.reload(nba_pipeline_consolidated)
from nba_pipeline_consolidated import *

In [5]:
parts = [build_season_features(s) for s in ['2019-20','2020-21','2021-22','2022-23','2023-24','2024-25']]
model_ready_df = pd.concat(parts, ignore_index=True)
market_df = load_market_odds('nba_2008-2026.csv')
model_ready_df = merge_market_data(model_ready_df, market_df)
model_ready_df = add_market_residual_targets(model_ready_df)
model_df = prepare_modeling_data(model_ready_df)
model_df = model_df.dropna(subset=['MISSING_PLAYERS_COUNT_HOME','MISSING_PLAYERS_COUNT_AWAY']).reset_index(drop=True)
feature_cols = get_feature_columns(model_df)
print(f"Feature count: {len(feature_cols)} (expect 76)")

Fetching game log for 2019-20...
Building team-game features for 2019-20...
Fetching player log for 2019-20 (injury/availability features)...
Fetching game log for 2020-21...
Building team-game features for 2020-21...
Fetching player log for 2020-21 (injury/availability features)...
Fetching game log for 2021-22...
Building team-game features for 2021-22...
Fetching player log for 2021-22 (injury/availability features)...
Fetching game log for 2022-23...
Building team-game features for 2022-23...
Fetching player log for 2022-23 (injury/availability features)...
Fetching game log for 2023-24...
Building team-game features for 2023-24...
Fetching player log for 2023-24 (injury/availability features)...
Fetching game log for 2024-25...
Building team-game features for 2024-25...
Fetching player log for 2024-25 (injury/availability features)...
Market data match rate: 100.0% (7051 / 7054 games)
Dropped 95 rows with incomplete rolling features (7054 -> 6959)
Selected 76 leakage-safe feature 

In [6]:
train_df_kelly = model_df[model_df['SEASON'].isin(['2019-20','2020-21','2021-22','2022-23'])]
calib_df_kelly = model_df[model_df['SEASON'] == '2023-24']
eval_df_kelly = model_df[model_df['SEASON'] == '2022-23']

X_train_k = train_df_kelly[feature_cols].fillna(train_df_kelly[feature_cols].median())
y_train_k = train_df_kelly['HOME_WIN']
train_medians_k = X_train_k.median()
X_calib_k = calib_df_kelly[feature_cols].fillna(train_medians_k)
X_eval_k = eval_df_kelly[feature_cols].fillna(train_medians_k)

model_k = XGBClassifier(max_depth=3, learning_rate=0.03, n_estimators=150,
                         min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
                         eval_metric='logloss', random_state=42)
model_k.fit(X_train_k, y_train_k)

raw_proba_calib_k = model_k.predict_proba(X_calib_k)[:, 1]
platt_k = fit_platt_calibrator(raw_proba_calib_k, calib_df_kelly['HOME_WIN'].values)
raw_proba_eval_k = model_k.predict_proba(X_eval_k)[:, 1]
calibrated_proba_eval_k = apply_platt_calibration(platt_k, raw_proba_eval_k)

bet_df_k = eval_df_kelly[eval_df_kelly['MONEYLINE_HOME'].notna()].copy()
bet_df_k['OUR_PROB'] = calibrated_proba_eval_k[eval_df_kelly['MONEYLINE_HOME'].notna().values]
bet_df_k['ACTUAL_OUTCOME'] = bet_df_k['HOME_WIN']
bet_df_k['BREAKEVEN_PROB'] = 1 / bet_df_k['MONEYLINE_HOME'].apply(american_to_decimal)
bet_df_k['EDGE'] = bet_df_k['OUR_PROB'] - bet_df_k['BREAKEVEN_PROB']
placed_k = bet_df_k[bet_df_k['EDGE'] > 0.02]
print(f"Rebuilt. Games with odds: {len(bet_df_k)}, bets placed: {len(placed_k)}")

Rebuilt. Games with odds: 648, bets placed: 239


In [7]:
from sklearn.calibration import calibration_curve

prob_true, prob_pred = calibration_curve(eval_df_kelly['HOME_WIN'], calibrated_proba_eval_k, n_bins=10, strategy='quantile')
print("=== Full-season calibration (648 games) ===")
for pt, pp in zip(prob_true, prob_pred):
    diff = pt - pp
    print(f"{pp:>10.3f} {pt:>10.3f} {diff:>8.3f}{'  <-- off by >5pts' if abs(diff) > 0.05 else ''}")

prob_true_bet, prob_pred_bet = calibration_curve(placed_k['ACTUAL_OUTCOME'], placed_k['OUR_PROB'], n_bins=5, strategy='quantile')
print(f"\n=== Calibration within the {len(placed_k)} BET games specifically ===")
for pt, pp in zip(prob_true_bet, prob_pred_bet):
    diff = pt - pp
    print(f"{pp:>10.3f} {pt:>10.3f} {diff:>8.3f}{'  <-- off by >5pts' if abs(diff) > 0.05 else ''}")

=== Full-season calibration (648 games) ===
     0.282      0.221   -0.061  <-- off by >5pts
     0.388      0.298   -0.091  <-- off by >5pts
     0.457      0.405   -0.052  <-- off by >5pts
     0.507      0.574    0.067  <-- off by >5pts
     0.552      0.496   -0.056  <-- off by >5pts
     0.595      0.661    0.066  <-- off by >5pts
     0.632      0.623   -0.009
     0.671      0.793    0.123  <-- off by >5pts
     0.708      0.826    0.118  <-- off by >5pts
     0.762      0.910    0.147  <-- off by >5pts

=== Calibration within the 239 BET games specifically ===
     0.398      0.479    0.081  <-- off by >5pts
     0.506      0.521    0.015
     0.577      0.553   -0.024
     0.645      0.667    0.021
     0.714      0.875    0.161  <-- off by >5pts


In [8]:
# Reuse model_df (built moments ago for the Kelly check) -- this only
# needs to fetch the NEW 2025-26 season, not re-fetch 2019-25 again
model_locked2, platt_locked2, holdout_df2, holdout_proba2 = run_true_holdout_validation(existing_model_df=model_df)

Reusing existing model_df in memory for train/calibrate seasons.
Dropped 0 rows with incomplete rolling features (6959 -> 6959)
Selected 76 leakage-safe feature columns.
Locked feature count: 76 (expected 76)

FETCHING TRUE HOLDOUT: 2025-26 (never seen before now)
Fetching game log for 2025-26...
Building team-game features for 2025-26...
Fetching player log for 2025-26 (injury/availability features)...
Market data match rate: 100.0% (1225 / 1225 games)
Dropped 16 rows with incomplete rolling features (1225 -> 1209)

True holdout size: 1209 games

TRUE HOLDOUT RESULTS: 2025-26 (LOCKED CONFIG, SEEN ONCE)
Naive baseline accuracy: 0.553
Model accuracy:          0.661
Model log loss:          0.620
Model precision:         0.666
Model recall:            0.776
Model Brier:             0.215

No moneyline odds available for 2025-26 in our free dataset -- market comparison not possible for this season (consistent with the known coverage gap).


In [9]:
from sklearn.calibration import calibration_curve

prob_true_holdout, prob_pred_holdout = calibration_curve(
    holdout_df2['HOME_WIN'], holdout_proba2, n_bins=10, strategy='quantile'
)

print("=== Calibration on the TRUE 2025-26 holdout (never used for any tuning decision) ===")
for pt, pp in zip(prob_true_holdout, prob_pred_holdout):
    diff = pt - pp
    print(f"{pp:>10.3f} {pt:>10.3f} {diff:>8.3f}{'  <-- off by >5pts' if abs(diff) > 0.05 else ''}")

print(f"\nBins off by >5pts: {sum(1 for pt, pp in zip(prob_true_holdout, prob_pred_holdout) if abs(pt-pp) > 0.05)} / {len(prob_true_holdout)}")
print(f"(For reference, 2022-23 showed 8/10 bins off by >5pts, with high-confidence bins systematically underconfident)")

# Specifically check the same high-confidence bins that were problematic before
high_conf_mask = holdout_proba2 >= 0.65
if high_conf_mask.sum() > 10:
    print(f"\nHigh-confidence (>=0.65) predictions in holdout: n={high_conf_mask.sum()}")
    print(f"Average predicted: {holdout_proba2[high_conf_mask].mean():.3f}")
    print(f"Average actual: {holdout_df2['HOME_WIN'].values[high_conf_mask].mean():.3f}")

=== Calibration on the TRUE 2025-26 holdout (never used for any tuning decision) ===
     0.279      0.223   -0.056  <-- off by >5pts
     0.379      0.380    0.001
     0.444      0.388   -0.056  <-- off by >5pts
     0.497      0.488   -0.010
     0.546      0.562    0.016
     0.589      0.550   -0.039
     0.631      0.653    0.021
     0.670      0.678    0.007
     0.711      0.760    0.050
     0.762      0.851    0.089  <-- off by >5pts

Bins off by >5pts: 3 / 10
(For reference, 2022-23 showed 8/10 bins off by >5pts, with high-confidence bins systematically underconfident)

High-confidence (>=0.65) predictions in holdout: n=366
Average predicted: 0.714
Average actual: 0.762


In [10]:
train_seasons_cal = ['2019-20', '2020-21']
eval_season_cal = '2023-24'

train_df_cal = model_df[model_df['SEASON'].isin(train_seasons_cal)]
eval_df_cal = model_df[model_df['SEASON'] == eval_season_cal]

X_train_cal = train_df_cal[feature_cols].fillna(train_df_cal[feature_cols].median())
y_train_cal = train_df_cal['HOME_WIN']
train_medians_cal = X_train_cal.median()
X_eval_cal = eval_df_cal[feature_cols].fillna(train_medians_cal)

model_cal = XGBClassifier(max_depth=3, learning_rate=0.03, n_estimators=150,
                           min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
                           eval_metric='logloss', random_state=42)
model_cal.fit(X_train_cal, y_train_cal)
raw_proba_eval_cal = model_cal.predict_proba(X_eval_cal)[:, 1]

# --- Calibration A: single season (old approach) ---
calib_A = model_df[model_df['SEASON'] == '2021-22']
X_calib_A = calib_A[feature_cols].fillna(train_medians_cal)
raw_proba_calib_A = model_cal.predict_proba(X_calib_A)[:, 1]
platt_A = fit_platt_calibrator(raw_proba_calib_A, calib_A['HOME_WIN'].values)
calibrated_A = apply_platt_calibration(platt_A, raw_proba_eval_cal)

# --- Calibration B: two seasons combined (new approach) ---
calib_B = model_df[model_df['SEASON'].isin(['2021-22', '2022-23'])]
X_calib_B = calib_B[feature_cols].fillna(train_medians_cal)
raw_proba_calib_B = model_cal.predict_proba(X_calib_B)[:, 1]
platt_B = fit_platt_calibrator(raw_proba_calib_B, calib_B['HOME_WIN'].values)
calibrated_B = apply_platt_calibration(platt_B, raw_proba_eval_cal)

print(f"Calibration A (1 season, n={len(calib_A)}): fit")
print(f"Calibration B (2 seasons, n={len(calib_B)}): fit")

from sklearn.calibration import calibration_curve
from sklearn.metrics import log_loss, brier_score_loss

for label, proba in [('A (1-season calib)', calibrated_A), ('B (2-season calib)', calibrated_B)]:
    print(f"\n=== {label} ===")
    print(f"Log loss: {log_loss(eval_df_cal['HOME_WIN'], proba):.4f}  Brier: {brier_score_loss(eval_df_cal['HOME_WIN'], proba):.4f}")
    high_conf_mask = proba >= 0.65
    if high_conf_mask.sum() > 5:
        print(f"High-conf (>=0.65) n={high_conf_mask.sum()}: predicted={proba[high_conf_mask].mean():.3f}, actual={eval_df_cal['HOME_WIN'].values[high_conf_mask].mean():.3f}")

Calibration A (1 season, n=1214): fit
Calibration B (2 seasons, n=2428): fit

=== A (1-season calib) ===
Log loss: 0.6305  Brier: 0.2199
High-conf (>=0.65) n=236: predicted=0.696, actual=0.771

=== B (2-season calib) ===
Log loss: 0.6300  Brier: 0.2198
High-conf (>=0.65) n=308: predicted=0.706, actual=0.776


In [11]:
raw_proba_calib_B_isotonic = model_cal.predict_proba(X_calib_B)[:, 1]
isotonic_B = fit_isotonic_calibrator(raw_proba_calib_B_isotonic, calib_B['HOME_WIN'].values)
calibrated_isotonic_B = apply_calibration(isotonic_B, raw_proba_eval_cal)

print("=== Isotonic (2-season calibration, n=2428) ===")
print(f"Log loss: {log_loss(eval_df_cal['HOME_WIN'], calibrated_isotonic_B):.4f}  Brier: {brier_score_loss(eval_df_cal['HOME_WIN'], calibrated_isotonic_B):.4f}")
high_conf_mask_iso = calibrated_isotonic_B >= 0.65
print(f"High-conf (>=0.65) n={high_conf_mask_iso.sum()}: predicted={calibrated_isotonic_B[high_conf_mask_iso].mean():.3f}, actual={eval_df_cal['HOME_WIN'].values[high_conf_mask_iso].mean():.3f}")

print(f"\n--- Comparison summary ---")
print(f"Platt (1 season):  logloss=0.6305  high-conf gap=0.075")
print(f"Platt (2 seasons): logloss=0.6300  high-conf gap=0.070")
print(f"Isotonic (2 seasons): logloss={log_loss(eval_df_cal['HOME_WIN'], calibrated_isotonic_B):.4f}  high-conf gap={calibrated_isotonic_B[high_conf_mask_iso].mean() - eval_df_cal['HOME_WIN'].values[high_conf_mask_iso].mean():.3f}")

=== Isotonic (2-season calibration, n=2428) ===
Log loss: 0.6480  Brier: 0.2185
High-conf (>=0.65) n=404: predicted=0.708, actual=0.748

--- Comparison summary ---
Platt (1 season):  logloss=0.6305  high-conf gap=0.075
Platt (2 seasons): logloss=0.6300  high-conf gap=0.070
Isotonic (2 seasons): logloss=0.6480  high-conf gap=-0.039
